In [1]:
# ============================================================
# CELL 0 — DEPENDENCIES
# ============================================================

import importlib.util
import subprocess
import sys

REQUIRED = {
    "requests": "requests",
    "numpy": "numpy",
    "pandas": "pandas",
    "PIL": "pillow",
    "torch": "torch",
    "torchvision": "torchvision",
    "transformers": "transformers",
    "sklearn": "scikit-learn",
    "flask": "flask",
    "flask_cors": "flask-cors",
}

missing = [
    package
    for module, package in REQUIRED.items()
    if importlib.util.find_spec(module) is None
]

if missing:
    print(
        "Installing:",
        ", ".join(missing)
    )

    subprocess.check_call([
        sys.executable,
        "-m",
        "pip",
        "install",
        *missing
    ])

    print(
        "Installed. Restart the kernel once "
        "if the next cell cannot import."
    )

else:
    print(
        "All dependencies are already installed."
    )

All dependencies are already installed.


In [2]:
# ============================================================
# CELL 1 — IMPORTS + CONFIGURATION
#
# SINGLE BACKEND:
# ROUTE CRAWL + MAPILLARY + OSM + SEGFORMER
# ============================================================

from pathlib import Path
from collections import Counter, defaultdict
from concurrent.futures import ThreadPoolExecutor, as_completed
from datetime import datetime

import os
import io
import re
import json
import math
import time
import threading
import warnings

import requests
import numpy as np
import pandas as pd

from PIL import (
    Image,
    ImageOps,
)

import torch
import torch.nn.functional as F

from transformers import (
    SegformerImageProcessor,
    SegformerForSemanticSegmentation,
)

from sklearn.preprocessing import StandardScaler
from sklearn.cluster import KMeans

from flask import (
    Flask,
    request,
    jsonify,
)

from flask_cors import CORS


# ============================================================
# PROJECT ROOT
# ============================================================

NOTEBOOK_DIR = Path.cwd()

PROJECT_ROOT = (
    NOTEBOOK_DIR.parent
    if NOTEBOOK_DIR.name.lower() == "notebooks"
    else NOTEBOOK_DIR
)


DATA_DIR = (
    PROJECT_ROOT
    /
    "data"
)

GENERATED_DIR = (
    DATA_DIR
    /
    "generated"
)

ROUTE_INTELLIGENCE_DIR = (
    GENERATED_DIR
    /
    "route_intelligence"
)

STREET_IMAGE_DIR = (
    GENERATED_DIR
    /
    "street_images"
)

SEGMENTATION_IMAGE_DIR = (
    GENERATED_DIR
    /
    "segmentation_images"
)

IMAGE_CACHE_DIR = (
    GENERATED_DIR
    /
    "image_analysis_cache"
)


IMAGE_COMPILER_PATH = (
    GENERATED_DIR
    /
    "image_compiler.json"
)

IMAGE_ANALYSIS_FULL_PATH = (
    GENERATED_DIR
    /
    "image_analysis_full.json"
)

MRT_CONTEXT_PATH = (
    GENERATED_DIR
    /
    "mrt_context_som.json"
)


for folder in [
    GENERATED_DIR,
    ROUTE_INTELLIGENCE_DIR,
    STREET_IMAGE_DIR,
    SEGMENTATION_IMAGE_DIR,
    IMAGE_CACHE_DIR,
]:
    folder.mkdir(
        parents=True,
        exist_ok=True
    )


# ============================================================
# ROUTE CRAWLING
# ============================================================

CHECKPOINT_SPACING_M = 15

MAPILLARY_RADIUS_M = 55

MAPILLARY_IMAGES_PER_CHECKPOINT = 3

MAPILLARY_ROUTE_CANDIDATE_LIMIT = 1200

MAPILLARY_PAGE_LIMIT = 1000

MAPILLARY_MIN_HEADING_SEPARATION_DEG = 40


OSM_CHECKPOINT_RADIUS_M = 60

OSM_ROUTE_BUFFER_M = 90


DOWNLOAD_WORKERS = 6

REQUEST_TIMEOUT = 45

OVERPASS_TIMEOUT = 90


# ============================================================
# SAFETY LIMITS
# ============================================================

# Start with 10 routes.
# Set to None later if you want EVERY discovered route.
MAX_ROUTES_TO_CRAWL = None


# Maximum number of images SegFormer will analyse
# in a single run.
MAX_TOTAL_IMAGES = 600


# Frontend-compatible representative grid.
COMPILER_GRID_SIZE = 100


# Optional:
# "ADMIRALTY"
# "CLEMENTI"
# etc.
#
# Leave None to process whichever routes are discovered.
STATION_FILTER = None


# ============================================================
# SEGFORMER
# ============================================================

SEGFORMER_MODEL_NAME = (
    "nvidia/"
    "segformer-b2-finetuned-ade-512-512"
)

DEVICE = (
    "cuda"
    if torch.cuda.is_available()
    else
    "cpu"
)


# ============================================================
# IMAGE QC
# ============================================================

MIN_IMAGE_WIDTH = 320

MIN_IMAGE_HEIGHT = 240

MIN_BRIGHTNESS = 16

MAX_BRIGHTNESS = 244

MIN_CONTRAST = 14

MIN_SHARPNESS = 6.0

DUPLICATE_HASH_DISTANCE = 4


# ============================================================
# CLUSTERING
# ============================================================

DEFAULT_VISUAL_CLUSTER_COUNT = 10

RANDOM_STATE = 42


# ============================================================
# SMOOTH ROUTE PROFILE
# ============================================================

ROUTE_PROFILE_STEP_M = 10

ROUTE_SMOOTHING_SIGMA_M = 38

ROUTE_SMOOTHING_MAX_GAP_M = 90


# ============================================================
# REMOTE SERVICES
# ============================================================

OVERPASS_URL = (
    "https://overpass-api.de/api/interpreter"
)

MAPILLARY_IMAGES_URL = (
    "https://graph.mapillary.com/images"
)

MAPILLARY_GRAPH_URL = (
    "https://graph.mapillary.com"
)


# ============================================================
# FIND YOUR EXISTING MAPILLARY TOKEN AUTOMATICALLY
#
# We do NOT print the token.
# ============================================================

def discover_mapillary_token():

    token = os.getenv(
        "MAPILLARY_ACCESS_TOKEN",
        ""
    ).strip()

    if token:
        return token


    candidates = [

        PROJECT_ROOT
        /
        "notebooks"
        /
        "route_imagery_backend.ipynb",

        PROJECT_ROOT
        /
        "notebooks"
        /
        "image_analysis.ipynb",

        PROJECT_ROOT
        /
        ".env",
    ]


    pattern = re.compile(
        r"MLY\|[A-Za-z0-9_\-|]+"
    )


    for path in candidates:

        if not path.exists():
            continue

        try:

            text = path.read_text(
                encoding="utf-8",
                errors="ignore"
            )

            match = pattern.search(
                text
            )

            if match:
                return match.group(0)

        except Exception:
            pass


    return ""


MAPILLARY_ACCESS_TOKEN = (
    discover_mapillary_token()
)


# ============================================================
# STATUS
# ============================================================

STATUS = {

    "running":
        False,

    "stage":
        "ready",

    "message":
        "Backend loaded.",

    "processed":
        0,

    "total":
        0,

    "lastUpdated":
        datetime.now().isoformat(
            timespec="seconds"
        ),

    "lastResult":
        None,
}


STATUS_LOCK = threading.Lock()

COMPILE_LOCK = threading.Lock()


def update_status(
    stage,
    message,
    **extra
):

    with STATUS_LOCK:

        STATUS["stage"] = stage

        STATUS["message"] = message

        STATUS["lastUpdated"] = (
            datetime.now().isoformat(
                timespec="seconds"
            )
        )

        STATUS.update(
            extra
        )


    print(
        f"[{stage}] {message}"
    )


print()
print(
    "=================================================="
)
print(
    "YOUNG FAMILIES — SINGLE CRAWL BACKEND"
)
print(
    "=================================================="
)
print(
    "Project:",
    PROJECT_ROOT
)
print(
    "Device:",
    DEVICE
)
print(
    "Mapillary token:",
    "FOUND"
    if MAPILLARY_ACCESS_TOKEN
    else
    "NOT FOUND"
)
print(
    "Checkpoint spacing:",
    CHECKPOINT_SPACING_M,
    "m"
)
print(
    "Backend port: 5001"
)
print(
    "=================================================="
)

C:\Users\MSI\AppData\Local\Programs\Python\Python313\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm



YOUNG FAMILIES — SINGLE CRAWL BACKEND
Project: C:\Users\MSI\OneDrive\Desktop\family-toilet-search
Device: cpu
Mapillary token: FOUND
Checkpoint spacing: 15 m
Backend port: 5001


In [3]:
# ============================================================
# CELL 2 — HELPERS + MRT CONTEXT + ROUTE DISCOVERY
# ============================================================

def safe_json_dump(
    path,
    payload
):

    path = Path(
        path
    )

    path.parent.mkdir(
        parents=True,
        exist_ok=True
    )


    temporary = (
        path.with_suffix(
            path.suffix
            +
            ".tmp"
        )
    )


    with open(
        temporary,
        "w",
        encoding="utf-8"
    ) as file:

        json.dump(
            payload,
            file,
            indent=2,
            ensure_ascii=False,
            allow_nan=False
        )


    temporary.replace(
        path
    )


def load_json(
    path
):

    with open(
        path,
        "r",
        encoding="utf-8"
    ) as file:

        return json.load(
            file
        )


def slugify(
    value
):

    return re.sub(

        r"[^a-z0-9]+",

        "-",

        str(
            value
            or
            ""
        )
        .lower()
        .strip()

    ).strip("-")


def safe_float(
    value,
    default=0.0
):

    try:

        number = float(
            value
        )

        if math.isfinite(
            number
        ):
            return number

    except Exception:
        pass


    return default


def normalize_station_name(
    value
):

    value = re.sub(
        r"<[^>]+>",
        " ",
        str(
            value
            or
            ""
        )
    )


    value = re.sub(

        r"^[A-Z]{1,4}\d{1,3}"
        r"(?:/[A-Z]{1,4}\d{1,3})*"
        r"\s*[-–—:]?\s*",

        "",

        value,

        flags=re.I
    )


    value = re.sub(

        r"\b(?:MRT|LRT)"
        r"(?:\s+STATION)?\b",

        "",

        value,

        flags=re.I
    )


    value = re.sub(
        r"\s+",
        " ",
        value
    )


    return (
        value
        .strip(
            " -–—,;"
        )
        .strip()
        .upper()
    )


def normalize_som_cell(
    value
):

    if value is None:
        return None


    if isinstance(
        value,
        (
            list,
            tuple
        )
    ):

        return "-".join(
            str(item)
            for item in value
        )


    if isinstance(
        value,
        dict
    ):

        if (
            "x" in value
            and
            "y" in value
        ):

            return (
                f"{value['x']}-"
                f"{value['y']}"
            )


        return json.dumps(
            value,
            sort_keys=True
        )


    text = str(
        value
    ).strip()


    return (
        text
        or
        None
    )


def to_web_path(
    path
):

    path = Path(
        path
    )


    try:

        relative = (
            path.resolve()
            .relative_to(
                PROJECT_ROOT.resolve()
            )
        )


        return (
            "./"
            +
            relative.as_posix()
        )


    except Exception:

        return path.as_posix()


def mean_numeric_dict(
    items
):

    keys = set()


    for item in items:

        if isinstance(
            item,
            dict
        ):

            keys.update(
                item.keys()
            )


    output = {}


    for key in sorted(
        keys
    ):

        values = []


        for item in items:

            if (
                not isinstance(
                    item,
                    dict
                )
                or
                item.get(
                    key
                )
                is None
            ):

                continue


            value = safe_float(
                item.get(
                    key
                ),
                np.nan
            )


            if math.isfinite(
                value
            ):

                values.append(
                    value
                )


        if values:

            output[
                key
            ] = round(

                float(
                    np.mean(
                        values
                    )
                ),

                7
            )


    return output


def deep_get(
    record,
    *paths
):

    for path in paths:

        current = record

        valid = True


        for key in path.split(
            "."
        ):

            if (
                not isinstance(
                    current,
                    dict
                )
                or
                key not in current
            ):

                valid = False

                break


            current = current[
                key
            ]


        if (
            valid
            and
            current is not None
        ):

            return current


    return None


# ============================================================
# MRT SOM
# ============================================================

STATION_FIELDS = [

    "station",
    "station_name",
    "stationName",
    "name",
    "mrt",
    "mrt_station"

]


SOM_FIELDS = [

    "somCell",
    "som_cell",
    "somCluster",
    "som_cluster",
    "bmu",
    "winner",
    "cell",
    "cluster"

]


DESCRIPTOR_FIELDS = [

    "contextDescriptor",
    "context_descriptor",
    "descriptor",
    "context",
    "profile"

]


def first_present(
    record,
    keys
):

    for key in keys:

        if (
            key in record
            and
            record.get(
                key
            )
            is not None
        ):

            return record.get(
                key
            )


    return None


def unwrap_records(
    payload
):

    if isinstance(
        payload,
        list
    ):

        return payload


    if isinstance(
        payload,
        dict
    ):

        for key in [

            "stations",
            "contexts",
            "records",
            "data",
            "items",
            "results"

        ]:

            if isinstance(
                payload.get(
                    key
                ),
                list
            ):

                return payload[
                    key
                ]


        candidate = []


        for (
            key,
            value
        ) in payload.items():

            if isinstance(
                value,
                dict
            ):

                candidate.append({

                    "station":
                        key,

                    **value

                })


        if candidate:
            return candidate


    return []


def load_mrt_contexts():

    if not MRT_CONTEXT_PATH.exists():

        print(
            "WARNING: MRT context file not found."
        )

        return []


    records = unwrap_records(

        load_json(
            MRT_CONTEXT_PATH
        )

    )


    output = []


    for record in records:

        if not isinstance(
            record,
            dict
        ):
            continue


        station = first_present(
            record,
            STATION_FIELDS
        )


        normalized = normalize_station_name(
            station
        )


        if not normalized:
            continue


        output.append({

            **record,

            "_stationNormalized":
                normalized,

            "_somCell":
                normalize_som_cell(
                    first_present(
                        record,
                        SOM_FIELDS
                    )
                ),

            "_contextDescriptor":
                str(
                    first_present(
                        record,
                        DESCRIPTOR_FIELDS
                    )
                    or
                    "mixed-context"
                ),

        })


    return output


MRT_CONTEXTS = (
    load_mrt_contexts()
)


MRT_CONTEXT_BY_STATION = {

    item[
        "_stationNormalized"
    ]:
        item

    for item in
    MRT_CONTEXTS

}


def context_for_station(
    station,
    fallback=None
):

    key = normalize_station_name(
        station
    )


    record = (
        MRT_CONTEXT_BY_STATION
        .get(
            key
        )
    )


    if record:

        return {

            "stationNormalized":
                key,

            "somCell":
                record.get(
                    "_somCell"
                ),

            "contextDescriptor":
                record.get(
                    "_contextDescriptor"
                ),

        }


    return {

        "stationNormalized":
            key,

        "somCell":
            None,

        "contextDescriptor":
            fallback
            or
            "unmatched-context",

    }


# ============================================================
# POINT / ROUTE EXTRACTION
# ============================================================

def valid_latlon(
    lat,
    lon
):

    return (

        math.isfinite(
            lat
        )

        and

        math.isfinite(
            lon
        )

        and

        1.10
        <=
        lat
        <=
        1.55

        and

        103.55
        <=
        lon
        <=
        104.15

    )


def point_from_any(
    value
):

    if isinstance(
        value,
        dict
    ):

        lat = safe_float(

            value.get(
                "lat",
                value.get(
                    "latitude"
                )
            ),

            np.nan
        )


        lon = safe_float(

            value.get(
                "lon",
                value.get(
                    "lng",
                    value.get(
                        "longitude"
                    )
                )
            ),

            np.nan
        )


        if valid_latlon(
            lat,
            lon
        ):

            return {

                "lat":
                    lat,

                "lon":
                    lon

            }


    if (
        isinstance(
            value,
            (
                list,
                tuple
            )
        )
        and
        len(
            value
        )
        >=
        2
    ):

        first = safe_float(
            value[0],
            np.nan
        )

        second = safe_float(
            value[1],
            np.nan
        )


        # GeoJSON [longitude, latitude]
        if valid_latlon(
            second,
            first
        ):

            return {

                "lat":
                    second,

                "lon":
                    first

            }


        # Also accept [latitude, longitude]
        if valid_latlon(
            first,
            second
        ):

            return {

                "lat":
                    first,

                "lon":
                    second

            }


    return None


def clean_points(
    values
):

    output = []


    if not isinstance(
        values,
        list
    ):

        return output


    for value in values:

        point = point_from_any(
            value
        )


        if not point:
            continue


        if output:

            previous = output[
                -1
            ]


            if (
                abs(
                    previous["lat"]
                    -
                    point["lat"]
                )
                <
                1e-10

                and

                abs(
                    previous["lon"]
                    -
                    point["lon"]
                )
                <
                1e-10
            ):

                continue


        output.append(
            point
        )


    return output


def geometry_points_from_record(
    record
):

    if not isinstance(
        record,
        dict
    ):

        return []


    candidates = [

        record.get(
            "points"
        ),

        deep_get(
            record,
            "route.points"
        ),

        record.get(
            "path"
        ),

        record.get(
            "polyline"
        ),

        deep_get(
            record,
            "route.path"
        ),

        deep_get(
            record,
            "route.polyline"
        ),

    ]


    geometry = record.get(
        "geometry"
    )


    if (
        isinstance(
            geometry,
            dict
        )
        and
        geometry.get(
            "type"
        )
        ==
        "LineString"
    ):

        candidates.append(
            geometry.get(
                "coordinates"
            )
        )


    route_geometry = deep_get(
        record,
        "route.geometry"
    )


    if (
        isinstance(
            route_geometry,
            dict
        )
        and
        route_geometry.get(
            "type"
        )
        ==
        "LineString"
    ):

        candidates.append(
            route_geometry.get(
                "coordinates"
            )
        )


    # IMPORTANT:
    # Your old 21 checkpoint route can be used
    # as a SOURCE polyline.
    #
    # We then resample it at 15 m.
    old_checkpoints = record.get(
        "checkpoints"
    )


    if isinstance(
        old_checkpoints,
        list
    ):

        candidates.append(
            old_checkpoints
        )


    for candidate in candidates:

        points = clean_points(
            candidate
        )


        if len(
            points
        ) >= 2:

            return points


    return []


def route_station(
    record
):

    return (

        deep_get(
            record,
            "station"
        )

        or

        deep_get(
            record,
            "origin.station"
        )

        or

        deep_get(
            record,
            "origin.name"
        )

        or

        deep_get(
            record,
            "mrt"
        )

        or

        deep_get(
            record,
            "mrtStation"
        )

        or

        "Unknown MRT"

    )


def route_destination(
    record
):

    value = (

        deep_get(
            record,
            "destination.name"
        )

        or

        deep_get(
            record,
            "home.name"
        )

        or

        deep_get(
            record,
            "target.name"
        )

        or

        deep_get(
            record,
            "toilet.name"
        )

        or

        deep_get(
            record,
            "destination"
        )

        or

        deep_get(
            record,
            "toilet"
        )

        or

        "Destination"

    )


    if isinstance(
        value,
        dict
    ):

        value = (

            value.get(
                "name"
            )

            or

            value.get(
                "id"
            )

            or

            "Destination"

        )


    return str(
        value
    )


def normalize_route_record(
    record,
    fallback_index=0,
    source=None
):

    points = geometry_points_from_record(
        record
    )


    if len(
        points
    ) < 2:

        return None


    station = str(
        route_station(
            record
        )
    )


    destination = route_destination(
        record
    )


    route_id = (

        record.get(
            "routeId"
        )

        or

        record.get(
            "route_id"
        )

        or

        record.get(
            "journeyId"
        )

        or

        record.get(
            "id"
        )

        or

        (
            f"{slugify(station)}-"
            f"{slugify(destination)}-"
            f"{fallback_index + 1}"
        )

    )


    return {

        "routeId":
            str(
                route_id
            ),

        "station":
            station,

        "stationType":
            (
                record.get(
                    "stationType"
                )

                or

                record.get(
                    "contextDescriptor"
                )

                or

                deep_get(
                    record,
                    "origin.contextDescriptor"
                )
            ),

        "destination":
            destination,

        "toilet":
            destination,

        "points":
            points,

        "source":
            (
                str(
                    source
                )
                if source
                else
                None
            )

    }


def collect_route_candidates(
    payload,
    source=None
):

    found = []


    def visit(
        value
    ):

        if isinstance(
            value,
            dict
        ):

            normalized = normalize_route_record(

                value,

                fallback_index=len(
                    found
                ),

                source=source

            )


            if normalized:

                found.append(
                    normalized
                )

                return


            for child in value.values():

                visit(
                    child
                )


        elif isinstance(
            value,
            list
        ):

            for child in value:

                visit(
                    child
                )


    visit(
        payload
    )


    return found


# ============================================================
# FIND ROUTE JSONs AUTOMATICALLY
# ============================================================

def candidate_route_files():

    priority = [

        PROJECT_ROOT
        /
        "family_route_analysis.json",

        DATA_DIR
        /
        "family_route_analysis.json",

        GENERATED_DIR
        /
        "family_route_analysis.json",

    ]


    discovered = []

    seen = set()


    for path in priority:

        if (
            path.exists()
            and
            path.resolve()
            not in seen
        ):

            discovered.append(
                path
            )

            seen.add(
                path.resolve()
            )


    excluded = {

        "image_compiler.json",

        "image_analysis_full.json",

        "latest_batch.json",

    }


    for path in PROJECT_ROOT.rglob(
        "*.json"
    ):

        if path.name in excluded:
            continue


        lower = path.name.lower()


        if (
            "route"
            not in lower
            and
            "journey"
            not in lower
        ):

            continue


        try:
            resolved = path.resolve()

        except Exception:
            continue


        if resolved in seen:
            continue


        discovered.append(
            path
        )

        seen.add(
            resolved
        )


    # Old route intelligence is usable too,
    # because its checkpoint sequence can reconstruct
    # the route geometry.
    for path in sorted(
        ROUTE_INTELLIGENCE_DIR.glob(
            "*.json"
        )
    ):

        if path.name == "latest_batch.json":
            continue


        try:
            resolved = path.resolve()

        except Exception:
            continue


        if resolved not in seen:

            discovered.append(
                path
            )

            seen.add(
                resolved
            )


    return discovered


def discover_routes():

    routes = []

    seen_ids = set()


    for path in candidate_route_files():

        try:

            payload = load_json(
                path
            )

        except Exception:
            continue


        found = collect_route_candidates(
            payload,
            source=path
        )


        for route in found:

            route_id = route[
                "routeId"
            ]


            if route_id in seen_ids:
                continue


            if STATION_FILTER:

                if (
                    STATION_FILTER.upper()
                    not in
                    route[
                        "station"
                    ].upper()
                ):

                    continue


            routes.append(
                route
            )

            seen_ids.add(
                route_id
            )


    if (
        MAX_ROUTES_TO_CRAWL
        is not None
    ):

        routes = routes[
            :int(
                MAX_ROUTES_TO_CRAWL
            )
        ]


    return routes


ROUTES_DISCOVERED = (
    discover_routes()
)


print()
print(
    "MRT SOM contexts:",
    len(
        MRT_CONTEXTS
    )
)

print(
    "Routes discovered:",
    len(
        ROUTES_DISCOVERED
    )
)


for route in (
    ROUTES_DISCOVERED[:12]
):

    print(
        "•",
        route[
            "routeId"
        ],
        "|",
        route[
            "station"
        ],
        "| source points:",
        len(
            route[
                "points"
            ]
        ),
        "|",
        route[
            "source"
        ]
    )


MRT SOM contexts: 73
Routes discovered: 192
• residential-0001__kranji-mrt | Residential Cluster 001 | source points: 81 | C:\Users\MSI\OneDrive\Desktop\family-toilet-search\data\generated\family_route_analysis.json
• residential-0002__springleaf-mrt | Residential Cluster 002 | source points: 86 | C:\Users\MSI\OneDrive\Desktop\family-toilet-search\data\generated\family_route_analysis.json
• residential-0003__khatib-mrt | Residential Cluster 003 | source points: 81 | C:\Users\MSI\OneDrive\Desktop\family-toilet-search\data\generated\family_route_analysis.json
• residential-0004__stevens-mrt | Residential Cluster 004 | source points: 177 | C:\Users\MSI\OneDrive\Desktop\family-toilet-search\data\generated\family_route_analysis.json
• residential-0005__sembawang-mrt | Residential Cluster 005 | source points: 63 | C:\Users\MSI\OneDrive\Desktop\family-toilet-search\data\generated\family_route_analysis.json
• residential-0006__springleaf-mrt | Residential Cluster 006 | source points: 33 | C:\

In [4]:
# ============================================================
# CELL 3 — DENSE ROUTE SAMPLING + LOCAL SUPPORT
# ============================================================

EARTH_RADIUS_M = 6_371_000


def haversine_m(
    lat1,
    lon1,
    lat2,
    lon2
):

    phi1 = math.radians(
        lat1
    )

    phi2 = math.radians(
        lat2
    )

    dphi = math.radians(
        lat2 - lat1
    )

    dlambda = math.radians(
        lon2 - lon1
    )


    a = (

        math.sin(
            dphi / 2
        )
        ** 2

        +

        math.cos(
            phi1
        )

        *

        math.cos(
            phi2
        )

        *

        math.sin(
            dlambda / 2
        )
        ** 2

    )


    return (

        2

        *

        EARTH_RADIUS_M

        *

        math.asin(
            math.sqrt(
                a
            )
        )

    )


def route_length_m(
    points
):

    return sum(

        haversine_m(

            points[
                index
            ][
                "lat"
            ],

            points[
                index
            ][
                "lon"
            ],

            points[
                index + 1
            ][
                "lat"
            ],

            points[
                index + 1
            ][
                "lon"
            ],

        )

        for index in range(
            len(
                points
            )
            -
            1
        )

    )


def interpolate_point(
    first,
    second,
    fraction
):

    return {

        "lat":
            first[
                "lat"
            ]

            +

            (
                second[
                    "lat"
                ]
                -
                first[
                    "lat"
                ]
            )

            *
            fraction,

        "lon":
            first[
                "lon"
            ]

            +

            (
                second[
                    "lon"
                ]
                -
                first[
                    "lon"
                ]
            )

            *
            fraction,

    }


def sample_route(
    points,
    spacing_m=
        CHECKPOINT_SPACING_M
):

    if len(
        points
    ) < 2:

        return []


    segment_lengths = []

    cumulative = [
        0.0
    ]


    for index in range(
        len(
            points
        )
        -
        1
    ):

        length = haversine_m(

            points[
                index
            ][
                "lat"
            ],

            points[
                index
            ][
                "lon"
            ],

            points[
                index + 1
            ][
                "lat"
            ],

            points[
                index + 1
            ][
                "lon"
            ],

        )


        segment_lengths.append(
            length
        )


        cumulative.append(

            cumulative[
                -1
            ]
            +
            length

        )


    total = cumulative[
        -1
    ]


    targets = list(

        np.arange(
            0.0,
            total,
            spacing_m
        )

    )


    if (
        not targets
        or
        abs(
            targets[-1]
            -
            total
        )
        >
        1.0
    ):

        targets.append(
            total
        )


    checkpoints = []

    segment_index = 0


    for (
        checkpoint_index,
        target
    ) in enumerate(
        targets
    ):

        while (

            segment_index
            <
            len(
                segment_lengths
            )
            -
            1

            and

            cumulative[
                segment_index + 1
            ]
            <
            target

        ):

            segment_index += 1


        start_distance = (
            cumulative[
                segment_index
            ]
        )


        segment_length = max(

            segment_lengths[
                segment_index
            ],

            1e-9

        )


        fraction = (

            target
            -
            start_distance

        ) / segment_length


        fraction = max(

            0.0,

            min(
                1.0,
                fraction
            )

        )


        point = interpolate_point(

            points[
                segment_index
            ],

            points[
                segment_index + 1
            ],

            fraction

        )


        checkpoints.append({

            "checkpointIndex":
                checkpoint_index,

            "distanceAlongRoute_m":
                round(
                    float(
                        target
                    ),
                    2
                ),

            "lat":
                round(
                    point[
                        "lat"
                    ],
                    7
                ),

            "lon":
                round(
                    point[
                        "lon"
                    ],
                    7
                ),

        })


    return checkpoints


# ============================================================
# LOCAL FAMILY SUPPORT
# ============================================================

def extract_support_records(
    payload
):

    output = []


    def visit(
        value
    ):

        if isinstance(
            value,
            dict
        ):

            point = point_from_any(
                value
            )


            if point:

                categories = (

                    value.get(
                        "categories"
                    )

                    or

                    value.get(
                        "category"
                    )

                    or

                    value.get(
                        "type"
                    )

                    or

                    value.get(
                        "supportType"
                    )

                    or

                    []

                )


                if isinstance(
                    categories,
                    str
                ):

                    categories = [
                        categories
                    ]


                output.append({

                    "id":
                        str(
                            value.get(
                                "id"
                            )
                            or
                            value.get(
                                "name"
                            )
                            or
                            len(
                                output
                            )
                        ),

                    "name":
                        value.get(
                            "name"
                        ),

                    "lat":
                        point[
                            "lat"
                        ],

                    "lon":
                        point[
                            "lon"
                        ],

                    "categories":
                        [
                            str(
                                item
                            )
                            for item in
                            categories
                        ],

                })


            for child in value.values():

                visit(
                    child
                )


        elif isinstance(
            value,
            list
        ):

            for child in value:

                visit(
                    child
                )


    visit(
        payload
    )


    deduped = {}


    for item in output:

        key = (

            round(
                item[
                    "lat"
                ],
                6
            ),

            round(
                item[
                    "lon"
                ],
                6
            ),

            item[
                "id"
            ]

        )


        deduped[
            key
        ] = item


    return list(
        deduped.values()
    )


def load_local_support():

    paths = [

        GENERATED_DIR
        /
        "family_support_library.json",

        DATA_DIR
        /
        "family_support_library.json",

        PROJECT_ROOT
        /
        "family_support_library.json",

    ]


    for path in paths:

        if not path.exists():
            continue


        try:

            records = extract_support_records(

                load_json(
                    path
                )

            )


            print(
                "Local support records:",
                len(
                    records
                ),
                "|",
                path
            )


            return records


        except Exception as error:

            print(
                "Could not load local support:",
                error
            )


    print(
        "Local support records: 0"
    )


    return []


LOCAL_SUPPORT = (
    load_local_support()
)


def support_near_checkpoint(
    checkpoint,
    radius_m=80
):

    nearby = []


    for item in LOCAL_SUPPORT:

        distance = haversine_m(

            checkpoint[
                "lat"
            ],

            checkpoint[
                "lon"
            ],

            item[
                "lat"
            ],

            item[
                "lon"
            ],

        )


        if distance <= radius_m:

            nearby.append({

                **item,

                "distance_m":
                    round(
                        distance,
                        1
                    )

            })


    return nearby


print(
    "Dense 15 m route sampler ready."
)

Local support records: 1167 | C:\Users\MSI\OneDrive\Desktop\family-toilet-search\data\generated\family_support_library.json
Dense 15 m route sampler ready.


In [5]:
# ============================================================
# CELL 4 — OSM + MAPILLARY CRAWLERS
# ============================================================

def bbox_for_points(
    points,
    padding_m=90
):

    lats = [
        point["lat"]
        for point in points
    ]

    lons = [
        point["lon"]
        for point in points
    ]


    mean_lat = float(
        np.mean(
            lats
        )
    )


    lat_pad = (
        padding_m
        /
        111_320
    )


    lon_pad = (

        padding_m

        /

        (
            111_320

            *

            max(
                math.cos(
                    math.radians(
                        mean_lat
                    )
                ),
                0.2
            )
        )

    )


    return {

        "south":
            min(
                lats
            )
            -
            lat_pad,

        "west":
            min(
                lons
            )
            -
            lon_pad,

        "north":
            max(
                lats
            )
            +
            lat_pad,

        "east":
            max(
                lons
            )
            +
            lon_pad,

    }


# ============================================================
# OSM
# ============================================================

def overpass_route_features(
    points
):

    bbox = bbox_for_points(
        points,
        OSM_ROUTE_BUFFER_M
    )


    bbox_text = (

        f'{bbox["south"]},'
        f'{bbox["west"]},'
        f'{bbox["north"]},'
        f'{bbox["east"]}'

    )


    query = f"""

    [out:json][timeout:{OVERPASS_TIMEOUT}];

    (

      nwr["highway"]({bbox_text});

      nwr["amenity"]({bbox_text});

      nwr["leisure"]({bbox_text});

      nwr["public_transport"]({bbox_text});

      nwr["shop"]({bbox_text});

      nwr["covered"]({bbox_text});

      nwr["wheelchair"]({bbox_text});

      nwr["tactile_paving"]({bbox_text});

      nwr["kerb"]({bbox_text});

      nwr["lit"]({bbox_text});

    );

    out center tags;

    """


    try:

        response = requests.post(

            OVERPASS_URL,

            data={
                "data":
                    query
            },

            timeout=
                OVERPASS_TIMEOUT
                +
                15

        )


        response.raise_for_status()


        elements = (
            response.json()
            .get(
                "elements",
                []
            )
        )


    except Exception as error:

        print(
            "OSM warning:",
            error
        )

        return []


    output = []


    for element in elements:

        tags = (
            element.get(
                "tags"
            )
            or
            {}
        )


        lat = element.get(
            "lat"
        )

        lon = element.get(
            "lon"
        )


        if (
            lat is None
            or
            lon is None
        ):

            center = (
                element.get(
                    "center"
                )
                or
                {}
            )


            lat = center.get(
                "lat"
            )

            lon = center.get(
                "lon"
            )


        lat = safe_float(
            lat,
            np.nan
        )

        lon = safe_float(
            lon,
            np.nan
        )


        if not valid_latlon(
            lat,
            lon
        ):

            continue


        output.append({

            "id":
                (
                    f'{element.get("type", "x")}/'
                    f'{element.get("id")}'
                ),

            "lat":
                lat,

            "lon":
                lon,

            "tags":
                tags,

        })


    return output


def osm_categories(
    tags
):

    tags = (
        tags
        or
        {}
    )


    categories = set()


    highway = str(
        tags.get(
            "highway"
        )
        or
        ""
    ).lower()


    amenity = str(
        tags.get(
            "amenity"
        )
        or
        ""
    ).lower()


    leisure = str(
        tags.get(
            "leisure"
        )
        or
        ""
    ).lower()


    public_transport = str(
        tags.get(
            "public_transport"
        )
        or
        ""
    ).lower()


    if highway == "crossing":
        categories.add(
            "crossing"
        )


    if highway in {
        "footway",
        "pedestrian",
        "path"
    }:

        categories.add(
            "footway"
        )


    if highway == "steps":

        categories.add(
            "steps"
        )


    if (
        highway == "bus_stop"
        or
        public_transport
        in {
            "platform",
            "stop_position"
        }
    ):

        categories.add(
            "bus_stop"
        )


    if amenity == "bench":

        categories.add(
            "bench"
        )


    if (
        amenity == "shelter"
        or
        str(
            tags.get(
                "shelter"
            )
            or
            ""
        ).lower()
        ==
        "yes"
    ):

        categories.add(
            "shelter"
        )


    if amenity == "toilets":

        categories.add(
            "toilet"
        )


    if amenity == "drinking_water":

        categories.add(
            "drinking_water"
        )


    if amenity in {
        "clinic",
        "hospital",
        "doctors",
        "pharmacy"
    }:

        categories.add(
            "healthcare"
        )


    if amenity in {
        "childcare",
        "kindergarten"
    }:

        categories.add(
            "childcare"
        )


    if amenity in {
        "restaurant",
        "cafe",
        "fast_food",
        "food_court"
    }:

        categories.add(
            "food"
        )


    if leisure == "playground":

        categories.add(
            "playground"
        )


    if leisure in {
        "park",
        "garden"
    }:

        categories.add(
            "park"
        )


    if tags.get(
        "shop"
    ):

        categories.add(
            "retail"
        )


    if str(
        tags.get(
            "tactile_paving"
        )
        or
        ""
    ).lower() == "yes":

        categories.add(
            "tactile_paving"
        )


    if str(
        tags.get(
            "kerb"
        )
        or
        ""
    ).lower() in {
        "lowered",
        "flush"
    }:

        categories.add(
            "kerb_lowered"
        )


    if str(
        tags.get(
            "wheelchair"
        )
        or
        ""
    ).lower() in {
        "yes",
        "designated"
    }:

        categories.add(
            "wheelchair"
        )


    if str(
        tags.get(
            "lit"
        )
        or
        ""
    ).lower() == "yes":

        categories.add(
            "lit"
        )


    if str(
        tags.get(
            "covered"
        )
        or
        ""
    ).lower() == "yes":

        categories.add(
            "covered"
        )


    crossing = str(
        tags.get(
            "crossing"
        )
        or
        ""
    ).lower()


    if crossing in {
        "traffic_signals",
        "signals"
    }:

        categories.add(
            "traffic_signal"
        )


    return sorted(
        categories
    )


def osm_near_checkpoint(
    checkpoint,
    features,
    radius_m=
        OSM_CHECKPOINT_RADIUS_M
):

    nearby = []

    counts = Counter()


    for feature in features:

        distance = haversine_m(

            checkpoint[
                "lat"
            ],

            checkpoint[
                "lon"
            ],

            feature[
                "lat"
            ],

            feature[
                "lon"
            ],

        )


        if distance > radius_m:
            continue


        categories = osm_categories(
            feature.get(
                "tags"
            )
        )


        counts.update(
            categories
        )


        nearby.append({

            "id":
                feature[
                    "id"
                ],

            "distance_m":
                round(
                    distance,
                    1
                ),

            "categories":
                categories,

            "tags":
                feature.get(
                    "tags"
                )
                or
                {},

        })


    return {

        "count":
            len(
                nearby
            ),

        "summary":
            dict(
                sorted(
                    counts.items()
                )
            ),

        "features":
            nearby,

    }


# ============================================================
# MAPILLARY
# ============================================================

def circular_difference(
    first,
    second
):

    try:

        first = float(
            first
        ) % 360

        second = float(
            second
        ) % 360


    except Exception:

        return 180.0


    difference = abs(
        first
        -
        second
    )


    return min(
        difference,
        360 - difference
    )


def mapillary_route_candidates(
    points
):

    if not MAPILLARY_ACCESS_TOKEN:

        raise RuntimeError(

            "Mapillary token was not found. "
            "Keep your token in the old route notebook "
            "or set MAPILLARY_ACCESS_TOKEN."

        )


    bbox = bbox_for_points(
        points,
        MAPILLARY_RADIUS_M
    )


    bbox_text = ",".join(

        str(
            value
        )

        for value in [

            bbox[
                "west"
            ],

            bbox[
                "south"
            ],

            bbox[
                "east"
            ],

            bbox[
                "north"
            ],

        ]

    )


    params = {

        "access_token":
            MAPILLARY_ACCESS_TOKEN,

        "bbox":
            bbox_text,

        "fields":
            (
                "id,"
                "geometry,"
                "computed_geometry,"
                "captured_at,"
                "compass_angle,"
                "sequence,"
                "thumb_1024_url"
            ),

        "limit":
            MAPILLARY_PAGE_LIMIT,

    }


    candidates = []

    next_url = (
        MAPILLARY_IMAGES_URL
    )

    page = 0


    while (

        next_url

        and

        len(
            candidates
        )
        <
        MAPILLARY_ROUTE_CANDIDATE_LIMIT

        and

        page < 4

    ):

        page += 1


        if page == 1:

            response = requests.get(

                next_url,

                params=params,

                timeout=
                    REQUEST_TIMEOUT

            )


        else:

            response = requests.get(

                next_url,

                timeout=
                    REQUEST_TIMEOUT

            )


        response.raise_for_status()


        payload = response.json()


        for item in payload.get(
            "data",
            []
        ):

            geometry = (

                item.get(
                    "computed_geometry"
                )

                or

                item.get(
                    "geometry"
                )

                or

                {}

            )


            coordinates = (
                geometry.get(
                    "coordinates"
                )
                or
                []
            )


            if len(
                coordinates
            ) < 2:

                continue


            lon = safe_float(
                coordinates[0],
                np.nan
            )

            lat = safe_float(
                coordinates[1],
                np.nan
            )


            if not valid_latlon(
                lat,
                lon
            ):

                continue


            candidates.append({

                "id":
                    str(
                        item.get(
                            "id"
                        )
                    ),

                "lat":
                    lat,

                "lon":
                    lon,

                "captured_at":
                    item.get(
                        "captured_at"
                    ),

                "compass_angle":
                    item.get(
                        "compass_angle"
                    ),

                "sequence":
                    item.get(
                        "sequence"
                    ),

                "thumb_1024_url":
                    item.get(
                        "thumb_1024_url"
                    ),

            })


            if (
                len(
                    candidates
                )
                >=
                MAPILLARY_ROUTE_CANDIDATE_LIMIT
            ):

                break


        next_url = (

            payload.get(
                "paging"
            )
            or
            {}

        ).get(
            "next"
        )


    return candidates


def assign_mapillary_to_checkpoints(
    checkpoints,
    candidates
):

    used_image_ids = set()

    output = []


    for checkpoint in checkpoints:

        ranked = []


        for image in candidates:

            if (
                image[
                    "id"
                ]
                in
                used_image_ids
            ):

                continue


            distance = haversine_m(

                checkpoint[
                    "lat"
                ],

                checkpoint[
                    "lon"
                ],

                image[
                    "lat"
                ],

                image[
                    "lon"
                ],

            )


            if distance <= MAPILLARY_RADIUS_M:

                ranked.append({

                    **image,

                    "distance_m":
                        round(
                            distance,
                            2
                        )

                })


        ranked.sort(

            key=lambda item:
                item[
                    "distance_m"
                ]

        )


        selected = []


        if ranked:

            selected.append(
                ranked[0]
            )


        # Prefer different camera headings.
        for image in ranked[
            1:
        ]:

            if (
                len(
                    selected
                )
                >=
                MAPILLARY_IMAGES_PER_CHECKPOINT
            ):

                break


            heading = image.get(
                "compass_angle"
            )


            if heading is None:
                continue


            if all(

                previous.get(
                    "compass_angle"
                )
                is None

                or

                circular_difference(

                    heading,

                    previous.get(
                        "compass_angle"
                    )

                )
                >=
                MAPILLARY_MIN_HEADING_SEPARATION_DEG

                for previous in
                selected

            ):

                selected.append(
                    image
                )


        # Fill any remaining slots by proximity.
        selected_ids = {

            item[
                "id"
            ]

            for item in
            selected

        }


        for image in ranked:

            if (
                len(
                    selected
                )
                >=
                MAPILLARY_IMAGES_PER_CHECKPOINT
            ):

                break


            if image[
                "id"
            ] in selected_ids:

                continue


            selected.append(
                image
            )

            selected_ids.add(
                image[
                    "id"
                ]
            )


        for image in selected:

            used_image_ids.add(
                image[
                    "id"
                ]
            )


        output.append({

            "available":
                bool(
                    selected
                ),

            "reason":
                (
                    None
                    if selected
                    else
                    "no_nearby_unique_imagery"
                ),

            "candidateCount":
                len(
                    ranked
                ),

            "images":
                selected,

        })


    return output


print(
    "OSM + Mapillary crawlers ready."
)

OSM + Mapillary crawlers ready.


In [6]:
# ============================================================
# CELL 5 — FULL ROUTE CRAWL
# ============================================================

def crawl_route(
    route,
    force=False
):

    started = time.time()


    route_points = (
        route[
            "points"
        ]
    )


    checkpoints = sample_route(

        route_points,

        spacing_m=
            CHECKPOINT_SPACING_M

    )


    route_distance = route_length_m(
        route_points
    )


    print()
    print(
        "--------------------------------------------------"
    )

    print(
        route[
            "station"
        ],
        "→",
        route[
            "destination"
        ]
    )

    print(
        "Route:",
        route[
            "routeId"
        ]
    )

    print(
        "Distance:",
        round(
            route_distance,
            1
        ),
        "m"
    )

    print(
        "Dense checkpoints:",
        len(
            checkpoints
        )
    )

    print(
        "--------------------------------------------------"
    )


    # ========================================================
    # OSM
    # ========================================================

    update_status(

        "osm",

        (
            f'{route["station"]}: '
            "crawling OSM corridor."
        )

    )


    osm_features = (
        overpass_route_features(
            route_points
        )
    )


    print(
        "OSM features:",
        len(
            osm_features
        )
    )


    # ========================================================
    # MAPILLARY
    # ========================================================

    update_status(

        "mapillary",

        (
            f'{route["station"]}: '
            "crawling Mapillary route bbox."
        )

    )


    mapillary_candidates = (
        mapillary_route_candidates(
            route_points
        )
    )


    print(
        "Mapillary candidates:",
        len(
            mapillary_candidates
        )
    )


    mapillary_by_checkpoint = (
        assign_mapillary_to_checkpoints(

            checkpoints,

            mapillary_candidates

        )
    )


    # ========================================================
    # CHECKPOINT RECORDS
    # ========================================================

    checkpoint_records = []


    for (
        index,
        checkpoint
    ) in enumerate(
        checkpoints
    ):

        local_support = (
            support_near_checkpoint(
                checkpoint,
                radius_m=80
            )
        )


        local_summary = Counter()


        for item in local_support:

            local_summary.update(

                item.get(
                    "categories"
                )
                or
                []

            )


        checkpoint_records.append({

            **checkpoint,

            "localSupport": {

                "count":
                    len(
                        local_support
                    ),

                "summary":
                    dict(
                        sorted(
                            local_summary.items()
                        )
                    ),

                "nodes":
                    local_support,

            },


            "osm":
                osm_near_checkpoint(

                    checkpoint,

                    osm_features,

                    radius_m=
                        OSM_CHECKPOINT_RADIUS_M

                ),


            "imagery":
                (
                    mapillary_by_checkpoint[
                        index
                    ]

                    if index
                    <
                    len(
                        mapillary_by_checkpoint
                    )

                    else

                    {

                        "available":
                            False,

                        "reason":
                            "not_crawled",

                        "images":
                            []

                    }
                )

        })


    context = context_for_station(

        route.get(
            "station"
        ),

        route.get(
            "stationType"
        )

    )


    result = {

        "version":
            6,

        "routeId":
            route[
                "routeId"
            ],

        "station":
            route[
                "station"
            ],

        "stationType":
            route.get(
                "stationType"
            ),

        "destination":
            route[
                "destination"
            ],

        "toilet":
            route.get(
                "toilet"
            ),

        "somCell":
            context.get(
                "somCell"
            ),

        "contextDescriptor":
            context.get(
                "contextDescriptor"
            ),

        "routePoints":
            route_points,


        "routeSummary": {

            "distance_m":
                round(
                    route_distance,
                    2
                ),

            "checkpointSpacing_m":
                CHECKPOINT_SPACING_M,

            "checkpointCount":
                len(
                    checkpoints
                ),

            "checkpointCountWithImages":
                sum(

                    1

                    for item in
                    checkpoint_records

                    if (
                        item.get(
                            "imagery"
                        )
                        or
                        {}
                    ).get(
                        "images"
                    )

                ),

            "mapillaryRouteCandidateCount":
                len(
                    mapillary_candidates
                ),

            "osmFeatureCount":
                len(
                    osm_features
                ),

            "analysisSeconds":
                round(
                    time.time()
                    -
                    started,
                    2
                ),

        },


        "checkpoints":
            checkpoint_records,

        "source":
            route.get(
                "source"
            ),

    }


    safe_json_dump(

        ROUTE_INTELLIGENCE_DIR

        /

        (
            f'{slugify(route["routeId"])}'
            ".json"
        ),

        result

    )


    return result


def crawl_routes(
    routes,
    force=False
):

    crawled = []


    for (
        index,
        route
    ) in enumerate(
        routes,
        start=1
    ):

        update_status(

            "route-crawl",

            (
                f"Route {index}/"
                f"{len(routes)}: "
                f'{route["station"]}'
            ),

            processed=
                index - 1,

            total=
                len(
                    routes
                )

        )


        crawled.append(

            crawl_route(
                route,
                force=force
            )

        )


    safe_json_dump(

        ROUTE_INTELLIGENCE_DIR
        /
        "latest_batch.json",

        {

            "version":
                6,

            "createdAt":
                datetime.now()
                .isoformat(
                    timespec="seconds"
                ),

            "results":
                crawled,

        }

    )


    return crawled


# ============================================================
# TURN CRAWLED CHECKPOINTS INTO IMAGE REFERENCES
# ============================================================

def extract_image_references(
    routes
):

    references = []


    for route in routes:

        context = context_for_station(

            route.get(
                "station"
            ),

            route.get(
                "stationType"
            )

        )


        som_cell = context.get(
            "somCell"
        )


        mrt_cluster = (

            f"SOM-{som_cell}"

            if som_cell is not None

            else

            (
                "CONTEXT-"
                +
                slugify(
                    context.get(
                        "contextDescriptor"
                    )
                )
            )

        )


        for checkpoint in route.get(
            "checkpoints",
            []
        ):

            imagery = (
                checkpoint.get(
                    "imagery"
                )
                or
                {}
            )


            for (
                rank,
                image
            ) in enumerate(

                (
                    imagery.get(
                        "images"
                    )
                    or
                    []
                )[
                    :
                    MAPILLARY_IMAGES_PER_CHECKPOINT
                ],

                start=1

            ):

                image_id = str(
                    image.get(
                        "id"
                    )
                    or
                    ""
                ).strip()


                if not image_id:
                    continue


                references.append({

                    "imageId":
                        image_id,

                    "routeId":
                        route.get(
                            "routeId"
                        ),

                    "station":
                        route.get(
                            "station"
                        ),

                    "stationNormalized":
                        context.get(
                            "stationNormalized"
                        ),

                    "stationType":
                        route.get(
                            "stationType"
                        ),

                    "somCell":
                        som_cell,

                    "mrtCluster":
                        mrt_cluster,

                    "contextDescriptor":
                        context.get(
                            "contextDescriptor"
                        ),

                    "destination":
                        route.get(
                            "destination"
                        ),

                    "checkpointIndex":
                        checkpoint.get(
                            "checkpointIndex"
                        ),

                    "distanceAlongRoute_m":
                        checkpoint.get(
                            "distanceAlongRoute_m"
                        ),

                    "checkpointLat":
                        checkpoint.get(
                            "lat"
                        ),

                    "checkpointLon":
                        checkpoint.get(
                            "lon"
                        ),

                    "imageLat":
                        image.get(
                            "lat"
                        ),

                    "imageLon":
                        image.get(
                            "lon"
                        ),

                    "distanceToCheckpoint_m":
                        image.get(
                            "distance_m"
                        ),

                    "capturedAt":
                        image.get(
                            "captured_at"
                        ),

                    "compassAngle":
                        image.get(
                            "compass_angle"
                        ),

                    "thumbUrl":
                        image.get(
                            "thumb_1024_url"
                        ),

                    "rank":
                        rank,

                })


    deduped = {}


    for item in references:

        key = (

            item[
                "routeId"
            ],

            item[
                "imageId"
            ]

        )


        deduped[
            key
        ] = item


    output = list(
        deduped.values()
    )


    output.sort(

        key=lambda item: (

            str(
                item.get(
                    "routeId"
                )
            ),

            safe_float(
                item.get(
                    "distanceAlongRoute_m"
                )
            ),

            int(
                item.get(
                    "rank"
                )
                or
                0
            ),

        )

    )


    return output


print(
    "Full route crawler ready."
)

Full route crawler ready.


In [7]:
# ============================================================
# CELL 6 — MAPILLARY DOWNLOAD + IMAGE QUALITY CONTROL
# ============================================================

def image_storage_path(
    reference
):

    cluster_slug = slugify(
        reference.get(
            "mrtCluster"
        )
        or
        "unknown-cluster"
    )


    station_slug = slugify(
        reference.get(
            "stationNormalized"
        )
        or
        "unknown-station"
    )


    route_slug = slugify(
        reference.get(
            "routeId"
        )
        or
        "unknown-route"
    )


    checkpoint = int(
        reference.get(
            "checkpointIndex"
        )
        or
        0
    )


    folder = (

        STREET_IMAGE_DIR

        /

        "by_mrt_cluster"

        /

        cluster_slug

        /

        station_slug

        /

        route_slug

        /

        f"cp-{checkpoint:04d}"

    )


    folder.mkdir(
        parents=True,
        exist_ok=True
    )


    return (

        folder

        /

        f'{reference["imageId"]}.jpg'

    )


def mapillary_image_details(
    image_id
):

    response = requests.get(

        (
            f"{MAPILLARY_GRAPH_URL}/"
            f"{image_id}"
        ),

        params={

            "access_token":
                MAPILLARY_ACCESS_TOKEN,

            "fields":
                (
                    "id,"
                    "thumb_1024_url,"
                    "captured_at,"
                    "compass_angle,"
                    "geometry,"
                    "computed_geometry,"
                    "sequence"
                )

        },

        timeout=
            REQUEST_TIMEOUT

    )


    response.raise_for_status()


    return response.json()


def download_one_image(
    reference,
    force=False
):

    image_path = (
        image_storage_path(
            reference
        )
    )


    if (
        image_path.exists()
        and
        not force
    ):

        return {

            **reference,

            "imagePath":
                str(
                    image_path
                ),

            "originalImage":
                to_web_path(
                    image_path
                ),

            "downloadStatus":
                "cached",

        }


    image_url = reference.get(
        "thumbUrl"
    )


    if not image_url:

        details = (
            mapillary_image_details(
                reference[
                    "imageId"
                ]
            )
        )


        image_url = details.get(
            "thumb_1024_url"
        )


    if not image_url:

        raise RuntimeError(
            "Mapillary thumbnail URL missing."
        )


    response = requests.get(

        image_url,

        timeout=
            REQUEST_TIMEOUT

    )


    response.raise_for_status()


    image = Image.open(

        io.BytesIO(
            response.content
        )

    ).convert(
        "RGB"
    )


    image.save(

        image_path,

        format="JPEG",

        quality=92

    )


    return {

        **reference,

        "imagePath":
            str(
                image_path
            ),

        "originalImage":
            to_web_path(
                image_path
            ),

        "downloadStatus":
            "downloaded",

    }


# ============================================================
# KEEP ROUTES BALANCED
# ============================================================

def balanced_image_queue(
    references,
    limit
):

    if len(
        references
    ) <= limit:

        return references


    groups = defaultdict(
        list
    )


    for item in references:

        groups[
            item.get(
                "routeId"
            )
            or
            "UNKNOWN"
        ].append(
            item
        )


    for items in groups.values():

        items.sort(

            key=lambda item:
                safe_float(
                    item.get(
                        "distanceAlongRoute_m"
                    )
                )

        )


    selected = []


    positions = {

        key:
            0

        for key in
        groups

    }


    while len(
        selected
    ) < limit:

        progressed = False


        for key in sorted(
            groups
        ):

            position = positions[
                key
            ]


            if position >= len(
                groups[
                    key
                ]
            ):

                continue


            selected.append(

                groups[
                    key
                ][
                    position
                ]

            )


            positions[
                key
            ] += 1


            progressed = True


            if len(
                selected
            ) >= limit:

                break


        if not progressed:
            break


    return selected


def download_images(
    references,
    force=False
):

    results = []

    errors = []


    with ThreadPoolExecutor(

        max_workers=
            DOWNLOAD_WORKERS

    ) as executor:

        futures = {

            executor.submit(

                download_one_image,

                item,

                force

            ):
                item

            for item in
            references

        }


        for future in as_completed(
            futures
        ):

            reference = futures[
                future
            ]


            try:

                results.append(
                    future.result()
                )


            except Exception as error:

                errors.append({

                    "imageId":
                        reference.get(
                            "imageId"
                        ),

                    "routeId":
                        reference.get(
                            "routeId"
                        ),

                    "checkpointIndex":
                        reference.get(
                            "checkpointIndex"
                        ),

                    "error":
                        str(
                            error
                        ),

                })


    order = {

        (
            item[
                "routeId"
            ],
            item[
                "imageId"
            ]
        ):
            index

        for (
            index,
            item
        ) in enumerate(
            references
        )

    }


    results.sort(

        key=lambda item:
            order.get(

                (
                    item[
                        "routeId"
                    ],
                    item[
                        "imageId"
                    ]
                ),

                999999

            )

    )


    return (
        results,
        errors
    )


# ============================================================
# QUALITY CONTROL
# ============================================================

def average_hash(
    image,
    hash_size=8
):

    gray = (

        ImageOps.grayscale(
            image
        )

        .resize(

            (
                hash_size,
                hash_size
            ),

            Image.Resampling.LANCZOS

        )

    )


    values = np.asarray(
        gray,
        dtype=np.float32
    )


    threshold = float(
        values.mean()
    )


    bits = (
        values
        >=
        threshold
    )


    return "".join(

        "1"
        if bit
        else
        "0"

        for bit in
        bits.flatten()

    )


def hamming_distance(
    first,
    second
):

    if (
        not first
        or
        not second
    ):

        return 999


    return sum(

        first_bit
        !=
        second_bit

        for (
            first_bit,
            second_bit
        ) in zip(
            first,
            second
        )

    )


def image_quality_metrics(
    image_path
):

    image = Image.open(
        image_path
    ).convert(
        "RGB"
    )


    gray = np.asarray(

        ImageOps.grayscale(
            image
        ),

        dtype=np.float32

    )


    dx = np.diff(
        gray,
        axis=1
    )


    dy = np.diff(
        gray,
        axis=0
    )


    return {

        "width":
            image.width,

        "height":
            image.height,

        "brightness":
            round(
                float(
                    gray.mean()
                ),
                3
            ),

        "contrast":
            round(
                float(
                    gray.std()
                ),
                3
            ),

        "sharpness":
            round(

                float(

                    (
                        np.mean(
                            np.abs(
                                dx
                            )
                        )

                        +

                        np.mean(
                            np.abs(
                                dy
                            )
                        )

                    )

                    /
                    2

                ),

                3
            ),

        "averageHash":
            average_hash(
                image
            ),

    }


def quality_filter_images(
    downloaded
):

    accepted = []

    rejected = []


    hashes_by_checkpoint = defaultdict(
        list
    )


    for item in downloaded:

        try:

            quality = image_quality_metrics(
                item[
                    "imagePath"
                ]
            )


            reason = None


            if (

                quality[
                    "width"
                ]
                <
                MIN_IMAGE_WIDTH

                or

                quality[
                    "height"
                ]
                <
                MIN_IMAGE_HEIGHT

            ):

                reason = (
                    "resolution_too_low"
                )


            elif (
                quality[
                    "brightness"
                ]
                <
                MIN_BRIGHTNESS
            ):

                reason = (
                    "too_dark"
                )


            elif (
                quality[
                    "brightness"
                ]
                >
                MAX_BRIGHTNESS
            ):

                reason = (
                    "too_bright"
                )


            elif (
                quality[
                    "contrast"
                ]
                <
                MIN_CONTRAST
            ):

                reason = (
                    "contrast_too_low"
                )


            elif (
                quality[
                    "sharpness"
                ]
                <
                MIN_SHARPNESS
            ):

                reason = (
                    "too_blurry"
                )


            if reason:

                rejected.append({

                    **item,

                    "quality":
                        quality,

                    "rejectionReason":
                        reason,

                })

                continue


            # Only reject visual duplicates
            # WITHIN THE SAME CHECKPOINT.
            key = (

                item.get(
                    "routeId"
                ),

                item.get(
                    "checkpointIndex"
                )

            )


            duplicate = None


            for previous in (
                hashes_by_checkpoint[
                    key
                ]
            ):

                if (

                    hamming_distance(

                        quality[
                            "averageHash"
                        ],

                        previous[
                            "hash"
                        ]

                    )

                    <=
                    DUPLICATE_HASH_DISTANCE

                ):

                    duplicate = previous[
                        "imageId"
                    ]

                    break


            if duplicate:

                rejected.append({

                    **item,

                    "quality":
                        quality,

                    "rejectionReason":
                        "checkpoint_visual_duplicate",

                    "duplicateOf":
                        duplicate,

                })

                continue


            hashes_by_checkpoint[
                key
            ].append({

                "imageId":
                    item[
                        "imageId"
                    ],

                "hash":
                    quality[
                        "averageHash"
                    ],

            })


            accepted.append({

                **item,

                "quality":
                    quality,

            })


        except Exception as error:

            rejected.append({

                **item,

                "rejectionReason":
                    "quality_check_failed",

                "error":
                    str(
                        error
                    ),

            })


    return (
        accepted,
        rejected
    )


print(
    "Image downloader + QC ready."
)

Image downloader + QC ready.


In [8]:
# ============================================================
# CELL 7 — SEGFORMER + VISUAL FEATURES + K-MEANS
# ============================================================

warnings.filterwarnings(
    "ignore",
    category=UserWarning
)


SEGFORMER_PROCESSOR = None

SEGFORMER_MODEL = None

SEGFORMER_GROUP_MAP = None


VISUAL_GROUP_RULES = {

    "pedestrian_surface": [
        "sidewalk",
        "pavement",
        "footpath",
        "path",
        "walkway"
    ],

    "road": [
        "road",
        "route",
        "street"
    ],

    "vegetation": [
        "tree",
        "grass",
        "plant",
        "flower",
        "field",
        "earth"
    ],

    "building": [
        "building",
        "house",
        "skyscraper"
    ],

    "sky": [
        "sky"
    ],

    "vehicle": [
        "car",
        "truck",
        "bus",
        "van",
        "motorbike",
        "motorcycle",
        "bicycle"
    ],

    "person": [
        "person"
    ],

    "barrier": [
        "fence",
        "wall",
        "railing",
        "guard rail"
    ],

    "street_furniture": [
        "bench",
        "signboard",
        "pole",
        "streetlight",
        "traffic light"
    ],

    "stairs": [
        "stairway",
        "stairs",
        "step"
    ],

    "shade_structure": [
        "awning",
        "canopy",
        "tent",
        "umbrella"
    ],

    "water": [
        "water",
        "river",
        "sea",
        "lake",
        "pool"
    ],

}


def build_group_map(
    model
):

    output = {}


    for (
        class_id,
        raw_label
    ) in (
        model.config
        .id2label
        .items()
    ):

        label = (

            str(
                raw_label
                or
                ""
            )

            .lower()

            .replace(
                "_",
                " "
            )

        )


        group = (
            "other"
        )


        for (
            candidate,
            keywords
        ) in (
            VISUAL_GROUP_RULES.items()
        ):

            if any(

                keyword in label

                for keyword in
                keywords

            ):

                group = candidate

                break


        output[
            int(
                class_id
            )
        ] = group


    return output


def load_segformer():

    global SEGFORMER_PROCESSOR

    global SEGFORMER_MODEL

    global SEGFORMER_GROUP_MAP


    if (
        SEGFORMER_PROCESSOR
        is not None
    ):

        return (

            SEGFORMER_PROCESSOR,

            SEGFORMER_MODEL,

            SEGFORMER_GROUP_MAP

        )


    update_status(

        "model",

        (
            "Loading SegFormer "
            f"on {DEVICE}."
        )

    )


    SEGFORMER_PROCESSOR = (

        SegformerImageProcessor
        .from_pretrained(
            SEGFORMER_MODEL_NAME
        )

    )


    SEGFORMER_MODEL = (

        SegformerForSemanticSegmentation

        .from_pretrained(
            SEGFORMER_MODEL_NAME
        )

        .to(
            DEVICE
        )

    )


    SEGFORMER_MODEL.eval()


    SEGFORMER_GROUP_MAP = (
        build_group_map(
            SEGFORMER_MODEL
        )
    )


    print(
        "SegFormer loaded."
    )


    return (

        SEGFORMER_PROCESSOR,

        SEGFORMER_MODEL,

        SEGFORMER_GROUP_MAP

    )


def segmentation_preview_path(
    item
):

    cluster_slug = slugify(
        item.get(
            "mrtCluster"
        )
        or
        "unknown-cluster"
    )


    station_slug = slugify(
        item.get(
            "stationNormalized"
        )
        or
        "unknown-station"
    )


    route_slug = slugify(
        item.get(
            "routeId"
        )
        or
        "unknown-route"
    )


    checkpoint = int(
        item.get(
            "checkpointIndex"
        )
        or
        0
    )


    folder = (

        SEGMENTATION_IMAGE_DIR

        /

        "by_mrt_cluster"

        /

        cluster_slug

        /

        station_slug

        /

        route_slug

        /

        f"cp-{checkpoint:04d}"

    )


    folder.mkdir(
        parents=True,
        exist_ok=True
    )


    return (

        folder

        /

        f'{item["imageId"]}.jpg'

    )


def deterministic_palette():

    rng = np.random.default_rng(
        42
    )


    return {

        index:
            tuple(

                int(
                    value
                )

                for value in

                rng.integers(
                    35,
                    225,
                    size=3
                )

            )

        for index in range(
            256
        )

    }


PALETTE = (
    deterministic_palette()
)


def grouped_shares_for_mask(
    mask,
    group_map
):

    counts = Counter(
        mask.flatten().tolist()
    )


    total = max(
        int(
            mask.size
        ),
        1
    )


    grouped = Counter()


    for (
        class_id,
        count
    ) in counts.items():

        grouped[

            group_map.get(
                int(
                    class_id
                ),
                "other"
            )

        ] += count


    return {

        key:
            round(
                value
                /
                total,
                7
            )

        for (
            key,
            value
        ) in grouped.items()

    }


def save_segmentation_preview(
    mask,
    image_path,
    output_path
):

    original = Image.open(
        image_path
    ).convert(
        "RGB"
    )


    overlay = np.zeros(

        (
            mask.shape[0],
            mask.shape[1],
            3
        ),

        dtype=np.uint8

    )


    for class_id in np.unique(
        mask
    ):

        overlay[
            mask == class_id
        ] = (

            PALETTE[
                int(
                    class_id
                )
                %
                256
            ]

        )


    overlay_image = (

        Image.fromarray(
            overlay
        )

        .resize(

            original.size,

            Image.Resampling.NEAREST

        )

    )


    Image.blend(

        original,

        overlay_image,

        alpha=0.44

    ).save(

        output_path,

        quality=90

    )


def segment_image(
    item,
    force=False
):

    cache_path = (

        IMAGE_CACHE_DIR

        /

        slugify(
            SEGFORMER_MODEL_NAME
        )

        /

        f'{item["imageId"]}.json'

    )


    cache_path.parent.mkdir(
        parents=True,
        exist_ok=True
    )


    preview_path = (
        segmentation_preview_path(
            item
        )
    )


    if (

        cache_path.exists()

        and

        preview_path.exists()

        and

        not force

    ):

        cached = load_json(
            cache_path
        )


        return {

            **item,

            **cached,

            "segmentationImage":
                to_web_path(
                    preview_path
                ),

            "segmentationStatus":
                "cached",

        }


    (
        processor,
        model,
        group_map
    ) = load_segformer()


    image = Image.open(
        item[
            "imagePath"
        ]
    ).convert(
        "RGB"
    )


    inputs = processor(

        images=image,

        return_tensors="pt"

    )


    inputs = {

        key:
            value.to(
                DEVICE
            )

        for (
            key,
            value
        ) in inputs.items()

    }


    with torch.no_grad():

        outputs = model(
            **inputs
        )


    logits = F.interpolate(

        outputs.logits,

        size=(
            image.height,
            image.width
        ),

        mode=
            "bilinear",

        align_corners=
            False

    )


    mask = (

        logits

        .argmax(
            dim=1
        )[0]

        .detach()

        .cpu()

        .numpy()

        .astype(
            np.int32
        )

    )


    height, width = (
        mask.shape
    )


    full = grouped_shares_for_mask(
        mask,
        group_map
    )


    bottom = grouped_shares_for_mask(

        mask[
            int(
                height * 0.62
            ):
            ,
            :
        ],

        group_map

    )


    middle = grouped_shares_for_mask(

        mask[
            int(
                height * 0.32
            )
            :
            int(
                height * 0.82
            ),
            :
        ],

        group_map

    )


    left = grouped_shares_for_mask(

        mask[
            :,
            :
            int(
                width * 0.5
            )
        ],

        group_map

    )


    right = grouped_shares_for_mask(

        mask[
            :,
            int(
                width * 0.5
            )
            :
        ],

        group_map

    )


    payload = {

        "groupShares":
            full,

        "spatialShares": {

            "bottomBand":
                bottom,

            "middleBand":
                middle,

            "leftHalf":
                left,

            "rightHalf":
                right,

        },

        "model":
            SEGFORMER_MODEL_NAME,

    }


    save_segmentation_preview(

        mask,

        item[
            "imagePath"
        ],

        preview_path

    )


    safe_json_dump(

        cache_path,

        payload

    )


    return {

        **item,

        **payload,

        "segmentationImage":
            to_web_path(
                preview_path
            ),

        "segmentationStatus":
            "processed",

    }


# ============================================================
# VISUAL METRICS
# ============================================================

def share(
    dictionary,
    key
):

    return safe_float(

        (
            dictionary
            or
            {}
        ).get(
            key
        ),

        0.0

    )


def clamp01(
    value
):

    return max(

        0.0,

        min(
            1.0,
            safe_float(
                value
            )
        )

    )


def visual_metrics(
    item
):

    shares = (
        item.get(
            "groupShares"
        )
        or
        {}
    )


    spatial = (
        item.get(
            "spatialShares"
        )
        or
        {}
    )


    bottom = (
        spatial.get(
            "bottomBand"
        )
        or
        {}
    )


    middle = (
        spatial.get(
            "middleBand"
        )
        or
        {}
    )


    left = (
        spatial.get(
            "leftHalf"
        )
        or
        {}
    )


    right = (
        spatial.get(
            "rightHalf"
        )
        or
        {}
    )


    pedestrian = share(
        shares,
        "pedestrian_surface"
    )


    road = share(
        shares,
        "road"
    )


    vegetation = share(
        shares,
        "vegetation"
    )


    sky = share(
        shares,
        "sky"
    )


    building = share(
        shares,
        "building"
    )


    vehicle = share(
        shares,
        "vehicle"
    )


    barrier = share(
        shares,
        "barrier"
    )


    stairs = share(
        shares,
        "stairs"
    )


    shade_structure = share(
        shares,
        "shade_structure"
    )


    bottom_pedestrian = share(
        bottom,
        "pedestrian_surface"
    )


    bottom_road = share(
        bottom,
        "road"
    )


    bottom_vehicle = share(
        bottom,
        "vehicle"
    )


    left_green = share(
        left,
        "vegetation"
    )


    right_green = share(
        right,
        "vegetation"
    )


    middle_green = share(
        middle,
        "vegetation"
    )


    greenery = clamp01(
        vegetation
        *
        2.8
    )


    green_balance = clamp01(

        1

        -

        abs(
            left_green
            -
            right_green
        )

        *
        3

    )


    pedestrian_space = clamp01(

        pedestrian
        *
        1.8

        +

        bottom_pedestrian
        *
        1.5

    )


    roadway_exposure = clamp01(

        road
        *
        1.1

        +

        bottom_road
        *
        1.2

        +

        vehicle
        *
        2.6

        +

        bottom_vehicle
        *
        2.0

    )


    obstruction = clamp01(

        barrier
        *
        2.0

        +

        bottom_vehicle
        *
        1.5

        +

        stairs
        *
        4.0

    )


    openness = clamp01(

        sky
        *
        1.45

        +

        vegetation
        *
        0.4

        -

        building
        *
        0.45

    )


    shade_proxy = clamp01(

        middle_green
        *
        1.7

        +

        vegetation
        *
        0.9

        +

        shade_structure
        *
        3.5

        +

        building
        *
        0.18

        -

        sky
        *
        0.20

    )


    stroller = clamp01(

        pedestrian_space
        *
        0.47

        +

        shade_proxy
        *
        0.17

        +

        openness
        *
        0.11

        +

        greenery
        *
        0.12

        -

        roadway_exposure
        *
        0.08

        -

        stairs
        *
        3.5
        *
        0.05

    )


    return {

        "greeneryScore":
            round(
                greenery,
                5
            ),

        "greenBalanceScore":
            round(
                green_balance,
                5
            ),

        "pedestrianSpaceScore":
            round(
                pedestrian_space,
                5
            ),

        "roadwayExposureScore":
            round(
                roadway_exposure,
                5
            ),

        "obstructionProxy":
            round(
                obstruction,
                5
            ),

        "opennessProxy":
            round(
                openness,
                5
            ),

        "shadeProxy":
            round(
                shade_proxy,
                5
            ),

        "strollerFriendlinessScore":
            round(
                stroller,
                5
            ),

        "bottomPedestrianShare":
            round(
                bottom_pedestrian,
                7
            ),

        "bottomRoadShare":
            round(
                bottom_road,
                7
            ),

        "bottomVehicleShare":
            round(
                bottom_vehicle,
                7
            ),

    }


GROUP_FEATURES = [

    "pedestrian_surface",
    "road",
    "vegetation",
    "building",
    "sky",
    "vehicle",
    "person",
    "barrier",
    "street_furniture",
    "stairs",
    "shade_structure",

]


def visual_feature_dict(
    item
):

    shares = (
        item.get(
            "groupShares"
        )
        or
        {}
    )


    metrics = (
        item.get(
            "visualMetrics"
        )
        or
        {}
    )


    output = {

        f"share_{key}":
            share(
                shares,
                key
            )

        for key in
        GROUP_FEATURES

    }


    for (
        key,
        value
    ) in metrics.items():

        if isinstance(
            value,
            (
                int,
                float
            )
        ):

            output[
                f"metric_{key}"
            ] = safe_float(
                value
            )


    return output


def cluster_visual_images(
    images,
    requested_clusters=
        DEFAULT_VISUAL_CLUSTER_COUNT
):

    if not images:

        return (
            [],
            []
        )


    dictionaries = [

        visual_feature_dict(
            item
        )

        for item in
        images

    ]


    keys = sorted(
        dictionaries[0].keys()
    )


    matrix = np.array(

        [

            [
                record[
                    key
                ]

                for key in
                keys
            ]

            for record in
            dictionaries

        ],

        dtype=np.float64

    )


    if len(
        images
    ) == 1:

        labels = np.array([
            0
        ])


    else:

        scaled = (
            StandardScaler()
            .fit_transform(
                matrix
            )
        )


        cluster_count = max(

            2,

            min(
                int(
                    requested_clusters
                ),
                len(
                    images
                )
            )

        )


        labels = (

            KMeans(

                n_clusters=
                    cluster_count,

                random_state=
                    RANDOM_STATE,

                n_init=20

            )

            .fit_predict(
                scaled
            )

        )


    enriched = []


    for (
        item,
        label
    ) in zip(
        images,
        labels
    ):

        enriched.append({

            **item,

            "visualMetrics":
                item.get(
                    "visualMetrics"
                )
                or
                visual_metrics(
                    item
                ),

            "cluster":
                int(
                    label
                ),

            "visualCluster":
                int(
                    label
                ),

        })


    summaries = []


    for cluster_id in sorted(

        set(
            int(
                value
            )
            for value in
            labels
        )

    ):

        members = [

            item

            for item in
            enriched

            if item[
                "visualCluster"
            ]
            ==
            cluster_id

        ]


        group_means = mean_numeric_dict(

            [

                item.get(
                    "groupShares",
                    {}
                )

                for item in
                members

            ]

        )


        dominant = sorted(

            group_means.items(),

            key=lambda pair:
                pair[1],

            reverse=True

        )[:4]


        summaries.append({

            "cluster":
                cluster_id,

            "count":
                len(
                    members
                ),

            "dominantFeatures":

                [

                    {

                        "feature":
                            feature,

                        "value":
                            round(
                                value,
                                5
                            ),

                        "share":
                            round(
                                value
                                *
                                100,
                                2
                            )

                    }

                    for (
                        feature,
                        value
                    ) in
                    dominant

                    if feature
                    !=
                    "other"

                ],

            "meanGroupShares":
                group_means,

            "meanVisualMetrics":
                mean_numeric_dict(

                    [

                        item.get(
                            "visualMetrics",
                            {}
                        )

                        for item in
                        members

                    ]

                ),

        })


    return (
        enriched,
        summaries
    )


print(
    "SegFormer + visual clustering ready."
)

SegFormer + visual clustering ready.


In [9]:
# ============================================================
# CELL 8 — CHECKPOINT AGGREGATION + SPATIAL SMOOTHING
# ============================================================

ROUTE_CHART_GROUPS = {

    "sky":
        "sky",

    "greenery":
        "vegetation",

    "road":
        "road",

    "pedestrian":
        "pedestrian_surface",

}


def checkpoint_aggregates(
    images
):

    grouped = defaultdict(
        list
    )


    for item in images:

        grouped[

            (
                item.get(
                    "routeId"
                ),

                int(
                    item.get(
                        "checkpointIndex"
                    )
                    or
                    0
                )
            )

        ].append(
            item
        )


    output = []


    for (
        route_checkpoint,
        members
    ) in grouped.items():

        (
            route_id,
            checkpoint_index
        ) = route_checkpoint


        first = members[0]


        output.append({

            "routeId":
                route_id,

            "checkpointIndex":
                checkpoint_index,

            "distanceAlongRoute_m":
                first.get(
                    "distanceAlongRoute_m"
                ),

            "lat":
                first.get(
                    "checkpointLat"
                ),

            "lon":
                first.get(
                    "checkpointLon"
                ),

            "imageCount":
                len(
                    members
                ),

            "imageIds":
                [
                    item[
                        "imageId"
                    ]
                    for item in
                    members
                ],

            "meanGroupShares":
                mean_numeric_dict(

                    [

                        item.get(
                            "groupShares",
                            {}
                        )

                        for item in
                        members

                    ]

                ),

            "meanVisualMetrics":
                mean_numeric_dict(

                    [

                        item.get(
                            "visualMetrics",
                            {}
                        )

                        for item in
                        members

                    ]

                ),

            "visualClusterCounts":
                dict(

                    Counter(

                        int(
                            item.get(
                                "visualCluster",
                                0
                            )
                        )

                        for item in
                        members

                    )

                ),

        })


    output.sort(

        key=lambda item: (

            str(
                item[
                    "routeId"
                ]
            ),

            safe_float(
                item[
                    "distanceAlongRoute_m"
                ]
            )

        )

    )


    return output


def chart_values(
    shares
):

    shares = (
        shares
        or
        {}
    )


    return {

        output_key:
            round(

                100

                *

                safe_float(
                    shares.get(
                        group_key
                    ),
                    0.0
                ),

                4

            )

        for (
            output_key,
            group_key
        ) in
        ROUTE_CHART_GROUPS.items()

    }


def raw_route_series(
    checkpoints
):

    return [

        {

            "distance_m":
                round(

                    safe_float(
                        item.get(
                            "distanceAlongRoute_m"
                        )
                    ),

                    2

                ),

            "checkpointIndex":
                item.get(
                    "checkpointIndex"
                ),

            "imageCount":
                item.get(
                    "imageCount"
                ),

            "values":
                chart_values(

                    item.get(
                        "meanGroupShares"
                    )

                )

        }

        for item in
        checkpoints

    ]


def smoothed_route_series(
    raw,
    route_distance_m
):

    if not raw:
        return []


    source_distances = np.array(

        [

            item[
                "distance_m"
            ]

            for item in
            raw

        ],

        dtype=np.float64

    )


    positions = np.arange(

        0.0,

        route_distance_m
        +
        ROUTE_PROFILE_STEP_M,

        ROUTE_PROFILE_STEP_M

    )


    output = []


    for position in positions:

        nearest = float(

            np.min(

                np.abs(

                    source_distances
                    -
                    position

                )

            )

        )


        # Never invent values across large
        # image coverage gaps.
        if (
            nearest
            >
            ROUTE_SMOOTHING_MAX_GAP_M
        ):

            output.append({

                "distance_m":
                    round(
                        float(
                            position
                        ),
                        2
                    ),

                "values": {

                    key:
                        None

                    for key in
                    ROUTE_CHART_GROUPS

                },

                "supportCount":
                    0,

                "nearestObservation_m":
                    round(
                        nearest,
                        2
                    )

            })

            continue


        delta = (

            source_distances
            -
            position

        )


        weights = np.exp(

            -0.5

            *

            (
                delta
                /
                ROUTE_SMOOTHING_SIGMA_M
            )

            ** 2

        )


        weights[
            np.abs(
                delta
            )
            >
            ROUTE_SMOOTHING_MAX_GAP_M
        ] = 0.0


        values = {}


        for key in ROUTE_CHART_GROUPS:

            source_values = np.array(

                [

                    safe_float(

                        item[
                            "values"
                        ].get(
                            key
                        ),

                        np.nan

                    )

                    for item in
                    raw

                ],

                dtype=np.float64

            )


            valid = (

                np.isfinite(
                    source_values
                )

                &

                (
                    weights
                    >
                    0
                )

            )


            if not valid.any():

                values[
                    key
                ] = None

                continue


            values[
                key
            ] = round(

                float(

                    np.average(

                        source_values[
                            valid
                        ],

                        weights=
                            weights[
                                valid
                            ]

                    )

                ),

                4

            )


        output.append({

            "distance_m":
                round(
                    float(
                        position
                    ),
                    2
                ),

            "values":
                values,

            "supportCount":
                int(

                    (

                        np.abs(
                            delta
                        )

                        <=

                        ROUTE_SMOOTHING_SIGMA_M
                        *
                        2

                    ).sum()

                ),

            "nearestObservation_m":
                round(
                    nearest,
                    2
                ),

        })


    return output


def build_route_profiles(
    crawled_routes,
    analysed_images
):

    aggregates = checkpoint_aggregates(
        analysed_images
    )


    by_route = defaultdict(
        list
    )


    for item in aggregates:

        by_route[
            item[
                "routeId"
            ]
        ].append(
            item
        )


    profiles = []


    for route in crawled_routes:

        route_id = route[
            "routeId"
        ]


        checkpoints = (
            by_route.get(
                route_id,
                []
            )
        )


        expected = int(

            (
                route.get(
                    "routeSummary"
                )
                or
                {}
            ).get(
                "checkpointCount"
            )

            or

            len(
                route.get(
                    "checkpoints",
                    []
                )
            )

        )


        analysed = len(
            checkpoints
        )


        coverage = (

            analysed
            /
            expected

            if expected

            else

            0.0

        )


        route_distance = safe_float(

            (
                route.get(
                    "routeSummary"
                )
                or
                {}
            ).get(
                "distance_m"
            ),

            0.0

        )


        raw = raw_route_series(
            checkpoints
        )


        smooth = smoothed_route_series(

            raw,

            route_distance

        )


        route_images = [

            item

            for item in
            analysed_images

            if item.get(
                "routeId"
            )
            ==
            route_id

        ]


        profiles.append({

            "routeId":
                route_id,

            "station":
                route.get(
                    "station"
                ),

            "destination":
                route.get(
                    "destination"
                ),

            "somCell":
                route.get(
                    "somCell"
                ),

            "contextDescriptor":
                route.get(
                    "contextDescriptor"
                ),

            "imageCount":
                len(
                    route_images
                ),

            "expectedCheckpointCount":
                expected,

            "analysedCheckpointCount":
                analysed,

            "visualCoverage":
                round(
                    coverage,
                    5
                ),

            "visualDataGap":
                (
                    "high"
                    if coverage
                    <
                    0.35

                    else
                    "moderate"
                    if coverage
                    <
                    0.70

                    else
                    "low"
                ),

            "meanGroupShares":
                mean_numeric_dict(

                    [

                        item.get(
                            "groupShares",
                            {}
                        )

                        for item in
                        route_images

                    ]

                ),

            "meanVisualMetrics":
                mean_numeric_dict(

                    [

                        item.get(
                            "visualMetrics",
                            {}
                        )

                        for item in
                        route_images

                    ]

                ),

            "routeSeries": {

                "routeDistance_m":
                    round(
                        route_distance,
                        2
                    ),

                "checkpointSpacing_m":
                    CHECKPOINT_SPACING_M,

                "smoothingStep_m":
                    ROUTE_PROFILE_STEP_M,

                "smoothingSigma_m":
                    ROUTE_SMOOTHING_SIGMA_M,

                "maximumInterpolationGap_m":
                    ROUTE_SMOOTHING_MAX_GAP_M,

                "raw":
                    raw,

                "smoothed":
                    smooth,

            }

        })


    return (
        profiles,
        aggregates
    )


print(
    "Checkpoint aggregation + route smoothing ready."
)

Checkpoint aggregation + route smoothing ready.


In [10]:
# ============================================================
# CELL 9 — OUTPUT COMPILER
# ============================================================

def representative_cells(
    images,
    limit=
        COMPILER_GRID_SIZE
):

    if not images:
        return []


    if len(
        images
    ) <= limit:

        selected = images


    else:

        # Sample across the WHOLE image collection.
        # Don't just take the first 100.
        indices = np.linspace(

            0,

            len(
                images
            )
            -
            1,

            num=
                limit,

            dtype=int

        )


        selected = [

            images[
                index
            ]

            for index in
            indices

        ]


    cells = []


    for (
        slot,
        item
    ) in enumerate(
        selected[
            :
            limit
        ]
    ):

        cells.append({

            "slot":
                slot,

            "imageId":
                item.get(
                    "imageId"
                ),

            "originalImage":
                item.get(
                    "originalImage"
                ),

            "segmentationImage":
                item.get(
                    "segmentationImage"
                ),

            "station":
                item.get(
                    "station"
                ),

            "somCell":
                item.get(
                    "somCell"
                ),

            "mrtCluster":
                item.get(
                    "mrtCluster"
                ),

            "routeId":
                item.get(
                    "routeId"
                ),

            "checkpointIndex":
                item.get(
                    "checkpointIndex"
                ),

            "distanceAlongRoute_m":
                item.get(
                    "distanceAlongRoute_m"
                ),

            "cluster":
                item.get(
                    "visualCluster"
                ),

            "visualCluster":
                item.get(
                    "visualCluster"
                ),

            "groupShares":
                item.get(
                    "groupShares"
                ),

            "visualIndicators":
                item.get(
                    "visualMetrics"
                ),

        })


    return cells


def simplified_image_record(
    item
):

    return {

        "imageId":
            item.get(
                "imageId"
            ),

        "originalImage":
            item.get(
                "originalImage"
            ),

        "segmentationImage":
            item.get(
                "segmentationImage"
            ),

        "station":
            item.get(
                "station"
            ),

        "somCell":
            item.get(
                "somCell"
            ),

        "mrtCluster":
            item.get(
                "mrtCluster"
            ),

        "routeId":
            item.get(
                "routeId"
            ),

        "checkpointIndex":
            item.get(
                "checkpointIndex"
            ),

        "distanceAlongRoute_m":
            item.get(
                "distanceAlongRoute_m"
            ),

        "cluster":
            item.get(
                "visualCluster"
            ),

        "groupShares":
            item.get(
                "groupShares"
            ),

        "visualMetrics":
            item.get(
                "visualMetrics"
            ),

    }


def write_analysis_outputs(
    crawled_routes,
    analysed_images,
    rejected_images,
    download_errors,
    clusters
):

    (
        route_profiles,
        checkpoint_profiles
    ) = build_route_profiles(

        crawled_routes,

        analysed_images

    )


    generated_at = (

        datetime.now()
        .isoformat(
            timespec="seconds"
        )

    )


    methodology = {

        "checkpointSpacing_m":
            CHECKPOINT_SPACING_M,

        "mapillarySearchRadius_m":
            MAPILLARY_RADIUS_M,

        "maxImagesPerCheckpoint":
            MAPILLARY_IMAGES_PER_CHECKPOINT,

        "segmentationModel":
            SEGFORMER_MODEL_NAME,

        "routeSmoothing": {

            "step_m":
                ROUTE_PROFILE_STEP_M,

            "sigma_m":
                ROUTE_SMOOTHING_SIGMA_M,

            "maximumGap_m":
                ROUTE_SMOOTHING_MAX_GAP_M,

        },

        "missingData":
            (
                "Missing OSM or Mapillary evidence "
                "is treated as unknown, not proof "
                "that the physical feature is absent."
            ),

    }


    full_payload = {

        "version":
            6,

        "generatedAt":
            generated_at,

        "methodology":
            methodology,

        "routeCount":
            len(
                crawled_routes
            ),

        "imageCount":
            len(
                analysed_images
            ),

        "rejectedImageCount":
            len(
                rejected_images
            ),

        "downloadErrorCount":
            len(
                download_errors
            ),

        "routes":
            route_profiles,

        "checkpoints":
            checkpoint_profiles,

        "clusters":
            clusters,

        "images":
            analysed_images,

        "rejectedImages":
            rejected_images,

        "downloadErrors":
            download_errors,

    }


    compiler_payload = {

        "version":
            6,

        "generatedAt":
            generated_at,

        # IMPORTANT:
        # this is ALL analysed images,
        # not merely the frontend grid.
        "imageCount":
            len(
                analysed_images
            ),

        "clusterCount":
            len(
                clusters
            ),

        # Representative frontend view
        "cells":
            representative_cells(

                analysed_images,

                COMPILER_GRID_SIZE

            ),

        # ALL ANALYSED IMAGES
        "allImages":

            [

                simplified_image_record(
                    item
                )

                for item in
                analysed_images

            ],

        "clusters":
            clusters,

        "routes":
            route_profiles,

        "methodology":
            methodology,

    }


    safe_json_dump(

        IMAGE_ANALYSIS_FULL_PATH,

        full_payload

    )


    safe_json_dump(

        IMAGE_COMPILER_PATH,

        compiler_payload

    )


    # ========================================================
    # AUDIT CSV
    # ========================================================

    rows = []


    for item in analysed_images:

        rows.append({

            "imageId":
                item.get(
                    "imageId"
                ),

            "station":
                item.get(
                    "station"
                ),

            "somCell":
                item.get(
                    "somCell"
                ),

            "routeId":
                item.get(
                    "routeId"
                ),

            "checkpointIndex":
                item.get(
                    "checkpointIndex"
                ),

            "distanceAlongRoute_m":
                item.get(
                    "distanceAlongRoute_m"
                ),

            "visualCluster":
                item.get(
                    "visualCluster"
                ),

            "originalImage":
                item.get(
                    "originalImage"
                ),

            "segmentationImage":
                item.get(
                    "segmentationImage"
                ),

            "sky":
                share(
                    item.get(
                        "groupShares"
                    ),
                    "sky"
                ),

            "greenery":
                share(
                    item.get(
                        "groupShares"
                    ),
                    "vegetation"
                ),

            "road":
                share(
                    item.get(
                        "groupShares"
                    ),
                    "road"
                ),

            "pedestrian":
                share(
                    item.get(
                        "groupShares"
                    ),
                    "pedestrian_surface"
                ),

        })


    pd.DataFrame(
        rows
    ).to_csv(

        GENERATED_DIR
        /
        "backend_image_catalog.csv",

        index=False

    )


    return (
        compiler_payload,
        full_payload
    )


print(
    "Output compiler ready."
)

Output compiler ready.


In [11]:
# ============================================================
# CELL 10 — COMPLETE BACKEND CRAWL
# ============================================================

def compile_everything(
    force_crawl=False,
    force_images=False,
    routes=None,
    max_images=
        MAX_TOTAL_IMAGES,
    cluster_count=
        DEFAULT_VISUAL_CLUSTER_COUNT
):

    if not COMPILE_LOCK.acquire(
        blocking=False
    ):

        raise RuntimeError(
            "A backend crawl is already running."
        )


    started = time.time()


    try:

        update_status(

            "starting",

            "Discovering route geometry.",

            running=True,

            processed=0,

            total=0

        )


        # ====================================================
        # ROUTES
        # ====================================================

        if routes is None:

            routes = discover_routes()


        else:

            normalized = []


            for (
                index,
                route
            ) in enumerate(
                routes
            ):

                if not isinstance(
                    route,
                    dict
                ):

                    continue


                candidate = normalize_route_record(

                    route,

                    fallback_index=
                        index,

                    source=
                        "API request"

                )


                if candidate:

                    normalized.append(
                        candidate
                    )


            routes = normalized


        if not routes:

            raise RuntimeError(

                "No route geometry was discovered. "
                "The backend looked for "
                "family_route_analysis.json and other "
                "route/journey JSON files."

            )


        print()
        print(
            "=============================================="
        )
        print(
            "STARTING COMPLETE BACKEND CRAWL"
        )
        print(
            "=============================================="
        )
        print(
            "Routes:",
            len(
                routes
            )
        )
        print(
            "Checkpoint spacing:",
            CHECKPOINT_SPACING_M,
            "m"
        )
        print(
            "Max images/checkpoint:",
            MAPILLARY_IMAGES_PER_CHECKPOINT
        )
        print(
            "SegFormer cap:",
            max_images
        )
        print(
            "=============================================="
        )


        # ====================================================
        # ROUTE + OSM + MAPILLARY CRAWL
        # ====================================================

        crawled_routes = crawl_routes(

            routes,

            force=
                force_crawl

        )


        # ====================================================
        # MAPILLARY REFERENCES
        # ====================================================

        references = extract_image_references(
            crawled_routes
        )


        print()
        print(
            "Mapillary image references:",
            len(
                references
            )
        )


        if not references:

            raise RuntimeError(

                "Routes were crawled successfully, "
                "but Mapillary returned no usable images."

            )


        # ====================================================
        # BALANCED IMAGE CAP
        # ====================================================

        selected = balanced_image_queue(

            references,

            min(
                int(
                    max_images
                ),
                MAX_TOTAL_IMAGES
            )

        )


        print(
            "Images selected for analysis:",
            len(
                selected
            )
        )


        # ====================================================
        # DOWNLOAD
        # ====================================================

        update_status(

            "download",

            (
                "Downloading "
                f"{len(selected)} "
                "Mapillary images."
            ),

            processed=0,

            total=
                len(
                    selected
                )

        )


        (
            downloaded,
            download_errors
        ) = download_images(

            selected,

            force=
                force_images

        )


        print(
            "Downloaded/cached:",
            len(
                downloaded
            )
        )

        print(
            "Download errors:",
            len(
                download_errors
            )
        )


        # ====================================================
        # QUALITY CONTROL
        # ====================================================

        (
            accepted,
            rejected
        ) = quality_filter_images(
            downloaded
        )


        print(
            "Accepted after QC:",
            len(
                accepted
            )
        )

        print(
            "Rejected after QC:",
            len(
                rejected
            )
        )


        if not accepted:

            raise RuntimeError(

                "Images were downloaded but none "
                "passed quality control."

            )


        # ====================================================
        # SEGFORMER
        # ====================================================

        update_status(

            "segmentation",

            (
                "Segmenting "
                f"{len(accepted)} "
                "accepted images."
            ),

            processed=0,

            total=
                len(
                    accepted
                )

        )


        analysed = []


        for (
            index,
            item
        ) in enumerate(
            accepted,
            start=1
        ):

            update_status(

                "segmentation",

                (
                    f"[{index}/"
                    f"{len(accepted)}] "
                    f'{item.get("station")} | '
                    f'route '
                    f'{item.get("routeId")} | '
                    f'cp '
                    f'{item.get("checkpointIndex")}'
                ),

                processed=
                    index - 1,

                total=
                    len(
                        accepted
                    )

            )


            segmented = segment_image(

                item,

                force=
                    force_images

            )


            analysed.append({

                **segmented,

                "visualMetrics":
                    visual_metrics(
                        segmented
                    )

            })


            if (
                DEVICE == "cuda"
                and
                index % 20 == 0
            ):

                torch.cuda.empty_cache()


        # ====================================================
        # K-MEANS
        # ====================================================

        update_status(

            "clustering",

            "Clustering street environments.",

            processed=
                len(
                    analysed
                ),

            total=
                len(
                    analysed
                )

        )


        (
            analysed,
            clusters
        ) = cluster_visual_images(

            analysed,

            requested_clusters=
                cluster_count

        )


        # ====================================================
        # OUTPUT
        # ====================================================

        (
            compiler_payload,
            full_payload
        ) = write_analysis_outputs(

            crawled_routes,

            analysed,

            rejected,

            download_errors,

            clusters

        )


        summary = {

            "ok":
                True,

            "routeCount":
                len(
                    crawled_routes
                ),

            "denseCheckpointCount":
                sum(

                    len(
                        route.get(
                            "checkpoints",
                            []
                        )
                    )

                    for route in
                    crawled_routes

                ),

            "checkpointCountWithImages":
                sum(

                    1

                    for route in
                    crawled_routes

                    for checkpoint in
                    route.get(
                        "checkpoints",
                        []
                    )

                    if (
                        checkpoint.get(
                            "imagery"
                        )
                        or
                        {}
                    ).get(
                        "images"
                    )

                ),

            "mapillaryImageReferences":
                len(
                    references
                ),

            "imagesSelected":
                len(
                    selected
                ),

            "imagesDownloaded":
                len(
                    downloaded
                ),

            "imagesAccepted":
                len(
                    analysed
                ),

            "imagesRejected":
                len(
                    rejected
                ),

            "clusterCount":
                len(
                    clusters
                ),

            "seconds":
                round(
                    time.time()
                    -
                    started,
                    1
                ),

            "imageCompiler":
                str(
                    IMAGE_COMPILER_PATH
                ),

            "fullAnalysis":
                str(
                    IMAGE_ANALYSIS_FULL_PATH
                ),

        }


        update_status(

            "complete",

            (
                "Complete: "
                f'{summary["denseCheckpointCount"]} '
                "dense checkpoints, "
                f'{summary["imagesAccepted"]} '
                "analysed images."
            ),

            running=False,

            processed=
                len(
                    analysed
                ),

            total=
                len(
                    analysed
                ),

            lastResult=
                summary

        )


        print()
        print(
            "=============================================="
        )
        print(
            "FULL BACKEND CRAWL COMPLETE"
        )
        print(
            "=============================================="
        )


        for (
            key,
            value
        ) in summary.items():

            print(
                key,
                ":",
                value
            )


        print(
            "=============================================="
        )


        return summary


    except Exception as error:

        update_status(

            "error",

            str(
                error
            ),

            running=False

        )


        raise


    finally:

        COMPILE_LOCK.release()


print(
    "Complete crawl pipeline ready."
)

Complete crawl pipeline ready.


In [12]:
# ============================================================
# CELL 11 — SINGLE FLASK BACKEND
#
# PORT 5001 ONLY.
# PORT 5000 IS NO LONGER NEEDED.
# ============================================================

app = Flask(
    __name__
)

CORS(
    app
)


@app.get("/")
def root():

    return jsonify({

        "ok":
            True,

        "service":
            "young-family-single-backend",

        "port":
            5001,

        "message":
            (
                "OSM crawl + Mapillary crawl + "
                "SegFormer are handled here."
            ),

    })


@app.get("/health")
def health():

    return jsonify({

        "ok":
            True,

        "service":
            "young-family-single-backend",

        "version":
            6,

        "mapillaryConfigured":
            bool(
                MAPILLARY_ACCESS_TOKEN
            ),

        "device":
            DEVICE,

        "checkpointSpacingM":
            CHECKPOINT_SPACING_M,

        "routesDiscovered":
            len(
                discover_routes()
            ),

        "compilerExists":
            IMAGE_COMPILER_PATH.exists(),

    })


@app.get("/status")
def status():

    with STATUS_LOCK:

        return jsonify({

            "ok":
                True,

            **STATUS

        })


@app.get("/routes")
def routes_endpoint():

    routes = discover_routes()


    return jsonify({

        "ok":
            True,

        "count":
            len(
                routes
            ),

        "routes":

            [

                {

                    "routeId":
                        route[
                            "routeId"
                        ],

                    "station":
                        route[
                            "station"
                        ],

                    "destination":
                        route[
                            "destination"
                        ],

                    "sourcePointCount":
                        len(
                            route[
                                "points"
                            ]
                        ),

                    "source":
                        route[
                            "source"
                        ],

                }

                for route in
                routes

            ]

    })


@app.post("/compile")
def compile_endpoint():

    payload = (
        request.get_json(
            silent=True
        )
        or
        {}
    )


    routes = (

        payload.get(
            "routes"
        )

        if isinstance(
            payload,
            dict
        )

        else

        None

    )


    if isinstance(
        payload,
        list
    ):

        routes = payload


    try:

        result = compile_everything(

            force_crawl=
                bool(
                    payload.get(
                        "forceCrawl",
                        False
                    )
                )
                if isinstance(
                    payload,
                    dict
                )
                else
                False,

            force_images=
                bool(
                    payload.get(
                        "forceImages",
                        False
                    )
                )
                if isinstance(
                    payload,
                    dict
                )
                else
                False,

            routes=
                routes,

            max_images=
                int(
                    payload.get(
                        "maxImages",
                        MAX_TOTAL_IMAGES
                    )
                )
                if isinstance(
                    payload,
                    dict
                )
                else
                MAX_TOTAL_IMAGES,

            cluster_count=
                int(
                    payload.get(
                        "clusterCount",
                        DEFAULT_VISUAL_CLUSTER_COUNT
                    )
                )
                if isinstance(
                    payload,
                    dict
                )
                else
                DEFAULT_VISUAL_CLUSTER_COUNT,

        )


        return jsonify(
            result
        )


    except Exception as error:

        return jsonify({

            "ok":
                False,

            "error":
                str(
                    error
                )

        }), 500


# Alias
@app.post("/crawl")
def crawl_endpoint():

    return compile_endpoint()


@app.get("/compiler")
def compiler_endpoint():

    if not IMAGE_COMPILER_PATH.exists():

        return jsonify({

            "ok":
                False,

            "error":
                (
                    "image_compiler.json "
                    "has not been generated."
                )

        }), 404


    return jsonify({

        "ok":
            True,

        "compiler":
            load_json(
                IMAGE_COMPILER_PATH
            )

    })


def run_backend():

    app.run(

        host=
            "127.0.0.1",

        port=
            5001,

        debug=
            False,

        use_reloader=
            False,

        threaded=
            True

    )


backend_thread = threading.Thread(

    target=
        run_backend,

    daemon=
        True

)


backend_thread.start()


time.sleep(
    1
)


print()
print(
    "=================================================="
)
print(
    "SINGLE BACKEND STARTED"
)
print(
    "http://127.0.0.1:5001"
)
print(
    "Health: http://127.0.0.1:5001/health"
)
print(
    "Status: http://127.0.0.1:5001/status"
)
print(
    "Routes: http://127.0.0.1:5001/routes"
)
print(
    "Compile: POST http://127.0.0.1:5001/compile"
)
print(
    "=================================================="
)

 * Serving Flask app '__main__'
 * Debug mode: off


 * Running on http://127.0.0.1:5001
Press CTRL+C to quit



SINGLE BACKEND STARTED
http://127.0.0.1:5001
Health: http://127.0.0.1:5001/health
Status: http://127.0.0.1:5001/status
Routes: http://127.0.0.1:5001/routes
Compile: POST http://127.0.0.1:5001/compile


In [13]:
# ============================================================
# CELL 12 — RUN THE ENTIRE CRAWL NOW
# ============================================================
#
# THIS CELL:
#
# route geometry
#      ↓
# 15 m checkpoints
#      ↓
# OSM
#      ↓
# Mapillary
#      ↓
# image download
#      ↓
# QC
#      ↓
# SegFormer
#      ↓
# checkpoint averaging
#      ↓
# K-means
#      ↓
# spatial smoothing
#      ↓
# JSON outputs
#
# ============================================================


RUN_FULL_BACKEND_CRAWL = True


# False = reuse downloaded/processed material
# where possible.
#
# This is much faster on repeated runs.
FORCE_CRAWL = False

FORCE_IMAGE_REPROCESS = False


if RUN_FULL_BACKEND_CRAWL:

    RESULT = compile_everything(

        force_crawl=
            FORCE_CRAWL,

        force_images=
            FORCE_IMAGE_REPROCESS,

        max_images=
            MAX_TOTAL_IMAGES,

        cluster_count=
            DEFAULT_VISUAL_CLUSTER_COUNT

    )


else:

    print(
        "Backend is ready."
    )

    print(
        "Set RUN_FULL_BACKEND_CRAWL = True "
        "to crawl now."
    )

[starting] Discovering route geometry.

STARTING COMPLETE BACKEND CRAWL
Routes: 192
Checkpoint spacing: 15 m
Max images/checkpoint: 3
SegFormer cap: 600
[route-crawl] Route 1/192: Residential Cluster 001

--------------------------------------------------
Residential Cluster 001 → Kranji MRT
Route: residential-0001__kranji-mrt
Distance: 2380.7 m
Dense checkpoints: 160
--------------------------------------------------
[osm] Residential Cluster 001: crawling OSM corridor.
OSM warning: 406 Client Error: Not Acceptable for url: https://overpass-api.de/api/interpreter
OSM features: 0
[mapillary] Residential Cluster 001: crawling Mapillary route bbox.
Mapillary candidates: 814
[route-crawl] Route 2/192: Residential Cluster 002

--------------------------------------------------
Residential Cluster 002 → Springleaf MRT
Route: residential-0002__springleaf-mrt
Distance: 3182.4 m
Dense checkpoints: 214
--------------------------------------------------
[osm] Residential Cluster 002: crawling OS

127.0.0.1 - - [14/Oct/2026 15:52:25] "GET /status HTTP/1.1" 200 -
127.0.0.1 - - [14/Oct/2026 15:52:26] "GET /favicon.ico HTTP/1.1" 404 -


Downloaded/cached: 600
Download errors: 0
Accepted after QC: 359
Rejected after QC: 241
[segmentation] Segmenting 359 accepted images.
[segmentation] [1/359] ADMIRALTY MRT | route ADMIRALTY__R01 | cp 0
[model] Loading SegFormer on cpu.


Loading weights: 100%|██████████| 380/380 [00:00<00:00, 10456.73it/s]


SegFormer loaded.
[segmentation] [2/359] ANG MO KIO MRT | route ANG_MO_KIO__R01 | cp 0
[segmentation] [3/359] BEDOK NORTH MRT | route BEDOK_NORTH__R01 | cp 33
[segmentation] [4/359] BOON KENG MRT | route BOON_KENG__R01 | cp 0
[segmentation] [5/359] BOTANIC GARDENS MRT | route BOTANIC_GARDENS__R01 | cp 0
[segmentation] [6/359] BUGIS MRT | route BUGIS__R01 | cp 0
[segmentation] [7/359] BUKIT BATOK MRT | route BUKIT_BATOK__R01 | cp 5
[segmentation] [8/359] BUKIT PANJANG MRT | route BUKIT_PANJANG__R01 | cp 0
[segmentation] [9/359] CALDECOTT MRT | route CALDECOTT__R01 | cp 1
[segmentation] [10/359] CASHEW MRT | route CASHEW__R01 | cp 0
[segmentation] [11/359] CHINATOWN MRT | route CHINATOWN__R01 | cp 0
[segmentation] [12/359] CHOA CHU KANG MRT | route CHOA_CHU_KANG__R01 | cp 4
[segmentation] [13/359] DAKOTA MRT | route DAKOTA__R01 | cp 0
[segmentation] [14/359] EUNOS MRT | route EUNOS__R01 | cp 0
[segmentation] [15/359] HAVELOCK MRT | route HAVELOCK__R01 | cp 0
[segmentation] [16/359] KAKI 

In [14]:
# ============================================================
# FINAL CELL — BUILD WEBSITE ML EVIDENCE
#
# DOES NOT RECRAWL
# DOES NOT REDOWNLOAD
# DOES NOT RERUN SEGFORMER
#
# Run this AFTER the full backend crawl has completed.
# ============================================================

from collections import Counter
import json
import math
import numpy as np

from sklearn.preprocessing import StandardScaler


# ============================================================
# 1. LOAD FINISHED BACKEND OUTPUTS
# ============================================================

if not IMAGE_ANALYSIS_FULL_PATH.exists():
    raise FileNotFoundError(
        "image_analysis_full.json does not exist yet. "
        "Let the full backend crawl finish first."
    )


if not IMAGE_COMPILER_PATH.exists():
    raise FileNotFoundError(
        "image_compiler.json does not exist yet. "
        "Let the full backend crawl finish first."
    )


with open(
    IMAGE_ANALYSIS_FULL_PATH,
    "r",
    encoding="utf-8"
) as file:
    full_analysis = json.load(file)


with open(
    IMAGE_COMPILER_PATH,
    "r",
    encoding="utf-8"
) as file:
    compiler = json.load(file)


images = (
    full_analysis.get("images")
    or compiler.get("allImages")
    or []
)


clusters = (
    full_analysis.get("clusters")
    or compiler.get("clusters")
    or []
)


print()
print("==============================================")
print("BUILDING WEBSITE ML EVIDENCE")
print("==============================================")

print(
    "Analysed images available:",
    len(images)
)

print(
    "Visual clusters available:",
    len(clusters)
)


if not images:
    raise RuntimeError(
        "No analysed images were found. "
        "The SegFormer stage must finish before running this cell."
    )


# ============================================================
# 2. HELPERS
# ============================================================

def ml_safe_float(
    value,
    default=0.0
):
    try:
        value = float(value)

        if math.isfinite(value):
            return value

    except Exception:
        pass

    return default


def ml_feature_dict(item):

    shares = (
        item.get("groupShares")
        or {}
    )

    metrics = (
        item.get("visualMetrics")
        or {}
    )

    return {

        # ----------------------------------------------------
        # Semantic segmentation proportions
        # ----------------------------------------------------

        "vegetation":
            ml_safe_float(
                shares.get("vegetation")
            ),

        "pedestrian_surface":
            ml_safe_float(
                shares.get("pedestrian_surface")
            ),

        "road":
            ml_safe_float(
                shares.get("road")
            ),

        "building":
            ml_safe_float(
                shares.get("building")
            ),

        "sky":
            ml_safe_float(
                shares.get("sky")
            ),

        "vehicle":
            ml_safe_float(
                shares.get("vehicle")
            ),

        "barrier":
            ml_safe_float(
                shares.get("barrier")
            ),


        # ----------------------------------------------------
        # Derived visual proxies
        # ----------------------------------------------------

        "greeneryScore":
            ml_safe_float(
                metrics.get("greeneryScore")
            ),

        "pedestrianSpaceScore":
            ml_safe_float(
                metrics.get("pedestrianSpaceScore")
            ),

        "roadwayExposureScore":
            ml_safe_float(
                metrics.get("roadwayExposureScore")
            ),

        "shadeProxy":
            ml_safe_float(
                metrics.get("shadeProxy")
            ),

        "strollerFriendlinessScore":
            ml_safe_float(
                metrics.get(
                    "strollerFriendlinessScore"
                )
            ),
    }


def ml_matrix_from_images(records):

    feature_records = [
        ml_feature_dict(item)
        for item in records
    ]

    feature_keys = list(
        feature_records[0].keys()
    )

    matrix = np.array(
        [
            [
                record[key]
                for key in feature_keys
            ]
            for record in feature_records
        ],
        dtype=np.float64
    )

    return (
        feature_keys,
        matrix
    )


# ============================================================
# 3. K-MEANS CLUSTER DISTRIBUTION
# ============================================================

cluster_distribution = []


for cluster in clusters:

    cluster_id = int(
        cluster.get(
            "cluster",
            cluster.get(
                "visualCluster",
                0
            )
        )
    )

    count = int(
        cluster.get(
            "count",
            cluster.get(
                "imageCount",
                0
            )
        )
    )

    cluster_distribution.append({

        "cluster":
            cluster_id,

        "count":
            count,

        "dominantFeatures":
            cluster.get(
                "dominantFeatures",
                []
            ),

        "meanGroupShares":
            cluster.get(
                "meanGroupShares",
                {}
            ),

        "meanVisualMetrics":
            cluster.get(
                "meanVisualMetrics",
                {}
            ),
    })


cluster_distribution.sort(
    key=lambda item:
        item["cluster"]
)


# ============================================================
# 4. SOM × VISUAL CLUSTER CROSS-TAB
#
# IMPORTANT:
# Demographic SOM and visual K-means remain separate models.
# ============================================================

cross_tab_counter = Counter()

som_totals = Counter()

visual_totals = Counter()


for item in images:

    som_cell = item.get(
        "somCell"
    )

    visual_cluster = item.get(
        "visualCluster",
        item.get(
            "cluster"
        )
    )

    if (
        som_cell is None
        or
        visual_cluster is None
    ):
        continue


    som_cell = str(
        som_cell
    )

    visual_cluster = int(
        visual_cluster
    )


    cross_tab_counter[
        (
            som_cell,
            visual_cluster
        )
    ] += 1


    som_totals[
        som_cell
    ] += 1


    visual_totals[
        visual_cluster
    ] += 1


som_cells = sorted(
    som_totals.keys()
)


visual_clusters = sorted(
    visual_totals.keys()
)


som_visual_matrix = []


for som_cell in som_cells:

    row = {

        "somCell":
            som_cell,

        "total":
            som_totals[
                som_cell
            ],

        "clusters":
            []
    }


    for visual_cluster in visual_clusters:

        count = cross_tab_counter[
            (
                som_cell,
                visual_cluster
            )
        ]


        share = (
            count
            /
            som_totals[
                som_cell
            ]

            if som_totals[
                som_cell
            ]

            else
            0.0
        )


        row[
            "clusters"
        ].append({

            "visualCluster":
                visual_cluster,

            "count":
                count,

            "share":
                round(
                    share,
                    6
                )

        })


    som_visual_matrix.append(
        row
    )


# ============================================================
# 5. EXPLORATORY MRT / SOM VISUAL RECOGNISABILITY
#
# Leave-one-out nearest-centroid test.
#
# This is descriptive ML evidence,
# NOT a validated predictive classifier.
# ============================================================

eligible = [

    item

    for item in images

    if item.get(
        "somCell"
    )
    is not None

]


recognition = {

    "eligibleImageCount":
        0,

    "correctImageCount":
        0,

    "accuracy":
        None,

    "confusion":
        [],

    "perSomCell":
        [],

    "note":
        (
            "Exploratory leave-one-out nearest-centroid "
            "recognisability test based on extracted visual "
            "features. This is descriptive evidence and is "
            "not a validated predictive classifier."
        )

}


if len(
    eligible
) >= 4:

    feature_keys, matrix = (
        ml_matrix_from_images(
            eligible
        )
    )


    scaler = (
        StandardScaler()
    )


    scaled = scaler.fit_transform(
        matrix
    )


    labels = [

        str(
            item[
                "somCell"
            ]
        )

        for item in eligible

    ]


    label_counts = Counter(
        labels
    )


    confusion_counter = Counter()

    correct = 0

    tested = 0

    per_cell_tested = Counter()

    per_cell_correct = Counter()


    for index, item in enumerate(
        eligible
    ):

        actual = labels[
            index
        ]


        # Cannot do leave-one-out
        # if only one image exists for this SOM cell.
        if label_counts[
            actual
        ] < 2:
            continue


        centroids = {}


        for candidate in sorted(
            label_counts.keys()
        ):

            candidate_indices = [

                other_index

                for (
                    other_index,
                    other_label
                ) in enumerate(
                    labels
                )

                if (
                    other_label
                    ==
                    candidate

                    and

                    other_index
                    !=
                    index
                )

            ]


            if not candidate_indices:
                continue


            centroids[
                candidate
            ] = np.mean(
                scaled[
                    candidate_indices
                ],
                axis=0
            )


        if len(
            centroids
        ) < 2:
            continue


        distances = {

            candidate:
                float(
                    np.linalg.norm(
                        scaled[
                            index
                        ]
                        -
                        centroid
                    )
                )

            for (
                candidate,
                centroid
            ) in centroids.items()

        }


        predicted = min(
            distances,
            key=distances.get
        )


        is_correct = (
            predicted
            ==
            actual
        )


        tested += 1

        correct += int(
            is_correct
        )


        per_cell_tested[
            actual
        ] += 1


        per_cell_correct[
            actual
        ] += int(
            is_correct
        )


        confusion_counter[
            (
                actual,
                predicted
            )
        ] += 1


    recognition[
        "eligibleImageCount"
    ] = tested


    recognition[
        "correctImageCount"
    ] = correct


    recognition[
        "accuracy"
    ] = (
        round(
            correct
            /
            tested,
            6
        )

        if tested

        else
        None
    )


    recognition[
        "confusion"
    ] = [

        {

            "actualSomCell":
                actual,

            "predictedSomCell":
                predicted,

            "count":
                count

        }

        for (
            (
                actual,
                predicted
            ),
            count
        ) in sorted(
            confusion_counter.items()
        )

    ]


    recognition[
        "perSomCell"
    ] = [

        {

            "somCell":
                som_cell,

            "tested":
                per_cell_tested[
                    som_cell
                ],

            "correct":
                per_cell_correct[
                    som_cell
                ],

            "accuracy":
                round(

                    per_cell_correct[
                        som_cell
                    ]

                    /

                    per_cell_tested[
                        som_cell
                    ],

                    6

                )

                if per_cell_tested[
                    som_cell
                ]

                else
                None

        }

        for som_cell in sorted(
            per_cell_tested.keys()
        )

    ]


    recognition[
        "featureKeys"
    ] = feature_keys


# ============================================================
# 6. CLUSTER FEATURE PROFILES
# ============================================================

feature_profiles = []


for cluster in cluster_distribution:

    cluster_id = cluster[
        "cluster"
    ]


    shares = (
        cluster.get(
            "meanGroupShares"
        )
        or
        {}
    )


    metrics = (
        cluster.get(
            "meanVisualMetrics"
        )
        or
        {}
    )


    feature_profiles.append({

        "cluster":
            cluster_id,

        "imageCount":
            cluster[
                "count"
            ],


        # ----------------------------------------------------
        # Semantic shares
        # ----------------------------------------------------

        "greenery":
            ml_safe_float(
                shares.get(
                    "vegetation"
                )
            ),

        "pedestrian":
            ml_safe_float(
                shares.get(
                    "pedestrian_surface"
                )
            ),

        "road":
            ml_safe_float(
                shares.get(
                    "road"
                )
            ),

        "building":
            ml_safe_float(
                shares.get(
                    "building"
                )
            ),

        "sky":
            ml_safe_float(
                shares.get(
                    "sky"
                )
            ),

        "vehicle":
            ml_safe_float(
                shares.get(
                    "vehicle"
                )
            ),


        # ----------------------------------------------------
        # Derived indicators
        # ----------------------------------------------------

        "greeneryScore":
            ml_safe_float(
                metrics.get(
                    "greeneryScore"
                )
            ),

        "pedestrianSpaceScore":
            ml_safe_float(
                metrics.get(
                    "pedestrianSpaceScore"
                )
            ),

        "roadwayExposureScore":
            ml_safe_float(
                metrics.get(
                    "roadwayExposureScore"
                )
            ),

        "shadeProxy":
            ml_safe_float(
                metrics.get(
                    "shadeProxy"
                )
            ),

        "strollerFriendliness":
            ml_safe_float(
                metrics.get(
                    "strollerFriendlinessScore"
                )
            ),

    })


# ============================================================
# 7. BUILD WEBSITE ML OBJECT
# ============================================================

ml_evidence = {

    "imageCount":
        len(
            images
        ),


    "visualClusterCount":
        len(
            cluster_distribution
        ),


    # --------------------------------------------------------
    # Graph 1
    # --------------------------------------------------------

    "clusterDistribution":
        cluster_distribution,


    # --------------------------------------------------------
    # Graph 2
    # --------------------------------------------------------

    "clusterFeatureProfiles":
        feature_profiles,


    # --------------------------------------------------------
    # Graph 3
    # --------------------------------------------------------

    "somVisualMatrix": {

        "somCells":
            som_cells,

        "visualClusters":
            visual_clusters,

        "rows":
            som_visual_matrix

    },


    # --------------------------------------------------------
    # Graph 4
    # --------------------------------------------------------

    "mrtClusterRecognisability":
        recognition,


    # --------------------------------------------------------
    # Methodology
    # --------------------------------------------------------

    "methodology": {

        "segmentation":
            SEGFORMER_MODEL_NAME,

        "visualClustering":
            (
                "K-means clustering of standardised "
                "computer-vision feature vectors."
            ),

        "demographicClustering":
            (
                "Demographic SOM assignment is independent "
                "from K-means visual clustering."
            ),

        "recognisability":
            recognition[
                "note"
            ]

    }

}


# ============================================================
# 8. ATTACH ML DATA TO FRONTEND COMPILER
# ============================================================

compiler[
    "mlEvidence"
] = ml_evidence


compiler[
    "mrtClusterRecognisability"
] = recognition


# Make sure full image list remains available
# to the frontend if it already exists.
if (
    "allImages"
    not in compiler
):

    compiler[
        "allImages"
    ] = [

        {

            "imageId":
                item.get(
                    "imageId"
                ),

            "originalImage":
                item.get(
                    "originalImage"
                ),

            "segmentationImage":
                item.get(
                    "segmentationImage"
                ),

            "station":
                item.get(
                    "station"
                ),

            "somCell":
                item.get(
                    "somCell"
                ),

            "mrtCluster":
                item.get(
                    "mrtCluster"
                ),

            "routeId":
                item.get(
                    "routeId"
                ),

            "checkpointIndex":
                item.get(
                    "checkpointIndex"
                ),

            "distanceAlongRoute_m":
                item.get(
                    "distanceAlongRoute_m"
                ),

            "cluster":
                item.get(
                    "visualCluster",
                    item.get(
                        "cluster"
                    )
                ),

            "visualCluster":
                item.get(
                    "visualCluster",
                    item.get(
                        "cluster"
                    )
                ),

            "groupShares":
                item.get(
                    "groupShares"
                ),

            "visualMetrics":
                item.get(
                    "visualMetrics"
                ),

        }

        for item in images

    ]


# ============================================================
# 9. SAVE FRONTEND COMPILER
# ============================================================

safe_json_dump(
    IMAGE_COMPILER_PATH,
    compiler
)


# ============================================================
# 10. SAVE SAME ML DATA TO FULL ANALYSIS
# ============================================================

full_analysis[
    "mlEvidence"
] = ml_evidence


safe_json_dump(
    IMAGE_ANALYSIS_FULL_PATH,
    full_analysis
)


# ============================================================
# 11. VERIFY OUTPUT
# ============================================================

print()
print(
    "=============================================="
)
print(
    "WEBSITE ML EVIDENCE GENERATED"
)
print(
    "=============================================="
)


print(
    "Images:",
    len(
        images
    )
)


print(
    "Visual clusters:",
    len(
        cluster_distribution
    )
)


print(
    "SOM cells represented:",
    len(
        som_cells
    )
)


print(
    "Recognition tested:",
    recognition[
        "eligibleImageCount"
    ]
)


print(
    "Recognition accuracy:",
    recognition[
        "accuracy"
    ]
)


print()
print(
    "Frontend ML sections created:"
)

print(
    "✓ K-means cluster distribution"
)

print(
    "✓ Cluster semantic feature profiles"
)

print(
    "✓ SOM × visual-cluster heatmap"
)

print(
    "✓ MRT visual recognisability"
)

print(
    "✓ Full segmentation image references"
)


print()
print(
    "Saved compiler:"
)

print(
    IMAGE_COMPILER_PATH
)


print()
print(
    "Saved full analysis:"
)

print(
    IMAGE_ANALYSIS_FULL_PATH
)


print(
    "=============================================="
)


# ============================================================
# 12. QUICK STRUCTURE CHECK
# ============================================================

with open(
    IMAGE_COMPILER_PATH,
    "r",
    encoding="utf-8"
) as file:

    verification = json.load(
        file
    )


print()
print(
    "Compiler sections:"
)

print(
    list(
        verification.keys()
    )
)


print()
print(
    "ML sections:"
)

print(
    list(
        verification.get(
            "mlEvidence",
            {}
        ).keys()
    )
)


BUILDING WEBSITE ML EVIDENCE
Analysed images available: 359
Visual clusters available: 10

WEBSITE ML EVIDENCE GENERATED
Images: 359
Visual clusters: 10
SOM cells represented: 37
Recognition tested: 108
Recognition accuracy: 0.175926

Frontend ML sections created:
✓ K-means cluster distribution
✓ Cluster semantic feature profiles
✓ SOM × visual-cluster heatmap
✓ MRT visual recognisability
✓ Full segmentation image references

Saved compiler:
C:\Users\MSI\OneDrive\Desktop\family-toilet-search\data\generated\image_compiler.json

Saved full analysis:
C:\Users\MSI\OneDrive\Desktop\family-toilet-search\data\generated\image_analysis_full.json

Compiler sections:
['version', 'generatedAt', 'imageCount', 'clusterCount', 'cells', 'allImages', 'clusters', 'routes', 'methodology', 'mlEvidence', 'mrtClusterRecognisability']

ML sections:
['imageCount', 'visualClusterCount', 'clusterDistribution', 'clusterFeatureProfiles', 'somVisualMatrix', 'mrtClusterRecognisability', 'methodology']


In [15]:
# ============================================================
# FINAL VISUALISATION CELL
# EXPORT PRESENTATION-QUALITY ML FIGURES FOR WEBSITE
#
# DOES NOT:
# - crawl Mapillary
# - download images
# - rerun SegFormer
#
# It only reads the completed analysis JSON.
# ============================================================

from pathlib import Path
from collections import defaultdict
import json
import math

import numpy as np
import matplotlib.pyplot as plt

from sklearn.preprocessing import StandardScaler
from sklearn.decomposition import PCA
from sklearn.tree import DecisionTreeClassifier, plot_tree

from scipy.cluster.hierarchy import linkage, dendrogram


# ============================================================
# PATHS
# ============================================================

try:
    PROJECT_ROOT
except NameError:
    cwd = Path.cwd()

    PROJECT_ROOT = (
        cwd.parent
        if cwd.name.lower() == "notebooks"
        else cwd
    )


GENERATED_DIR = (
    PROJECT_ROOT
    / "data"
    / "generated"
)

FULL_ANALYSIS_PATH = (
    GENERATED_DIR
    / "image_analysis_full.json"
)

COMPILER_PATH = (
    GENERATED_DIR
    / "image_compiler.json"
)

FIGURE_DIR = (
    GENERATED_DIR
    / "ml_figures"
)

FIGURE_DIR.mkdir(
    parents=True,
    exist_ok=True
)


# ============================================================
# LOAD COMPLETED ANALYSIS
# ============================================================

with open(
    FULL_ANALYSIS_PATH,
    "r",
    encoding="utf-8"
) as file:

    full_analysis = json.load(file)


with open(
    COMPILER_PATH,
    "r",
    encoding="utf-8"
) as file:

    compiler = json.load(file)


images = (
    full_analysis.get("images")
    or
    compiler.get("allImages")
    or
    []
)


print(
    "Analysed images:",
    len(images)
)


if not images:

    raise RuntimeError(
        "No analysed images found."
    )


# ============================================================
# FEATURE EXTRACTION
# ============================================================

FEATURE_KEYS = [

    "sky_exposure",
    "greenery",
    "road_exposure",
    "pedestrian_space",
    "built_frontage",
    "vehicle_presence"

]


FEATURE_LABELS = [

    "Sky exposure",
    "Greenery",
    "Road exposure",
    "Pedestrian space",
    "Built frontage",
    "Vehicle presence"

]


def safe_float(
    value,
    default=0.0
):

    try:

        value = float(value)

        if math.isfinite(value):
            return value

    except Exception:
        pass

    return default


def first_value(
    dictionary,
    *keys
):

    for key in keys:

        if key in dictionary:

            value = safe_float(
                dictionary.get(key),
                None
            )

            if value is not None:
                return value

    return 0.0


def extract_features(
    item
):

    shares = (
        item.get("groupShares")
        or
        item.get("semanticShares")
        or
        {}
    )


    return {

        "sky_exposure":
            first_value(
                shares,
                "sky"
            ),

        "greenery":
            first_value(
                shares,
                "vegetation",
                "greenery"
            ),

        "road_exposure":
            first_value(
                shares,
                "road",
                "road_surface"
            ),

        "pedestrian_space":
            first_value(
                shares,
                "pedestrian_surface",
                "pedestrian"
            ),

        "built_frontage":
            first_value(
                shares,
                "building",
                "built_frontage"
            ),

        "vehicle_presence":
            first_value(
                shares,
                "vehicle",
                "vehicles"
            )

    }


records = []


for item in images:

    cluster = item.get(
        "visualCluster",
        item.get("cluster")
    )


    if cluster is None:
        continue


    features = extract_features(
        item
    )


    vector = np.array(

        [
            features[key]
            for key in FEATURE_KEYS
        ],

        dtype=float

    )


    if not np.any(
        np.isfinite(vector)
    ):
        continue


    if np.allclose(
        vector,
        0
    ):
        continue


    records.append({

        "image":
            item,

        "cluster":
            int(cluster),

        "features":
            features,

        "vector":
            vector

    })


if len(records) < 10:

    raise RuntimeError(
        "Too few usable image records."
    )


X = np.vstack(
    [
        record["vector"]
        for record in records
    ]
)


y = np.array(
    [
        record["cluster"]
        for record in records
    ],
    dtype=int
)


cluster_ids = sorted(
    np.unique(y).tolist()
)


print(
    "Usable ML samples:",
    len(records)
)

print(
    "Visual clusters:",
    cluster_ids
)


# ============================================================
# STANDARDISE
# ============================================================

scaler = StandardScaler()

X_scaled = scaler.fit_transform(
    X
)


# ============================================================
# PCA
# ============================================================

pca = PCA(
    n_components=2
)

X_pca = pca.fit_transform(
    X_scaled
)


pc1_variance = (
    pca.explained_variance_ratio_[0]
    * 100
)

pc2_variance = (
    pca.explained_variance_ratio_[1]
    * 100
)

total_variance = (
    pc1_variance
    +
    pc2_variance
)


# ============================================================
# FIGURE 1
# PCA CLUSTER SPECTRUM
# ============================================================

fig, ax = plt.subplots(
    figsize=(10, 7)
)


for cluster_id in cluster_ids:

    mask = (
        y
        ==
        cluster_id
    )


    ax.scatter(

        X_pca[
            mask,
            0
        ],

        X_pca[
            mask,
            1
        ],

        s=18,

        alpha=0.62,

        label=(
            f"Cluster "
            f"{cluster_id}"
        )

    )


ax.axhline(
    0,
    linewidth=0.8,
    alpha=0.25
)

ax.axvline(
    0,
    linewidth=0.8,
    alpha=0.25
)


ax.set_xlabel(
    f"PC1 ({pc1_variance:.1f}% variance)"
)

ax.set_ylabel(
    f"PC2 ({pc2_variance:.1f}% variance)"
)

ax.set_title(
    "Environmental spectrum of young-family walking routes"
)

ax.legend(
    frameon=False,
    fontsize=8
)

fig.tight_layout()


PCA_CLUSTER_PATH = (
    FIGURE_DIR
    / "pca_clusters.png"
)

fig.savefig(
    PCA_CLUSTER_PATH,
    dpi=220,
    bbox_inches="tight"
)

plt.close(
    fig
)


# ============================================================
# FIGURE 2
# PCA BIPLOT
# ============================================================

fig, ax = plt.subplots(
    figsize=(10, 7)
)


ax.scatter(

    X_pca[:, 0],

    X_pca[:, 1],

    s=14,

    alpha=0.24

)


ax.axhline(
    0,
    linewidth=0.8,
    alpha=0.25
)

ax.axvline(
    0,
    linewidth=0.8,
    alpha=0.25
)


loadings = (
    pca.components_.T
)


score_scale = max(

    np.max(
        np.abs(
            X_pca[:, 0]
        )
    ),

    np.max(
        np.abs(
            X_pca[:, 1]
        )
    )

)


arrow_scale = (
    score_scale
    *
    0.72
)


for index, label_text in enumerate(
    FEATURE_LABELS
):

    dx = (
        loadings[
            index,
            0
        ]
        *
        arrow_scale
    )

    dy = (
        loadings[
            index,
            1
        ]
        *
        arrow_scale
    )


    ax.arrow(

        0,
        0,

        dx,
        dy,

        width=0.008,

        head_width=0.12,

        length_includes_head=True,

        alpha=0.82

    )


    ax.text(

        dx * 1.08,

        dy * 1.08,

        label_text,

        fontsize=9

    )


ax.set_xlabel(
    f"PC1 ({pc1_variance:.1f}% variance)"
)

ax.set_ylabel(
    f"PC2 ({pc2_variance:.1f}% variance)"
)

ax.set_title(
    "PCA biplot — environmental spectrum and contributing features"
)

fig.tight_layout()


PCA_BIPLOT_PATH = (
    FIGURE_DIR
    / "pca_biplot.png"
)

fig.savefig(
    PCA_BIPLOT_PATH,
    dpi=220,
    bbox_inches="tight"
)

plt.close(
    fig
)


# ============================================================
# CLUSTER MEAN MATRIX
# ============================================================

cluster_matrix = []


for cluster_id in cluster_ids:

    mask = (
        y
        ==
        cluster_id
    )


    means = np.mean(
        X[
            mask
        ],
        axis=0
    )


    cluster_matrix.append(
        means
    )


cluster_matrix = np.array(
    cluster_matrix
)


# Convert proportions to percentages for presentation.
display_cluster_matrix = (
    cluster_matrix
    *
    100
    if np.nanmax(
        cluster_matrix
    ) <= 1.5
    else cluster_matrix
)


# ============================================================
# FIGURE 3
# CLUSTER FEATURE HEATMAP
# ============================================================

fig, ax = plt.subplots(
    figsize=(11, 5.8)
)


image = ax.imshow(
    display_cluster_matrix,
    aspect="auto"
)


ax.set_xticks(
    range(
        len(
            FEATURE_LABELS
        )
    )
)

ax.set_xticklabels(
    FEATURE_LABELS,
    rotation=25,
    ha="right"
)


ax.set_yticks(
    range(
        len(
            cluster_ids
        )
    )
)

ax.set_yticklabels(
    [
        f"Cluster {cluster_id}"
        for cluster_id
        in cluster_ids
    ]
)


for row in range(
    display_cluster_matrix.shape[0]
):

    for column in range(
        display_cluster_matrix.shape[1]
    ):

        value = (
            display_cluster_matrix[
                row,
                column
            ]
        )


        ax.text(

            column,

            row,

            f"{value:.1f}",

            ha="center",

            va="center",

            fontsize=8

        )


ax.set_title(
    "Environmental character of each K-means cluster"
)


colour_bar = fig.colorbar(
    image,
    ax=ax
)

colour_bar.set_label(
    "Mean environmental proportion (%)"
)


fig.tight_layout()


HEATMAP_PATH = (
    FIGURE_DIR
    / "cluster_feature_heatmap.png"
)

fig.savefig(
    HEATMAP_PATH,
    dpi=220,
    bbox_inches="tight"
)

plt.close(
    fig
)


# ============================================================
# DECISION TREE
# ============================================================

tree_model = DecisionTreeClassifier(

    max_depth=4,

    min_samples_leaf=8,

    random_state=42

)


tree_model.fit(
    X_scaled,
    y
)


# ============================================================
# FIGURE 4
# DECISION-TREE FEATURE IMPORTANCE
# ============================================================

importances = (
    tree_model.feature_importances_
)


order = np.argsort(
    importances
)[::-1]


fig, ax = plt.subplots(
    figsize=(9, 5.8)
)


ax.bar(

    [
        FEATURE_LABELS[index]
        for index
        in order
    ],

    importances[
        order
    ]

)


ax.set_ylabel(
    "Feature importance"
)

ax.set_title(
    "Environmental variables most useful for distinguishing route typologies"
)

ax.tick_params(
    axis="x",
    rotation=28
)


fig.tight_layout()


IMPORTANCE_PATH = (
    FIGURE_DIR
    / "feature_importance.png"
)

fig.savefig(
    IMPORTANCE_PATH,
    dpi=220,
    bbox_inches="tight"
)

plt.close(
    fig
)


# ============================================================
# FIGURE 5
# DECISION TREE
# ============================================================

fig, ax = plt.subplots(
    figsize=(22, 12)
)


plot_tree(

    tree_model,

    feature_names=
        FEATURE_LABELS,

    class_names=
        [
            f"C{cluster_id}"
            for cluster_id
            in tree_model.classes_
        ],

    filled=True,

    rounded=True,

    impurity=False,

    proportion=True,

    fontsize=7,

    ax=ax

)


ax.set_title(
    "Decision tree — rules separating recurring environmental typologies"
)


fig.tight_layout()


TREE_PATH = (
    FIGURE_DIR
    / "decision_tree.png"
)

fig.savefig(
    TREE_PATH,
    dpi=220,
    bbox_inches="tight"
)

plt.close(
    fig
)


# ============================================================
# FIGURE 6
# HIERARCHICAL DENDROGRAM
# ============================================================

MAX_DENDROGRAM_SAMPLES = 260


rng = np.random.default_rng(
    42
)


selected_indices = []


per_cluster_target = max(

    1,

    math.ceil(
        MAX_DENDROGRAM_SAMPLES
        /
        len(
            cluster_ids
        )
    )

)


for cluster_id in cluster_ids:

    indices = np.where(
        y
        ==
        cluster_id
    )[0]


    take = min(
        len(indices),
        per_cluster_target
    )


    if take > 0:

        selected_indices.extend(

            rng.choice(

                indices,

                size=take,

                replace=False

            )
            .tolist()

        )


selected_indices = np.array(
    selected_indices[
        :MAX_DENDROGRAM_SAMPLES
    ],
    dtype=int
)


hierarchy_matrix = (
    X_scaled[
        selected_indices
    ]
)


linkage_matrix = linkage(

    hierarchy_matrix,

    method="ward"

)


fig, ax = plt.subplots(
    figsize=(16, 7)
)


dendrogram(

    linkage_matrix,

    no_labels=True,

    ax=ax

)


ax.set_xlabel(
    "Sampled street checkpoints"
)

ax.set_ylabel(
    "Ward linkage distance"
)

ax.set_title(
    "Hierarchical clustering of route environments"
)


fig.tight_layout()


DENDROGRAM_PATH = (
    FIGURE_DIR
    / "hierarchical_dendrogram.png"
)

fig.savefig(
    DENDROGRAM_PATH,
    dpi=220,
    bbox_inches="tight"
)

plt.close(
    fig
)


# ============================================================
# FIGURE 7
# REPRESENTATIVE ENVIRONMENTAL ROUTE PROFILE
# ============================================================

route_profile_path = None

route_example = None


routes = (
    compiler.get("routes")
    or
    []
)


route_candidates = []


for route in routes:

    smoothed = (
        route
        .get(
            "routeSeries",
            {}
        )
        .get(
            "smoothed",
            []
        )
    )


    if len(smoothed) >= 4:

        route_candidates.append(
            route
        )


if route_candidates:

    route_example = max(

        route_candidates,

        key=lambda route:
            len(
                route
                .get(
                    "routeSeries",
                    {}
                )
                .get(
                    "smoothed",
                    []
                )
            )

    )


    smoothed = (
        route_example
        .get(
            "routeSeries",
            {}
        )
        .get(
            "smoothed",
            []
        )
    )


    route_distance = np.array(

        [
            safe_float(
                point.get(
                    "distance_m"
                )
            )
            for point
            in smoothed
        ],

        dtype=float

    )


    series_definition = [

        (
            "sky",
            "Sky"
        ),

        (
            "greenery",
            "Greenery"
        ),

        (
            "road",
            "Road"
        ),

        (
            "pedestrian",
            "Pedestrian"
        )

    ]


    fig, ax = plt.subplots(
        figsize=(13, 5.8)
    )


    for key, label_text in series_definition:

        values = np.array(

            [
                safe_float(

                    point
                    .get(
                        "values",
                        {}
                    )
                    .get(
                        key
                    ),

                    np.nan

                )

                for point
                in smoothed
            ],

            dtype=float

        )


        if (
            np.nanmax(
                values
            )
            <= 1.5
        ):

            values = (
                values
                *
                100
            )


        ax.plot(

            route_distance,

            values,

            linewidth=2.4,

            label=label_text

        )


    ax.set_ylim(
        0,
        100
    )

    ax.set_xlabel(
        "Distance along walking route (m)"
    )

    ax.set_ylabel(
        "Environmental proportion (%)"
    )


    station_name = (
        route_example.get("station")
        or
        "MRT"
    )


    destination_name = (
        route_example.get("destination")
        or
        route_example.get("toilet")
        or
        "destination"
    )


    ax.set_title(

        f"Environmental route profile — "
        f"{station_name} → {destination_name}"

    )


    ax.legend(
        frameon=False,
        ncol=4
    )


    fig.tight_layout()


    route_profile_path = (
        FIGURE_DIR
        / "route_profile_example.png"
    )


    fig.savefig(
        route_profile_path,
        dpi=220,
        bbox_inches="tight"
    )


    plt.close(
        fig
    )


# ============================================================
# WEBSITE MANIFEST
# ============================================================

manifest = {

    "sampleCount":
        len(records),

    "clusterCount":
        len(cluster_ids),

    "features":
        FEATURE_LABELS,

    "pca": {

        "pc1VariancePercent":
            round(
                pc1_variance,
                3
            ),

        "pc2VariancePercent":
            round(
                pc2_variance,
                3
            ),

        "firstTwoComponentsPercent":
            round(
                total_variance,
                3
            )

    },

    "figures": {

        "pcaClusters":
            "./data/generated/ml_figures/pca_clusters.png",

        "pcaBiplot":
            "./data/generated/ml_figures/pca_biplot.png",

        "clusterHeatmap":
            "./data/generated/ml_figures/cluster_feature_heatmap.png",

        "featureImportance":
            "./data/generated/ml_figures/feature_importance.png",

        "dendrogram":
            "./data/generated/ml_figures/hierarchical_dendrogram.png",

        "decisionTree":
            "./data/generated/ml_figures/decision_tree.png",

        "routeProfile":
            (
                "./data/generated/ml_figures/route_profile_example.png"
                if route_profile_path
                else None
            )

    }

}


MANIFEST_PATH = (
    FIGURE_DIR
    / "ml_figures_manifest.json"
)


with open(
    MANIFEST_PATH,
    "w",
    encoding="utf-8"
) as file:

    json.dump(
        manifest,
        file,
        indent=2
    )


print()
print(
    "============================================"
)

print(
    "WEBSITE ML FIGURES EXPORTED"
)

print(
    "============================================"
)

print(
    f"PCA variance, first two components: "
    f"{total_variance:.1f}%"
)

print(
    "Figures folder:"
)

print(
    FIGURE_DIR
)

print()
print(
    "✓ PCA clusters"
)

print(
    "✓ PCA biplot"
)

print(
    "✓ K-means feature heatmap"
)

print(
    "✓ Decision-tree feature importance"
)

print(
    "✓ Decision tree"
)

print(
    "✓ Hierarchical dendrogram"
)

if route_profile_path:

    print(
        "✓ Environmental route profile"
    )

else:

    print(
        "○ No compiled route profile was available"
    )

print(
    "============================================"
)

Analysed images: 359
Usable ML samples: 356
Visual clusters: [0, 1, 2, 3, 4, 5, 6, 7, 8, 9]

WEBSITE ML FIGURES EXPORTED
PCA variance, first two components: 52.3%
Figures folder:
C:\Users\MSI\OneDrive\Desktop\family-toilet-search\data\generated\ml_figures

✓ PCA clusters
✓ PCA biplot
✓ K-means feature heatmap
✓ Decision-tree feature importance
✓ Decision tree
✓ Hierarchical dendrogram
✓ Environmental route profile


In [17]:
# ============================================================
# V2 PATCH — STANDALONE ROUTE LOADER
#
# This does NOT crawl anything.
# This does NOT rerun SegFormer.
#
# It only reloads the already-generated route_intelligence JSON
# files so the final V2 cell can run independently.
# ============================================================

from pathlib import Path
import json


# ------------------------------------------------------------
# Resolve project folders if they are not already in memory
# ------------------------------------------------------------

if "PROJECT_ROOT" not in globals():

    NOTEBOOK_DIR = Path.cwd()

    PROJECT_ROOT = (
        NOTEBOOK_DIR.parent
        if NOTEBOOK_DIR.name.lower() == "notebooks"
        else NOTEBOOK_DIR
    )


if "GENERATED_DIR" not in globals():

    GENERATED_DIR = (
        PROJECT_ROOT
        / "data"
        / "generated"
    )


if "ROUTE_INTELLIGENCE_DIR" not in globals():

    ROUTE_INTELLIGENCE_DIR = (
        GENERATED_DIR
        / "route_intelligence"
    )


# ------------------------------------------------------------
# JSON reader
# ------------------------------------------------------------

def _v2_read_json(path):

    try:

        with open(
            path,
            "r",
            encoding="utf-8"
        ) as file:

            return json.load(file)

    except Exception as error:

        print(
            "Could not read:",
            path.name,
            "|",
            error
        )

        return None


# ------------------------------------------------------------
# Check whether something looks like one route
# ------------------------------------------------------------

def _v2_valid_route(route):

    return (

        isinstance(
            route,
            dict
        )

        and

        bool(
            route.get(
                "routeId"
            )
        )

        and

        isinstance(
            route.get(
                "checkpoints",
                []
            ),
            list
        )

    )


# ------------------------------------------------------------
# Extract routes from different JSON layouts
# ------------------------------------------------------------

def _v2_extract_routes(payload):

    if isinstance(
        payload,
        list
    ):

        return [
            item
            for item
            in payload
            if _v2_valid_route(
                item
            )
        ]


    if not isinstance(
        payload,
        dict
    ):

        return []


    if _v2_valid_route(
        payload
    ):

        return [
            payload
        ]


    for key in [
        "results",
        "routes",
        "data",
        "items"
    ]:

        values = payload.get(
            key
        )


        if isinstance(
            values,
            list
        ):

            valid = [
                item
                for item
                in values
                if _v2_valid_route(
                    item
                )
            ]


            if valid:
                return valid


    return []


# ------------------------------------------------------------
# New independent route loader
# ------------------------------------------------------------

def load_routes_for_analysis():

    collected = {}


    print()
    print(
        "Reloading completed route intelligence..."
    )

    print(
        "Route folder:",
        ROUTE_INTELLIGENCE_DIR
    )


    if not ROUTE_INTELLIGENCE_DIR.exists():

        raise RuntimeError(

            "route_intelligence folder does not exist:\n"
            f"{ROUTE_INTELLIGENCE_DIR}"

        )


    # ========================================================
    # 1. latest_batch.json
    # ========================================================

    latest_path = (
        ROUTE_INTELLIGENCE_DIR
        / "latest_batch.json"
    )


    if latest_path.exists():

        payload = _v2_read_json(
            latest_path
        )


        routes = _v2_extract_routes(
            payload
        )


        print(
            "latest_batch.json:",
            len(
                routes
            ),
            "routes"
        )


        for route in routes:

            collected[
                str(
                    route[
                        "routeId"
                    ]
                )
            ] = route


    # ========================================================
    # 2. Individual route files
    # ========================================================

    individual_count = 0


    for path in sorted(
        ROUTE_INTELLIGENCE_DIR.glob(
            "*.json"
        )
    ):

        if (
            path.name
            ==
            "latest_batch.json"
        ):

            continue


        payload = _v2_read_json(
            path
        )


        routes = _v2_extract_routes(
            payload
        )


        for route in routes:

            collected[
                str(
                    route[
                        "routeId"
                    ]
                )
            ] = route


            individual_count += 1


    print(
        "Individual route records:",
        individual_count
    )


    # ========================================================
    # 3. Recursive fallback
    #
    # Useful if some route files are inside subfolders.
    # ========================================================

    recursive_count = 0


    for path in sorted(
        ROUTE_INTELLIGENCE_DIR.glob(
            "**/*.json"
        )
    ):

        if (
            path.parent
            ==
            ROUTE_INTELLIGENCE_DIR
        ):

            continue


        payload = _v2_read_json(
            path
        )


        routes = _v2_extract_routes(
            payload
        )


        for route in routes:

            route_id = str(
                route[
                    "routeId"
                ]
            )


            if (
                route_id
                not in
                collected
            ):

                recursive_count += 1


            collected[
                route_id
            ] = route


    if recursive_count:

        print(
            "Recursive additional routes:",
            recursive_count
        )


    routes = list(
        collected.values()
    )


    routes.sort(
        key=lambda route:
            str(
                route.get(
                    "routeId",
                    ""
                )
            )
    )


    print()
    print(
        "============================================"
    )

    print(
        "ROUTES LOADED:",
        len(
            routes
        )
    )

    print(
        "============================================"
    )


    for route in routes[:10]:

        print(

            "•",

            route.get(
                "routeId"
            ),

            "|",

            route.get(
                "station"
            ),

            "| checkpoints:",

            len(
                route.get(
                    "checkpoints",
                    []
                )
            )

        )


    if not routes:

        raise RuntimeError(

            "No completed route records were found in:\n"
            f"{ROUTE_INTELLIGENCE_DIR}"

        )


    return routes


# ------------------------------------------------------------
# Test ONLY the loader
# ------------------------------------------------------------

ROUTE_RECORDS_V2_TEST = (
    load_routes_for_analysis()
)


print()
print(
    "✓ V2 route loader is ready."
)


Reloading completed route intelligence...
Route folder: C:\Users\MSI\OneDrive\Desktop\family-toilet-search\data\generated\route_intelligence
latest_batch.json: 192 routes
Individual route records: 192

ROUTES LOADED: 192
• ADMIRALTY__R01 | ADMIRALTY MRT | checkpoints: 79
• ALJUNIED__R01 | ALJUNIED MRT | checkpoints: 3
• ANG_MO_KIO__R01 | ANG MO KIO MRT | checkpoints: 93
• BEAUTY_WORLD__R01 | BEAUTY WORLD MRT | checkpoints: 13
• BEDOK_NORTH__R01 | BEDOK NORTH MRT | checkpoints: 46
• BEDOK_RESERVOIR__R01 | BEDOK RESERVOIR MRT | checkpoints: 22
• BEDOK__R01 | BEDOK MRT | checkpoints: 95
• BISHAN__R01 | BISHAN MRT | checkpoints: 45
• BOON_KENG__R01 | BOON KENG MRT | checkpoints: 84
• BOON_LAY__R01 | BOON LAY MRT | checkpoints: 5

✓ V2 route loader is ready.


In [18]:
# ============================================================
# FINAL CELL — ROUTE-ORDERED IMAGE ANALYSIS V2
#
# APPEND-ONLY VERSION.
#
# Does NOT modify the previous compiler files.
#
# New outputs:
#
# data/generated/image_compiler_route_v2.json
# data/generated/image_analysis_route_v2.json
# data/generated/route_image_index_v2.json
#
# Images:
#
# data/generated/street_images/by_mrt_route_v2/
#
# Segmentations:
#
# data/generated/segmentation_images/by_mrt_route_v2/
#
# DISTANCE CONVENTION:
#
# MRT = 0 m
# HOME = full walking-route distance
#
# ============================================================


from pathlib import Path
from collections import defaultdict, Counter
from datetime import datetime

import math
import time
import json
import re
import requests
import numpy as np


# ============================================================
# SETTINGS
# ============================================================

V2_IMAGE_SPACING_M = 60.0

# Maximum number of sampled street images on one journey.
# 60 images still allows a 3 km route to be represented
# at approximately 50–60 m intervals.
V2_MAX_IMAGES_PER_ROUTE = 70

# Mapillary image must lie reasonably close to route checkpoint.
V2_MAX_IMAGE_DISTANCE_M = 45.0

# Small padding around full walking-route bounding box.
V2_ROUTE_PADDING_M = 55.0

V2_MAPILLARY_PAGE_LIMIT = 1000
V2_MAX_MAPILLARY_PAGES = 8

# More gradual spatial profile.
# This changes ONLY the new V2 compile below.
ROUTE_PROFILE_STEP_M = 10.0
ROUTE_SMOOTHING_SIGMA_M = 90.0
ROUTE_SMOOTHING_MAX_GAP_M = 180.0


# ============================================================
# NEW OUTPUT PATHS
# ============================================================

V2_FULL_ANALYSIS_PATH = (
    GENERATED_DIR
    / "image_analysis_route_v2.json"
)

V2_COMPILER_PATH = (
    GENERATED_DIR
    / "image_compiler_route_v2.json"
)

V2_ROUTE_IMAGE_INDEX_PATH = (
    GENERATED_DIR
    / "route_image_index_v2.json"
)

V2_ROUTE_MANIFEST_DIR = (
    GENERATED_DIR
    / "route_image_manifests_v2"
)

V2_MAPILLARY_CACHE_DIR = (
    GENERATED_DIR
    / "mapillary_route_candidates_v2"
)


V2_STREET_DIR = (
    STREET_IMAGE_DIR
    / "by_mrt_route_v2"
)

V2_SEGMENTATION_DIR = (
    SEGMENTATION_IMAGE_DIR
    / "by_mrt_route_v2"
)


for folder in [
    V2_ROUTE_MANIFEST_DIR,
    V2_MAPILLARY_CACHE_DIR,
    V2_STREET_DIR,
    V2_SEGMENTATION_DIR,
]:
    folder.mkdir(
        parents=True,
        exist_ok=True
    )


# ============================================================
# TEXT HELPERS
# ============================================================

def v2_clean(value):
    return str(
        value
        or
        ""
    ).strip()


def v2_is_mrt(value):

    return bool(
        re.search(
            r"\bMRT\b",
            v2_clean(value),
            flags=re.I
        )
    )


def v2_is_residential(value):

    return (
        "RESIDENTIAL"
        in
        v2_clean(value).upper()
    )


def v2_title_slug(value):

    return (
        v2_clean(value)
        .replace("__", " ")
        .replace("_", " ")
        .replace("-", " ")
        .title()
        .strip()
    )


def v2_normalise_mrt(value):

    text = (
        v2_clean(value)
        .upper()
    )

    text = re.sub(
        r"\s+MRT\s+STATION$",
        "",
        text
    )

    text = re.sub(
        r"\s+STATION$",
        "",
        text
    )

    text = re.sub(
        r"\s+MRT$",
        "",
        text
    )

    text = text.strip()

    return (
        f"{text} MRT"
        if text
        else None
    )


def v2_residential_name(value):

    text = v2_clean(
        value
    )

    match = re.search(
        r"residential[-_ ]?0*(\d+)",
        text,
        flags=re.I
    )

    if match:

        return (
            "Residential Cluster "
            +
            str(
                int(
                    match.group(1)
                )
            ).zfill(3)
        )

    return v2_title_slug(
        text
    )


def v2_residential_number(value):

    match = re.search(
        r"(\d+)",
        v2_clean(value)
    )

    return (
        int(
            match.group(1)
        )
        if match
        else 999999
    )


# ============================================================
# ROUTE IDENTITY
#
# Example:
#
# residential-0026__jurong-east-mrt
#
# Original route:
#
# HOME -> MRT
#
# New research representation:
#
# MRT -> HOME
# ============================================================

def v2_route_identity(route):

    route_id = v2_clean(
        route.get(
            "routeId"
        )
    )


    # --------------------------------------------------------
    # Route ID is our strongest identity source.
    # --------------------------------------------------------

    if (
        route_id.lower().startswith(
            "residential"
        )
        and
        "__"
        in
        route_id
    ):

        parts = route_id.split(
            "__"
        )

        home_slug = parts[0]

        mrt_slug = parts[-1]

        return {

            "routeId":
                route_id,

            "mrtName":
                v2_normalise_mrt(
                    v2_title_slug(
                        mrt_slug
                    )
                ),

            "homeName":
                v2_residential_name(
                    home_slug
                ),

            "sourceOrientation":
                "home_to_mrt",
        }


    station = v2_clean(
        route.get(
            "station"
        )
    )

    destination = v2_clean(
        route.get(
            "destination"
        )
        or
        route.get(
            "toilet"
        )
    )


    if (
        v2_is_residential(
            station
        )
        and
        v2_is_mrt(
            destination
        )
    ):

        return {

            "routeId":
                route_id,

            "mrtName":
                v2_normalise_mrt(
                    destination
                ),

            "homeName":
                station,

            "sourceOrientation":
                "home_to_mrt",
        }


    if (
        v2_is_mrt(
            station
        )
        and
        v2_is_residential(
            destination
        )
    ):

        return {

            "routeId":
                route_id,

            "mrtName":
                v2_normalise_mrt(
                    station
                ),

            "homeName":
                destination,

            "sourceOrientation":
                "mrt_to_home",
        }


    if v2_is_mrt(
        destination
    ):

        return {

            "routeId":
                route_id,

            "mrtName":
                v2_normalise_mrt(
                    destination
                ),

            "homeName":
                station
                or
                "Residential destination",

            "sourceOrientation":
                "home_to_mrt",
        }


    if v2_is_mrt(
        station
    ):

        return {

            "routeId":
                route_id,

            "mrtName":
                v2_normalise_mrt(
                    station
                ),

            "homeName":
                destination
                or
                "Residential destination",

            "sourceOrientation":
                "mrt_to_home",
        }


    return {

        "routeId":
            route_id,

        "mrtName":
            None,

        "homeName":
            station
            or
            destination
            or
            "Residential destination",

        "sourceOrientation":
            "unknown",
    }


# ============================================================
# ROUTE DISTANCE
# ============================================================

def v2_route_distance(route):

    summary = (
        route.get(
            "routeSummary"
        )
        or
        {}
    )


    distance = safe_float(
        summary.get(
            "distance_m"
        ),
        0.0
    )


    if distance > 0:
        return float(
            distance
        )


    distances = [

        safe_float(
            checkpoint.get(
                "distanceAlongRoute_m"
            ),
            0.0
        )

        for checkpoint
        in (
            route.get(
                "checkpoints"
            )
            or
            []
        )

    ]


    return max(
        distances,
        default=0.0
    )


# ============================================================
# GEOGRAPHIC HELPERS
# ============================================================

def v2_haversine(
    lat1,
    lon1,
    lat2,
    lon2
):

    radius = 6_371_000.0

    phi1 = math.radians(
        lat1
    )

    phi2 = math.radians(
        lat2
    )

    dphi = math.radians(
        lat2
        -
        lat1
    )

    dlambda = math.radians(
        lon2
        -
        lon1
    )


    a = (
        math.sin(
            dphi / 2
        ) ** 2
        +
        math.cos(
            phi1
        )
        *
        math.cos(
            phi2
        )
        *
        math.sin(
            dlambda / 2
        ) ** 2
    )


    return (
        2
        *
        radius
        *
        math.atan2(
            math.sqrt(a),
            math.sqrt(1 - a)
        )
    )


def v2_bearing(
    lat1,
    lon1,
    lat2,
    lon2
):

    phi1 = math.radians(
        lat1
    )

    phi2 = math.radians(
        lat2
    )

    delta_lon = math.radians(
        lon2
        -
        lon1
    )


    x = (
        math.sin(
            delta_lon
        )
        *
        math.cos(
            phi2
        )
    )


    y = (
        math.cos(
            phi1
        )
        *
        math.sin(
            phi2
        )
        -
        math.sin(
            phi1
        )
        *
        math.cos(
            phi2
        )
        *
        math.cos(
            delta_lon
        )
    )


    return (
        math.degrees(
            math.atan2(
                x,
                y
            )
        )
        +
        360
    ) % 360


def v2_angle_difference(
    first,
    second
):

    difference = abs(
        float(first)
        -
        float(second)
    ) % 360


    return min(
        difference,
        360 - difference
    )


# ============================================================
# ORIENT ORIGINAL CHECKPOINTS
#
# Every route returned from this function is:
#
# MRT = 0
# HOME = routeDistance
# ============================================================

def v2_oriented_checkpoints(
    route
):

    identity = v2_route_identity(
        route
    )


    route_distance = (
        v2_route_distance(
            route
        )
    )


    source = (
        route.get(
            "checkpoints"
        )
        or
        []
    )


    output = []


    for fallback_index, checkpoint in enumerate(
        source
    ):

        try:

            lat = float(
                checkpoint.get(
                    "lat"
                )
            )

            lon = float(
                checkpoint.get(
                    "lon"
                )
            )

        except Exception:
            continue


        original_distance = safe_float(
            checkpoint.get(
                "distanceAlongRoute_m"
            ),
            np.nan
        )


        if not math.isfinite(
            original_distance
        ):

            if len(
                source
            ) > 1:

                original_distance = (
                    route_distance
                    *
                    fallback_index
                    /
                    (
                        len(source)
                        -
                        1
                    )
                )

            else:

                original_distance = 0.0


        if (
            identity[
                "sourceOrientation"
            ]
            ==
            "home_to_mrt"
        ):

            distance_from_mrt = max(
                0.0,
                route_distance
                -
                original_distance
            )

        else:

            distance_from_mrt = max(
                0.0,
                original_distance
            )


        output.append({

            **checkpoint,

            "_lat":
                lat,

            "_lon":
                lon,

            "_distanceFromMrt_m":
                distance_from_mrt,

            "_sourceDistance_m":
                original_distance,

            "_sourceCheckpointIndex":
                int(
                    checkpoint.get(
                        "checkpointIndex"
                    )
                    or
                    fallback_index
                ),
        })


    output.sort(
        key=lambda checkpoint:
            checkpoint[
                "_distanceFromMrt_m"
            ]
    )


    # --------------------------------------------------------
    # Estimate route direction at every checkpoint.
    # --------------------------------------------------------

    for index, checkpoint in enumerate(
        output
    ):

        previous = output[
            max(
                0,
                index - 1
            )
        ]

        following = output[
            min(
                len(output) - 1,
                index + 1
            )
        ]


        if (
            previous["_lat"]
            ==
            following["_lat"]
            and
            previous["_lon"]
            ==
            following["_lon"]
        ):

            checkpoint[
                "_routeBearing"
            ] = None

        else:

            checkpoint[
                "_routeBearing"
            ] = v2_bearing(
                previous["_lat"],
                previous["_lon"],
                following["_lat"],
                following["_lon"]
            )


    return output


# ============================================================
# SPATIALLY EVEN ROUTE CHECKPOINTS
# ============================================================

def v2_sample_checkpoints(
    route
):

    checkpoints = (
        v2_oriented_checkpoints(
            route
        )
    )


    if not checkpoints:
        return []


    route_distance = (
        v2_route_distance(
            route
        )
    )


    targets = list(
        np.arange(
            0.0,
            route_distance
            +
            V2_IMAGE_SPACING_M,
            V2_IMAGE_SPACING_M
        )
    )


    if not targets:
        targets = [
            0.0
        ]


    if (
        targets[-1]
        <
        route_distance
    ):

        targets.append(
            route_distance
        )


    selected = []

    used = set()


    for target in targets:

        nearest_index = min(

            range(
                len(
                    checkpoints
                )
            ),

            key=lambda index:
                abs(
                    checkpoints[index][
                        "_distanceFromMrt_m"
                    ]
                    -
                    target
                )
        )


        if nearest_index in used:
            continue


        used.add(
            nearest_index
        )

        selected.append(
            checkpoints[
                nearest_index
            ]
        )


    if (
        len(selected)
        >
        V2_MAX_IMAGES_PER_ROUTE
    ):

        indices = np.linspace(
            0,
            len(selected) - 1,
            V2_MAX_IMAGES_PER_ROUTE
        )


        indices = sorted({

            int(
                round(
                    index
                )
            )

            for index
            in indices

        })


        selected = [

            selected[index]

            for index
            in indices

        ]


    return selected


# ============================================================
# ROUTE BOUNDING BOX
# ============================================================

def v2_route_bbox(
    checkpoints
):

    if not checkpoints:
        return None


    latitudes = [
        checkpoint["_lat"]
        for checkpoint
        in checkpoints
    ]


    longitudes = [
        checkpoint["_lon"]
        for checkpoint
        in checkpoints
    ]


    centre_latitude = float(
        np.mean(
            latitudes
        )
    )


    latitude_padding = (
        V2_ROUTE_PADDING_M
        /
        111_320.0
    )


    longitude_padding = (
        V2_ROUTE_PADDING_M
        /
        (
            111_320.0
            *
            max(
                0.25,
                math.cos(
                    math.radians(
                        centre_latitude
                    )
                )
            )
        )
    )


    return (

        min(longitudes)
        -
        longitude_padding,

        min(latitudes)
        -
        latitude_padding,

        max(longitudes)
        +
        longitude_padding,

        max(latitudes)
        +
        latitude_padding,

    )


# ============================================================
# FRESH MAPILLARY CRAWL — ONE BBOX PER ROUTE
# ============================================================

def v2_mapillary_candidates(
    route,
    checkpoints,
    force=True
):

    route_id = v2_clean(
        route.get(
            "routeId"
        )
    )


    cache_path = (
        V2_MAPILLARY_CACHE_DIR
        /
        f"{slugify(route_id)}.json"
    )


    if (
        cache_path.exists()
        and
        not force
    ):

        cached = load_json(
            cache_path
        )

        return (
            cached.get(
                "images"
            )
            or
            []
        )


    bbox = v2_route_bbox(
        checkpoints
    )


    if bbox is None:
        return []


    bbox_string = ",".join(
        f"{value:.7f}"
        for value
        in bbox
    )


    url = (
        f"{MAPILLARY_GRAPH_URL}/images"
    )


    params = {

        "access_token":
            MAPILLARY_ACCESS_TOKEN,

        "bbox":
            bbox_string,

        "fields":
            (
                "id,"
                "geometry,"
                "computed_geometry,"
                "captured_at,"
                "compass_angle,"
                "sequence"
            ),

        "limit":
            V2_MAPILLARY_PAGE_LIMIT,
    }


    images = []

    page = 0


    while (
        url
        and
        page
        <
        V2_MAX_MAPILLARY_PAGES
    ):

        page += 1


        response = requests.get(
            url,
            params=params,
            timeout=REQUEST_TIMEOUT
        )


        response.raise_for_status()


        payload = response.json()


        images.extend(
            payload.get(
                "data"
            )
            or
            []
        )


        url = (
            payload
            .get(
                "paging",
                {}
            )
            .get(
                "next"
            )
        )


        # paging.next already contains query parameters.
        params = None


    unique = {}


    for image in images:

        image_id = v2_clean(
            image.get(
                "id"
            )
        )


        if image_id:

            unique[
                image_id
            ] = image


    images = list(
        unique.values()
    )


    safe_json_dump(
        cache_path,
        {
            "routeId":
                route_id,

            "bbox":
                list(
                    bbox
                ),

            "candidateCount":
                len(
                    images
                ),

            "images":
                images,
        }
    )


    return images


# ============================================================
# IMAGE COORDINATE
# ============================================================

def v2_image_coordinates(
    image
):

    geometry = (
        image.get(
            "computed_geometry"
        )
        or
        image.get(
            "geometry"
        )
        or
        {}
    )


    coordinates = (
        geometry.get(
            "coordinates"
        )
        or
        []
    )


    if len(
        coordinates
    ) < 2:

        return None


    lon = safe_float(
        coordinates[0],
        np.nan
    )

    lat = safe_float(
        coordinates[1],
        np.nan
    )


    if (
        not math.isfinite(
            lat
        )
        or
        not math.isfinite(
            lon
        )
    ):

        return None


    return (
        lat,
        lon
    )


# ============================================================
# FIND BEST STREET IMAGE FOR CHECKPOINT
# ============================================================

def v2_best_image(
    checkpoint,
    candidates,
    used_ids,
    previous_sequence=None
):

    options = []


    route_bearing = (
        checkpoint.get(
            "_routeBearing"
        )
    )


    for image in candidates:

        image_id = v2_clean(
            image.get(
                "id"
            )
        )


        if (
            not image_id
            or
            image_id in used_ids
        ):

            continue


        coordinates = (
            v2_image_coordinates(
                image
            )
        )


        if coordinates is None:
            continue


        image_lat, image_lon = (
            coordinates
        )


        distance = v2_haversine(
            checkpoint["_lat"],
            checkpoint["_lon"],
            image_lat,
            image_lon
        )


        if (
            distance
            >
            V2_MAX_IMAGE_DISTANCE_M
        ):

            continue


        score = distance


        # ----------------------------------------------------
        # Orientation preference.
        # ----------------------------------------------------

        compass = safe_float(
            image.get(
                "compass_angle"
            ),
            np.nan
        )


        if (
            route_bearing
            is not None
            and
            math.isfinite(
                compass
            )
        ):

            forward_error = (
                v2_angle_difference(
                    compass,
                    route_bearing
                )
            )


            reverse_error = (
                v2_angle_difference(
                    compass,
                    (
                        route_bearing
                        +
                        180
                    )
                    %
                    360
                )
            )


            heading_error = min(
                forward_error,
                reverse_error
            )


            score += (
                heading_error
                *
                0.07
            )


        # ----------------------------------------------------
        # Slight preference for sequence continuity.
        # ----------------------------------------------------

        sequence = v2_clean(
            image.get(
                "sequence"
            )
        )


        if (
            previous_sequence
            and
            sequence
            ==
            previous_sequence
        ):

            score -= 5.0


        options.append(
            (
                score,
                distance,
                image
            )
        )


    if not options:
        return None


    options.sort(
        key=lambda item:
            item[0]
    )


    _, distance, image = (
        options[0]
    )


    return {
        "image":
            image,

        "distanceToCheckpoint_m":
            distance
    }


# ============================================================
# DETERMINE ROUTE ORDER WITHIN EACH MRT
# ============================================================

def v2_route_orders(
    routes
):

    grouped = defaultdict(
        list
    )


    for route in routes:

        identity = (
            v2_route_identity(
                route
            )
        )


        if not identity[
            "mrtName"
        ]:

            continue


        grouped[
            identity[
                "mrtName"
            ]
        ].append(
            (
                v2_residential_number(
                    identity[
                        "homeName"
                    ]
                ),

                identity[
                    "homeName"
                ],

                identity[
                    "routeId"
                ]
            )
        )


    output = {}


    for mrt_name, members in (
        grouped.items()
    ):

        members.sort(
            key=lambda item: (
                item[0],
                item[1],
                item[2]
            )
        )


        for route_order, (
            _,
            _,
            route_id
        ) in enumerate(
            members,
            start=1
        ):

            output[
                route_id
            ] = route_order


    return output


# ============================================================
# NEW IMAGE STORAGE CONVENTION
#
# These REDEFINE the functions only after all previous cells
# have finished. Existing download/SegFormer functions will
# automatically call these newer definitions.
# ============================================================

def image_storage_paths(
    reference
):

    mrt_key = (
        reference[
            "mrtKey"
        ]
    )


    route_key = (
        reference[
            "routeKey"
        ]
    )


    storage_stem = (
        reference[
            "storageStem"
        ]
    )


    folder = (
        V2_STREET_DIR
        /
        mrt_key
        /
        route_key
    )


    folder.mkdir(
        parents=True,
        exist_ok=True
    )


    return (

        folder
        /
        f"{storage_stem}.jpg",

        folder
        /
        f"{storage_stem}.json"

    )


def segmentation_preview_path(
    item
):

    folder = (
        V2_SEGMENTATION_DIR
        /
        item["mrtKey"]
        /
        item["routeKey"]
    )


    folder.mkdir(
        parents=True,
        exist_ok=True
    )


    return (
        folder
        /
        f"{item['storageStem']}.jpg"
    )


# ============================================================
# FRESH ROUTE-SPECIFIC REFERENCE CRAWL
# ============================================================

def v2_build_image_references(
    routes,
    force=True,
    mrt_filter=None
):

    route_orders = (
        v2_route_orders(
            routes
        )
    )


    wanted_mrts = None


    if mrt_filter:

        wanted_mrts = {

            v2_normalise_mrt(
                station
            )

            for station
            in mrt_filter

        }


    usable_routes = []


    for route in routes:

        identity = (
            v2_route_identity(
                route
            )
        )


        if not identity[
            "mrtName"
        ]:

            continue


        if (
            wanted_mrts
            and
            identity[
                "mrtName"
            ]
            not in
            wanted_mrts
        ):

            continue


        usable_routes.append(
            route
        )


    usable_routes.sort(

        key=lambda route: (

            v2_route_identity(
                route
            )[
                "mrtName"
            ],

            route_orders.get(
                v2_clean(
                    route.get(
                        "routeId"
                    )
                ),
                999999
            )

        )

    )


    references = []


    for route_number, route in enumerate(
        usable_routes,
        start=1
    ):

        identity = (
            v2_route_identity(
                route
            )
        )


        mrt_name = (
            identity[
                "mrtName"
            ]
        )


        home_name = (
            identity[
                "homeName"
            ]
        )


        route_id = (
            identity[
                "routeId"
            ]
        )


        route_order = (
            route_orders.get(
                route_id,
                route_number
            )
        )


        mrt_key = slugify(
            mrt_name
        )


        route_key = (
            f"r{route_order:03d}"
            f"__{slugify(home_name)}"
        )


        checkpoints = (
            v2_sample_checkpoints(
                route
            )
        )


        print()
        print(
            "============================================"
        )

        print(
            f"[{route_number}/{len(usable_routes)}]"
        )

        print(
            mrt_name,
            "→",
            home_name
        )

        print(
            "Route:",
            route_id
        )

        print(
            "Distance:",
            round(
                v2_route_distance(
                    route
                ),
                1
            ),
            "m"
        )

        print(
            "Target checkpoints:",
            len(
                checkpoints
            )
        )


        if not checkpoints:
            continue


        try:

            candidates = (
                v2_mapillary_candidates(
                    route,
                    checkpoints,
                    force=force
                )
            )

        except Exception as error:

            print(
                "Mapillary crawl failed:",
                error
            )

            continue


        print(
            "Mapillary candidates:",
            len(
                candidates
            )
        )


        used_ids = set()

        selected = []

        previous_sequence = None


        for checkpoint in checkpoints:

            match = v2_best_image(
                checkpoint,
                candidates,
                used_ids,
                previous_sequence
            )


            if match is None:
                continue


            image = (
                match[
                    "image"
                ]
            )


            image_id = v2_clean(
                image.get(
                    "id"
                )
            )


            used_ids.add(
                image_id
            )


            sequence = v2_clean(
                image.get(
                    "sequence"
                )
            )


            if sequence:
                previous_sequence = sequence


            coordinates = (
                v2_image_coordinates(
                    image
                )
            )


            context = (
                context_for_station(
                    mrt_name
                )
            )


            som_cell = (
                context.get(
                    "somCell"
                )
            )


            mrt_cluster = (
                f"SOM-{som_cell}"
                if som_cell is not None
                else
                "UNKNOWN"
            )


            selected.append({

                "imageId":
                    image_id,

                "mrtName":
                    mrt_name,

                "station":
                    mrt_name,

                "stationNormalized":
                    normalize_station_name(
                        mrt_name
                    ),

                "homeName":
                    home_name,

                "routeId":
                    route_id,

                "routeOrder":
                    route_order,

                "routeKey":
                    route_key,

                "mrtKey":
                    mrt_key,

                "sourceOrientation":
                    identity[
                        "sourceOrientation"
                    ],

                "checkpointIndex":
                    int(
                        checkpoint[
                            "_sourceCheckpointIndex"
                        ]
                    ),

                "distanceFromMrt_m":
                    round(
                        checkpoint[
                            "_distanceFromMrt_m"
                        ],
                        2
                    ),

                # Existing aggregation functions read this key.
                # In V2 this now explicitly means MRT -> HOME.
                "distanceAlongRoute_m":
                    round(
                        checkpoint[
                            "_distanceFromMrt_m"
                        ],
                        2
                    ),

                "checkpointLat":
                    checkpoint[
                        "_lat"
                    ],

                "checkpointLon":
                    checkpoint[
                        "_lon"
                    ],

                "imageLat":
                    coordinates[0]
                    if coordinates
                    else None,

                "imageLon":
                    coordinates[1]
                    if coordinates
                    else None,

                "distanceToCheckpoint_m":
                    round(
                        match[
                            "distanceToCheckpoint_m"
                        ],
                        2
                    ),

                "capturedAt":
                    image.get(
                        "captured_at"
                    ),

                "compassAngle":
                    image.get(
                        "compass_angle"
                    ),

                "sequenceId":
                    sequence
                    or
                    None,

                "somCell":
                    som_cell,

                "mrtCluster":
                    mrt_cluster,

                "contextDescriptor":
                    context.get(
                        "contextDescriptor"
                    ),
            })


        # ----------------------------------------------------
        # Definitive spatial ordering:
        #
        # MRT -> HOME
        # ----------------------------------------------------

        selected.sort(
            key=lambda item:
                item[
                    "distanceFromMrt_m"
                ]
        )


        for image_order, item in enumerate(
            selected,
            start=1
        ):

            item[
                "imageOrder"
            ] = image_order


            item[
                "imageKey"
            ] = (
                f"IMG{image_order:04d}"
            )


            item[
                "crawlKey"
            ] = (

                f"{mrt_key}"
                f"::R{route_order:03d}"
                f"::IMG{image_order:04d}"

            )


            item[
                "storageStem"
            ] = (

                f"r{route_order:03d}"
                f"__img{image_order:04d}"
                f"__d{int(round(item['distanceFromMrt_m'])):05d}m"
                f"__cp{int(item['checkpointIndex']):04d}"
                f"__{item['imageId']}"

            )


        references.extend(
            selected
        )


        manifest_folder = (
            V2_ROUTE_MANIFEST_DIR
            /
            mrt_key
        )


        manifest_folder.mkdir(
            parents=True,
            exist_ok=True
        )


        safe_json_dump(

            manifest_folder
            /
            f"{route_key}.json",

            {

                "version":
                    2,

                "mrtName":
                    mrt_name,

                "homeName":
                    home_name,

                "routeId":
                    route_id,

                "routeOrder":
                    route_order,

                "routeKey":
                    route_key,

                "distanceConvention":
                    "MRT_0_TO_HOME",

                "routeDistance_m":
                    round(
                        v2_route_distance(
                            route
                        ),
                        2
                    ),

                "targetImageSpacing_m":
                    V2_IMAGE_SPACING_M,

                "imageCount":
                    len(
                        selected
                    ),

                "images":
                    selected,
            }
        )


        print(
            "Selected route images:",
            len(
                selected
            )
        )


    references.sort(

        key=lambda item: (

            item[
                "mrtName"
            ],

            item[
                "routeOrder"
            ],

            item[
                "imageOrder"
            ]

        )

    )


    safe_json_dump(

        V2_ROUTE_IMAGE_INDEX_PATH,

        {

            "version":
                2,

            "createdAt":
                datetime.now()
                .isoformat(
                    timespec="seconds"
                ),

            "distanceConvention":
                "MRT_0_TO_HOME",

            "targetImageSpacing_m":
                V2_IMAGE_SPACING_M,

            "imageCount":
                len(
                    references
                ),

            "routeCount":
                len({
                    item[
                        "routeId"
                    ]
                    for item
                    in references
                }),

            "images":
                references,
        }

    )


    return references


# ============================================================
# LIGHTWEIGHT FRONTEND IMAGE RECORD
# ============================================================

def v2_frontend_image(
    item
):

    return {

        "imageId":
            item.get(
                "imageId"
            ),

        "mrtName":
            item.get(
                "mrtName"
            ),

        "station":
            item.get(
                "mrtName"
            )
            or
            item.get(
                "station"
            ),

        "homeName":
            item.get(
                "homeName"
            ),

        "routeId":
            item.get(
                "routeId"
            ),

        "routeOrder":
            item.get(
                "routeOrder"
            ),

        "routeKey":
            item.get(
                "routeKey"
            ),

        "imageOrder":
            item.get(
                "imageOrder"
            ),

        "imageKey":
            item.get(
                "imageKey"
            ),

        "crawlKey":
            item.get(
                "crawlKey"
            ),

        "checkpointIndex":
            item.get(
                "checkpointIndex"
            ),

        "distanceFromMrt_m":
            item.get(
                "distanceFromMrt_m"
            ),

        "distanceAlongRoute_m":
            item.get(
                "distanceFromMrt_m"
            ),

        "originalImage":
            item.get(
                "originalImage"
            ),

        "segmentationImage":
            item.get(
                "segmentationImage"
            ),

        "visualCluster":
            item.get(
                "visualCluster"
            ),

        "cluster":
            item.get(
                "visualCluster"
            ),

        "somCell":
            item.get(
                "somCell"
            ),

        "mrtCluster":
            item.get(
                "mrtCluster"
            ),

        "groupShares":
            item.get(
                "groupShares"
            ),

        "visualMetrics":
            item.get(
                "visualMetrics"
            ),

        "distanceToCheckpoint_m":
            item.get(
                "distanceToCheckpoint_m"
            ),
    }


# ============================================================
# PATCH ROUTE PROFILES WITH V2 IDENTITY + COVERAGE
# ============================================================

def v2_patch_profiles(
    profiles,
    routes,
    references,
    analysed
):

    identity_by_route = {

        route.get(
            "routeId"
        ):
            v2_route_identity(
                route
            )

        for route
        in routes

        if route.get(
            "routeId"
        )

    }


    route_order_lookup = (
        v2_route_orders(
            routes
        )
    )


    selected_counts = Counter(

        reference[
            "routeId"
        ]

        for reference
        in references

    )


    analysed_checkpoint_counts = defaultdict(
        set
    )


    for item in analysed:

        analysed_checkpoint_counts[
            item.get(
                "routeId"
            )
        ].add(
            item.get(
                "checkpointIndex"
            )
        )


    route_lookup = {

        route.get(
            "routeId"
        ):
            route

        for route
        in routes

    }


    for profile in (
        profiles.get(
            "routes"
        )
        or
        []
    ):

        route_id = (
            profile.get(
                "routeId"
            )
        )


        identity = (
            identity_by_route.get(
                route_id
            )
            or
            {}
        )


        mrt_name = (
            identity.get(
                "mrtName"
            )
        )


        home_name = (
            identity.get(
                "homeName"
            )
        )


        route_order = (
            route_order_lookup.get(
                route_id
            )
        )


        profile[
            "station"
        ] = mrt_name


        profile[
            "mrtName"
        ] = mrt_name


        profile[
            "homeName"
        ] = home_name


        profile[
            "routeOrder"
        ] = route_order


        profile[
            "routeKey"
        ] = (

            f"r{route_order:03d}"
            f"__{slugify(home_name)}"

            if route_order
            else
            slugify(
                route_id
            )

        )


        profile[
            "distanceConvention"
        ] = "MRT_0_TO_HOME"


        selected_count = (
            selected_counts.get(
                route_id,
                0
            )
        )


        analysed_count = len(
            analysed_checkpoint_counts.get(
                route_id,
                set()
            )
        )


        route_record = (
            route_lookup.get(
                route_id
            )
            or
            {}
        )


        profile[
            "geometryCheckpointCount"
        ] = len(
            route_record.get(
                "checkpoints"
            )
            or
            []
        )


        profile[
            "expectedCheckpointCount"
        ] = selected_count


        profile[
            "analysedCheckpointCount"
        ] = analysed_count


        profile[
            "visualCoverage"
        ] = round(

            analysed_count
            /
            selected_count,

            5

        ) if selected_count else 0.0


        if (
            profile.get(
                "routeSeries"
            )
        ):

            route_series = (
                profile[
                    "routeSeries"
                ]
            )


            route_series[
                "distanceOrigin"
            ] = "MRT"


            route_series[
                "distanceDestination"
            ] = "HOME"


            route_series[
                "distanceConvention"
            ] = "MRT_0_TO_HOME"


            route_series[
                "imageSampleSpacing_m"
            ] = V2_IMAGE_SPACING_M


            route_series[
                "smoothingSigma_m"
            ] = ROUTE_SMOOTHING_SIGMA_M


            route_series[
                "maximumInterpolationGap_m"
            ] = ROUTE_SMOOTHING_MAX_GAP_M


    return profiles


# ============================================================
# SIMPLE NEW ML SUMMARY
# ============================================================

def v2_ml_evidence(
    analysed,
    visual_clusters,
    recognition_summary
):

    cluster_counts = Counter(

        int(
            item.get(
                "visualCluster"
            )
            or
            0
        )

        for item
        in analysed

    )


    som_cells = sorted({

        str(
            item.get(
                "somCell"
            )
        )

        for item
        in analysed

        if item.get(
            "somCell"
        )
        is not None

    })


    visual_ids = sorted(
        cluster_counts.keys()
    )


    matrix = []


    for som_cell in som_cells:

        row = []


        for visual_id in visual_ids:

            count = sum(

                1

                for item
                in analysed

                if (
                    str(
                        item.get(
                            "somCell"
                        )
                    )
                    ==
                    som_cell

                    and

                    int(
                        item.get(
                            "visualCluster"
                        )
                        or
                        0
                    )
                    ==
                    visual_id
                )

            )


            row.append(
                count
            )


        matrix.append(
            row
        )


    return {

        "imageCount":
            len(
                analysed
            ),

        "visualClusterCount":
            len(
                visual_clusters
            ),

        "clusterDistribution":
            [

                {
                    "cluster":
                        cluster_id,

                    "count":
                        cluster_counts[
                            cluster_id
                        ],
                }

                for cluster_id
                in visual_ids

            ],

        "clusterFeatureProfiles":
            visual_clusters,

        "somVisualMatrix": {

            "somCells":
                som_cells,

            "visualClusters":
                visual_ids,

            "counts":
                matrix,

        },

        "mrtClusterRecognisability":
            recognition_summary,
    }


# ============================================================
# COMPLETE V2 COMPILE
# ============================================================

def compile_route_environment_v2(
    cluster_count=10,
    force_crawl=True,
    force_images=True,
    mrt_filter=None
):

    started = time.time()


    print()
    print(
        "================================================"
    )

    print(
        "ROUTE-ORDERED IMAGE ANALYSIS V2"
    )

    print(
        "================================================"
    )

    print(
        "Distance convention: MRT = 0 m -> HOME"
    )

    print(
        "Image target spacing:",
        V2_IMAGE_SPACING_M,
        "m"
    )

    print(
        "Smoothing sigma:",
        ROUTE_SMOOTHING_SIGMA_M,
        "m"
    )

    print(
        "Maximum smoothing gap:",
        ROUTE_SMOOTHING_MAX_GAP_M,
        "m"
    )


    # --------------------------------------------------------
    # Reload current routes.
    # --------------------------------------------------------

    routes = (
        load_routes_for_analysis()
    )


    if mrt_filter:

        wanted = {

            v2_normalise_mrt(
                station
            )

            for station
            in mrt_filter

        }


        routes = [

            route

            for route
            in routes

            if (
                v2_route_identity(
                    route
                )[
                    "mrtName"
                ]
                in
                wanted
            )

        ]


    if not routes:

        raise RuntimeError(
            "No route records available for V2 crawl."
        )


    print(
        "Routes:",
        len(
            routes
        )
    )


    # --------------------------------------------------------
    # FRESH ROUTE-FIRST MAPILLARY CRAWL.
    # --------------------------------------------------------

    references = (
        v2_build_image_references(
            routes,
            force=force_crawl,
            mrt_filter=mrt_filter
        )
    )


    if not references:

        raise RuntimeError(
            "V2 route crawl returned zero image references."
        )


    print()
    print(
        "Route-specific image references:",
        len(
            references
        )
    )

    print(
        "Routes with imagery:",
        len({
            reference[
                "routeId"
            ]
            for reference
            in references
        })
    )


    # --------------------------------------------------------
    # DOWNLOAD USING NEW NAMING CONVENTION.
    # --------------------------------------------------------

    print()
    print(
        "Downloading route-ordered images..."
    )


    downloaded, download_errors = (
        download_images(
            references,
            force=force_images
        )
    )


    print(
        "Downloaded / cached:",
        len(
            downloaded
        )
    )

    print(
        "Download errors:",
        len(
            download_errors
        )
    )


    # --------------------------------------------------------
    # QUALITY CONTROL.
    # --------------------------------------------------------

    accepted, rejected = (
        quality_filter_images(
            downloaded
        )
    )


    print(
        "Quality accepted:",
        len(
            accepted
        )
    )

    print(
        "Quality rejected:",
        len(
            rejected
        )
    )


    if not accepted:

        raise RuntimeError(
            "No V2 images survived quality control."
        )


    # --------------------------------------------------------
    # SEGFORMER.
    # --------------------------------------------------------

    analysed = []

    segmentation_errors = []


    print()
    print(
        "Running route-specific SegFormer..."
    )


    for index, item in enumerate(
        accepted,
        start=1
    ):

        print(

            f"[{index}/{len(accepted)}] "

            f"{item.get('mrtName')} | "

            f"R{int(item.get('routeOrder') or 0):03d} | "

            f"IMG{int(item.get('imageOrder') or 0):04d} | "

            f"{float(item.get('distanceFromMrt_m') or 0):.0f} m"

        )


        try:

            segmented = (
                segment_image(
                    item,
                    force=force_images
                )
            )


            analysed.append(
                enrich_visual_metrics(
                    segmented
                )
            )


        except Exception as error:

            segmentation_errors.append({

                "routeId":
                    item.get(
                        "routeId"
                    ),

                "imageId":
                    item.get(
                        "imageId"
                    ),

                "crawlKey":
                    item.get(
                        "crawlKey"
                    ),

                "error":
                    str(
                        error
                    ),

            })


            print(
                "  FAILED:",
                error
            )


    if not analysed:

        raise RuntimeError(
            "No V2 image completed SegFormer."
        )


    # --------------------------------------------------------
    # K-MEANS.
    # --------------------------------------------------------

    print()
    print(
        "Running K-means..."
    )


    (
        analysed,
        visual_clusters,
        clustering_metadata
    ) = cluster_visual_images(
        analysed,
        requested_clusters=cluster_count
    )


    # --------------------------------------------------------
    # MRT RECOGNISABILITY.
    # --------------------------------------------------------

    (
        analysed,
        recognition_summary
    ) = attach_mrt_visual_recognition(
        analysed
    )


    # --------------------------------------------------------
    # ROUTE PROFILES.
    #
    # Crucially:
    #
    # distanceAlongRoute_m in this new dataset already means
    # DISTANCE FROM MRT.
    # --------------------------------------------------------

    profiles = build_profiles(
        analysed,
        routes
    )


    profiles = v2_patch_profiles(
        profiles,
        routes,
        references,
        analysed
    )


    # --------------------------------------------------------
    # 100 REPRESENTATIVE CELLS.
    # --------------------------------------------------------

    compiler_cells = (
        balanced_compiler_cells(
            analysed,
            COMPILER_GRID_SIZE
        )
    )


    # --------------------------------------------------------
    # ADD V2 ROUTE KEYS TO COMPILER CELLS.
    # --------------------------------------------------------

    analysed_by_image_id = {

        str(
            item.get(
                "imageId"
            )
        ):
            item

        for item
        in analysed

    }


    for cell in compiler_cells:

        source = analysed_by_image_id.get(
            str(
                cell.get(
                    "imageId"
                )
            )
        )


        if source:

            cell.update({

                "mrtName":
                    source.get(
                        "mrtName"
                    ),

                "homeName":
                    source.get(
                        "homeName"
                    ),

                "routeOrder":
                    source.get(
                        "routeOrder"
                    ),

                "routeKey":
                    source.get(
                        "routeKey"
                    ),

                "imageOrder":
                    source.get(
                        "imageOrder"
                    ),

                "imageKey":
                    source.get(
                        "imageKey"
                    ),

                "crawlKey":
                    source.get(
                        "crawlKey"
                    ),

                "distanceFromMrt_m":
                    source.get(
                        "distanceFromMrt_m"
                    ),

            })


    ml_evidence = v2_ml_evidence(
        analysed,
        visual_clusters,
        recognition_summary
    )


    elapsed = (
        time.time()
        -
        started
    )


    # ========================================================
    # FULL V2 OUTPUT
    # ========================================================

    full_payload = {

        "version":
            6,

        "dataset":
            "route_v2",

        "createdAt":
            datetime.now()
            .isoformat(
                timespec="seconds"
            ),

        "distanceConvention":
            "MRT_0_TO_HOME",

        "targetImageSpacing_m":
            V2_IMAGE_SPACING_M,

        "smoothingSigma_m":
            ROUTE_SMOOTHING_SIGMA_M,

        "maximumInterpolationGap_m":
            ROUTE_SMOOTHING_MAX_GAP_M,

        "routeCount":
            len(
                routes
            ),

        "imageReferenceCount":
            len(
                references
            ),

        "downloadedCount":
            len(
                downloaded
            ),

        "qualityAcceptedCount":
            len(
                accepted
            ),

        "qualityRejectedCount":
            len(
                rejected
            ),

        "imageCount":
            len(
                analysed
            ),

        "clusterCount":
            len(
                visual_clusters
            ),

        "analysisSeconds":
            round(
                elapsed,
                2
            ),

        "device":
            DEVICE,

        "model":
            SEGFORMER_MODEL_NAME,

        "clustering":
            clustering_metadata,

        "clusters":
            visual_clusters,

        "cells":
            compiler_cells,

        "images":
            analysed,

        "downloadErrors":
            download_errors,

        "segmentationErrors":
            segmentation_errors,

        "mlEvidence":
            ml_evidence,

        **profiles,

    }


    safe_json_dump(
        V2_FULL_ANALYSIS_PATH,
        full_payload
    )


    # ========================================================
    # LIGHTWEIGHT FRONTEND COMPILER
    # ========================================================

    frontend_images = [

        v2_frontend_image(
            item
        )

        for item
        in analysed

    ]


    compiler_payload = {

        "version":
            6,

        "dataset":
            "route_v2",

        "createdAt":
            full_payload[
                "createdAt"
            ],

        "distanceConvention":
            "MRT_0_TO_HOME",

        "targetImageSpacing_m":
            V2_IMAGE_SPACING_M,

        "imageCount":
            len(
                analysed
            ),

        "clusterCount":
            len(
                visual_clusters
            ),

        "cells":
            compiler_cells,

        "allImages":
            frontend_images,

        "clusters":
            visual_clusters,

        "routes":
            profiles[
                "routes"
            ],

        "stations":
            profiles[
                "stations"
            ],

        "mrtClusters":
            profiles[
                "mrtClusters"
            ],

        "checkpoints":
            profiles[
                "checkpoints"
            ],

        "mlEvidence":
            ml_evidence,

        "mrtClusterRecognisability":
            recognition_summary,

        "fullAnalysisFile":
            "./data/generated/image_analysis_route_v2.json",

        "routeImageIndex":
            "./data/generated/route_image_index_v2.json",

    }


    safe_json_dump(
        V2_COMPILER_PATH,
        compiler_payload
    )


    print()
    print(
        "================================================"
    )

    print(
        "ROUTE V2 ANALYSIS COMPLETE"
    )

    print(
        "================================================"
    )

    print(
        "Routes:",
        len(
            routes
        )
    )

    print(
        "Route-specific references:",
        len(
            references
        )
    )

    print(
        "Images analysed:",
        len(
            analysed
        )
    )

    print(
        "Visual clusters:",
        len(
            visual_clusters
        )
    )

    print(
        "Time:",
        round(
            elapsed / 60,
            2
        ),
        "minutes"
    )

    print()
    print(
        "NEW COMPILER:"
    )

    print(
        V2_COMPILER_PATH
    )

    print()
    print(
        "NEW FULL ANALYSIS:"
    )

    print(
        V2_FULL_ANALYSIS_PATH
    )

    print()
    print(
        "NEW ROUTE IMAGE INDEX:"
    )

    print(
        V2_ROUTE_IMAGE_INDEX_PATH
    )

    print(
        "================================================"
    )


    return compiler_payload


# ============================================================
# RUN
# ============================================================
#
# This performs the new full crawl.
#
# To TEST just one MRT first:
#
# RESULT_V2 = compile_route_environment_v2(
#     cluster_count=10,
#     force_crawl=True,
#     force_images=True,
#     mrt_filter=["JURONG EAST MRT"]
# )
#
# For the COMPLETE dataset, use the version below.
# ============================================================

RESULT_V2 = compile_route_environment_v2(
    cluster_count=10,
    force_crawl=True,
    force_images=True,
    mrt_filter=None
)


ROUTE-ORDERED IMAGE ANALYSIS V2
Distance convention: MRT = 0 m -> HOME
Image target spacing: 60.0 m
Smoothing sigma: 90.0 m
Maximum smoothing gap: 180.0 m

Reloading completed route intelligence...
Route folder: C:\Users\MSI\OneDrive\Desktop\family-toilet-search\data\generated\route_intelligence
latest_batch.json: 192 routes
Individual route records: 192

ROUTES LOADED: 192
• ADMIRALTY__R01 | ADMIRALTY MRT | checkpoints: 79
• ALJUNIED__R01 | ALJUNIED MRT | checkpoints: 3
• ANG_MO_KIO__R01 | ANG MO KIO MRT | checkpoints: 93
• BEAUTY_WORLD__R01 | BEAUTY WORLD MRT | checkpoints: 13
• BEDOK_NORTH__R01 | BEDOK NORTH MRT | checkpoints: 46
• BEDOK_RESERVOIR__R01 | BEDOK RESERVOIR MRT | checkpoints: 22
• BEDOK__R01 | BEDOK MRT | checkpoints: 95
• BISHAN__R01 | BISHAN MRT | checkpoints: 45
• BOON_KENG__R01 | BOON KENG MRT | checkpoints: 84
• BOON_LAY__R01 | BOON LAY MRT | checkpoints: 5
Routes: 192

[1/192]
ADMIRALTY MRT → Residential Cluster 113
Route: residential-0113__admiralty-mrt
Distance

KeyboardInterrupt: 

In [19]:
# ============================================================
# V2 PATCH — LOAD FAMILY-ORIENTED VISUAL METRICS
# ============================================================

def share(dictionary, key):
    return safe_float(
        (dictionary or {}).get(key),
        0.0
    )


def clamp01(value):
    return max(
        0.0,
        min(
            1.0,
            safe_float(value)
        )
    )


def visual_metrics(item):

    shares = (
        item.get("groupShares")
        or {}
    )

    spatial = (
        item.get("spatialShares")
        or {}
    )


    bottom = (
        spatial.get("bottomBand")
        or {}
    )

    middle = (
        spatial.get("middleBand")
        or {}
    )

    left = (
        spatial.get("leftHalf")
        or {}
    )

    right = (
        spatial.get("rightHalf")
        or {}
    )


    pedestrian = share(
        shares,
        "pedestrian_surface"
    )

    bottom_pedestrian = share(
        bottom,
        "pedestrian_surface"
    )

    road = share(
        shares,
        "road"
    )

    bottom_road = share(
        bottom,
        "road"
    )

    vegetation = share(
        shares,
        "vegetation"
    )

    sky = share(
        shares,
        "sky"
    )

    building = share(
        shares,
        "building"
    )

    vehicle = share(
        shares,
        "vehicle"
    )

    bottom_vehicle = share(
        bottom,
        "vehicle"
    )

    person = share(
        shares,
        "person"
    )

    barrier = share(
        shares,
        "barrier"
    )

    stairs = share(
        shares,
        "stairs"
    )

    furniture = share(
        shares,
        "street_furniture"
    )

    shade_structure = share(
        shares,
        "shade_structure"
    )


    left_green = share(
        left,
        "vegetation"
    )

    right_green = share(
        right,
        "vegetation"
    )

    middle_green = share(
        middle,
        "vegetation"
    )


    greenery = clamp01(
        vegetation * 2.8
    )


    green_balance = clamp01(
        1
        -
        abs(
            left_green
            -
            right_green
        )
        *
        3
    )


    pedestrian_space = clamp01(
        pedestrian * 1.8
        +
        bottom_pedestrian * 1.5
    )


    roadway_exposure = clamp01(
        road * 1.1
        +
        bottom_road * 1.2
        +
        vehicle * 2.6
        +
        bottom_vehicle * 2.0
    )


    obstruction = clamp01(
        barrier * 2.0
        +
        bottom_vehicle * 1.5
        +
        stairs * 4.0
    )


    openness = clamp01(
        sky * 1.45
        +
        vegetation * 0.4
        -
        building * 0.45
    )


    enclosure = clamp01(
        building * 1.55
        +
        barrier * 0.7
        -
        sky * 0.45
    )


    shade_proxy = clamp01(
        middle_green * 1.7
        +
        vegetation * 0.9
        +
        shade_structure * 3.5
        +
        building * 0.18
        -
        sky * 0.20
    )


    pedestrian_comfort = clamp01(
        pedestrian_space * 0.34
        +
        greenery * 0.23
        +
        shade_proxy * 0.20
        +
        green_balance * 0.05
        +
        openness * 0.06
        -
        roadway_exposure * 0.09
        -
        obstruction * 0.03
    )


    stroller_friendliness = clamp01(
        pedestrian_space * 0.47
        +
        shade_proxy * 0.17
        +
        openness * 0.11
        +
        greenery * 0.12
        -
        roadway_exposure * 0.08
        -
        stairs * 3.5 * 0.05
    )


    return {

        "greeneryScore":
            round(
                greenery,
                5
            ),

        "greenBalanceScore":
            round(
                green_balance,
                5
            ),

        "pedestrianSpaceScore":
            round(
                pedestrian_space,
                5
            ),

        "roadwayExposureScore":
            round(
                roadway_exposure,
                5
            ),

        "obstructionProxy":
            round(
                obstruction,
                5
            ),

        "opennessProxy":
            round(
                openness,
                5
            ),

        "enclosureProxy":
            round(
                enclosure,
                5
            ),

        "shadeProxy":
            round(
                shade_proxy,
                5
            ),

        "pedestrianComfortScore":
            round(
                pedestrian_comfort,
                5
            ),

        "strollerFriendlinessScore":
            round(
                stroller_friendliness,
                5
            ),

        "personShare":
            round(
                person,
                7
            ),

        "streetFurnitureShare":
            round(
                furniture,
                7
            ),

        "bottomPedestrianShare":
            round(
                bottom_pedestrian,
                7
            ),

        "bottomRoadShare":
            round(
                bottom_road,
                7
            ),

        "bottomVehicleShare":
            round(
                bottom_vehicle,
                7
            ),
    }


def enrich_visual_metrics(item):

    return {
        **item,

        "visualMetrics":
            visual_metrics(
                item
            ),
    }


print(
    "✓ V2 visual metrics functions loaded."
)

print(
    "visual_metrics:",
    "visual_metrics" in globals()
)

print(
    "enrich_visual_metrics:",
    "enrich_visual_metrics" in globals()
)

✓ V2 visual metrics functions loaded.
visual_metrics: True
enrich_visual_metrics: True


In [20]:
RESULT_V2 = compile_route_environment_v2(
    cluster_count=10,

    # Reuse the Mapillary route crawl
    force_crawl=False,

    # Reuse downloaded images and any cached segmentation
    force_images=False,

    mrt_filter=None
)


ROUTE-ORDERED IMAGE ANALYSIS V2
Distance convention: MRT = 0 m -> HOME
Image target spacing: 60.0 m
Smoothing sigma: 90.0 m
Maximum smoothing gap: 180.0 m

Reloading completed route intelligence...
Route folder: C:\Users\MSI\OneDrive\Desktop\family-toilet-search\data\generated\route_intelligence
latest_batch.json: 192 routes
Individual route records: 192

ROUTES LOADED: 192
• ADMIRALTY__R01 | ADMIRALTY MRT | checkpoints: 79
• ALJUNIED__R01 | ALJUNIED MRT | checkpoints: 3
• ANG_MO_KIO__R01 | ANG MO KIO MRT | checkpoints: 93
• BEAUTY_WORLD__R01 | BEAUTY WORLD MRT | checkpoints: 13
• BEDOK_NORTH__R01 | BEDOK NORTH MRT | checkpoints: 46
• BEDOK_RESERVOIR__R01 | BEDOK RESERVOIR MRT | checkpoints: 22
• BEDOK__R01 | BEDOK MRT | checkpoints: 95
• BISHAN__R01 | BISHAN MRT | checkpoints: 45
• BOON_KENG__R01 | BOON KENG MRT | checkpoints: 84
• BOON_LAY__R01 | BOON LAY MRT | checkpoints: 5
Routes: 192

[1/192]
ADMIRALTY MRT → Residential Cluster 113
Route: residential-0113__admiralty-mrt
Distance

ValueError: not enough values to unpack (expected 3, got 2)

In [21]:
# ============================================================
# V2 / V3 PATCH — CLUSTER FUNCTION COMPATIBILITY
#
# Handles BOTH versions of cluster_visual_images():
#
# old:
#   return analysed, clusters
#
# new:
#   return analysed, clusters, metadata
#
# No recrawling.
# No redownloading.
# No SegFormer rerun if cache exists.
# ============================================================

if "_cluster_visual_images_original" not in globals():

    _cluster_visual_images_original = (
        cluster_visual_images
    )


def cluster_visual_images(
    images,
    requested_clusters=None
):

    result = (
        _cluster_visual_images_original(
            images,
            requested_clusters=requested_clusters
        )
    )


    # --------------------------------------------------------
    # New/current form:
    # analysed, clusters, metadata
    # --------------------------------------------------------

    if (
        isinstance(result, tuple)
        and
        len(result) == 3
    ):

        return result


    # --------------------------------------------------------
    # Older form:
    # analysed, clusters
    #
    # Build the missing metadata here.
    # --------------------------------------------------------

    if (
        isinstance(result, tuple)
        and
        len(result) == 2
    ):

        analysed, clusters = (
            result
        )


        feature_keys = []


        if analysed:

            try:

                feature_keys, _ = (
                    matrix_from_images(
                        analysed
                    )
                )

            except Exception:

                # Metadata is useful but not required
                # for the visual results themselves.
                feature_keys = []


        metadata = {

            "featureKeys":
                list(
                    feature_keys
                ),

            "clusterCount":
                len(
                    clusters
                    or
                    []
                ),

            "compatibilityPatch":
                True,

            "note":
                (
                    "The active Jupyter clustering function "
                    "returned two values. Metadata was rebuilt "
                    "by the V2/V3 compatibility patch."
                ),
        }


        return (
            analysed,
            clusters,
            metadata
        )


    raise RuntimeError(
        "cluster_visual_images() returned an unexpected "
        f"result type: {type(result)}"
    )


print(
    "✓ cluster_visual_images compatibility patch loaded"
)


# Quick verification — do NOT run clustering here.
print(
    "V2/V3 now expects the clustering call to return "
    "analysed + clusters + metadata."
)

✓ cluster_visual_images compatibility patch loaded
V2/V3 now expects the clustering call to return analysed + clusters + metadata.


In [22]:
RESULT_V2 = compile_route_environment_v2(
    cluster_count=10,

    # Reuse the route crawl
    force_crawl=False,

    # Reuse downloads + cached segmentations
    force_images=False,

    mrt_filter=None
)


ROUTE-ORDERED IMAGE ANALYSIS V2
Distance convention: MRT = 0 m -> HOME
Image target spacing: 60.0 m
Smoothing sigma: 90.0 m
Maximum smoothing gap: 180.0 m

Reloading completed route intelligence...
Route folder: C:\Users\MSI\OneDrive\Desktop\family-toilet-search\data\generated\route_intelligence
latest_batch.json: 192 routes
Individual route records: 192

ROUTES LOADED: 192
• ADMIRALTY__R01 | ADMIRALTY MRT | checkpoints: 79
• ALJUNIED__R01 | ALJUNIED MRT | checkpoints: 3
• ANG_MO_KIO__R01 | ANG MO KIO MRT | checkpoints: 93
• BEAUTY_WORLD__R01 | BEAUTY WORLD MRT | checkpoints: 13
• BEDOK_NORTH__R01 | BEDOK NORTH MRT | checkpoints: 46
• BEDOK_RESERVOIR__R01 | BEDOK RESERVOIR MRT | checkpoints: 22
• BEDOK__R01 | BEDOK MRT | checkpoints: 95
• BISHAN__R01 | BISHAN MRT | checkpoints: 45
• BOON_KENG__R01 | BOON KENG MRT | checkpoints: 84
• BOON_LAY__R01 | BOON LAY MRT | checkpoints: 5
Routes: 192

[1/192]
ADMIRALTY MRT → Residential Cluster 113
Route: residential-0113__admiralty-mrt
Distance

NameError: name 'attach_mrt_visual_recognition' is not defined

In [23]:
# ============================================================
# V2 COMPLETION PATCH
# Missing optional MRT-recognition function
#
# This lets V2 finish and save its route-specific dataset.
#
# It DOES NOT:
# - recrawl Mapillary
# - rerun the model from scratch
# - change K-means
# - change route profiles
#
# The experimental MRT-recognition metric is simply marked
# as unavailable instead of blocking the whole pipeline.
# ============================================================


def attach_mrt_visual_recognition(images):

    output = []


    for item in images:

        copied = dict(item)

        # Preserve compatibility with frontend / JSON schema.
        copied.setdefault(
            "predictedSomCell",
            None
        )

        copied.setdefault(
            "mrtClusterMatch",
            None
        )

        output.append(
            copied
        )


    summary = {

        "evaluated":
            False,

        "accuracy":
            None,

        "tested":
            0,

        "testedCount":
            0,

        "correct":
            0,

        "method":
            "not_evaluated",

        "reason":
            (
                "MRT visual recognisability was skipped in V2 "
                "because the experimental recognition function "
                "was unavailable in the current Jupyter kernel. "
                "This does not affect SegFormer, K-means or "
                "route-specific environmental profiles."
            )

    }


    return (
        output,
        summary
    )


print(
    "✓ attach_mrt_visual_recognition compatibility patch loaded"
)

✓ attach_mrt_visual_recognition compatibility patch loaded


In [24]:
# ============================================================
# CHECK ALL REMAINING V2 DEPENDENCIES
# ============================================================

required_v2_functions = [

    "load_routes_for_analysis",

    "download_images",

    "quality_filter_images",

    "segment_image",

    "enrich_visual_metrics",

    "cluster_visual_images",

    "attach_mrt_visual_recognition",

    "build_profiles",

    "v2_patch_profiles",

    "balanced_compiler_cells",

    "v2_ml_evidence",

]


missing = [

    name

    for name
    in required_v2_functions

    if name
    not in globals()

]


if missing:

    print(
        "❌ Still missing:"
    )

    for name in missing:
        print(
            "  -",
            name
        )

else:

    print(
        "✓ ALL V2 DEPENDENCIES ARE LOADED"
    )

    print(
        "Safe to rerun cached V2 compile."
    )

❌ Still missing:
  - build_profiles
  - balanced_compiler_cells


In [25]:
# ============================================================
# V2 FINAL DEPENDENCY PATCH
#
# Adds:
#   build_profiles
#   balanced_compiler_cells
#
# Uses the already-analysed route-specific images.
# No crawling.
# No redownload.
# No fresh SegFormer run required.
# ============================================================

from collections import defaultdict
import numpy as np
import math


# ============================================================
# 1. BUILD ROUTE PROFILES
# ============================================================

def build_profiles(
    analysed_images,
    route_records
):

    # --------------------------------------------------------
    # Route lookup
    # --------------------------------------------------------

    route_lookup = {

        str(
            route.get(
                "routeId"
            )
        ):
            route

        for route
        in route_records

        if route.get(
            "routeId"
        )

    }


    # --------------------------------------------------------
    # Group analysed images by exact route
    # --------------------------------------------------------

    images_by_route = defaultdict(
        list
    )


    for item in analysed_images:

        route_id = str(
            item.get(
                "routeId"
            )
            or
            ""
        )


        if not route_id:
            continue


        images_by_route[
            route_id
        ].append(
            item
        )


    for route_id in images_by_route:

        images_by_route[
            route_id
        ].sort(

            key=lambda item:

                float(
                    item.get(
                        "distanceFromMrt_m",
                        item.get(
                            "distanceAlongRoute_m",
                            0
                        )
                    )
                    or
                    0
                )

        )


    route_profiles = []

    station_groups = defaultdict(
        list
    )

    mrt_cluster_groups = defaultdict(
        list
    )

    checkpoint_records = []


    # --------------------------------------------------------
    # Build exact route profile
    # --------------------------------------------------------

    for route_id, route in route_lookup.items():

        route_images = (
            images_by_route.get(
                route_id
            )
            or
            []
        )


        identity = (
            v2_route_identity(
                route
            )
            if "v2_route_identity" in globals()
            else
            {}
        )


        mrt_name = (
            identity.get(
                "mrtName"
            )
            or
            route.get(
                "station"
            )
        )


        home_name = (
            identity.get(
                "homeName"
            )
            or
            route.get(
                "destination"
            )
            or
            "Residential destination"
        )


        # ----------------------------------------------------
        # Route distance
        # ----------------------------------------------------

        route_distance = 0.0


        if "v2_route_distance" in globals():

            route_distance = float(
                v2_route_distance(
                    route
                )
                or
                0
            )


        if route_distance <= 0:

            summary = (
                route.get(
                    "routeSummary"
                )
                or
                {}
            )


            route_distance = float(
                summary.get(
                    "distance_m"
                )
                or
                0
            )


        if (
            route_distance <= 0
            and
            route_images
        ):

            route_distance = max(

                float(
                    item.get(
                        "distanceFromMrt_m",
                        item.get(
                            "distanceAlongRoute_m",
                            0
                        )
                    )
                    or
                    0
                )

                for item
                in route_images

            )


        # ----------------------------------------------------
        # Raw environmental route points
        # ----------------------------------------------------

        raw_points = []


        for item in route_images:

            shares = (
                item.get(
                    "groupShares"
                )
                or
                {}
            )


            distance = float(
                item.get(
                    "distanceFromMrt_m",
                    item.get(
                        "distanceAlongRoute_m",
                        0
                    )
                )
                or
                0
            )


            values = {

                "sky":
                    float(
                        shares.get(
                            "sky"
                        )
                        or
                        0
                    )
                    *
                    100,

                "greenery":
                    float(
                        shares.get(
                            "vegetation"
                        )
                        or
                        0
                    )
                    *
                    100,

                "road":
                    float(
                        shares.get(
                            "road"
                        )
                        or
                        0
                    )
                    *
                    100,

                "pedestrian":
                    float(
                        shares.get(
                            "pedestrian_surface"
                        )
                        or
                        0
                    )
                    *
                    100,

                "building":
                    float(
                        shares.get(
                            "building"
                        )
                        or
                        0
                    )
                    *
                    100,

                "vehicle":
                    float(
                        shares.get(
                            "vehicle"
                        )
                        or
                        0
                    )
                    *
                    100,

            }


            point = {

                "distance_m":
                    round(
                        distance,
                        2
                    ),

                "values":
                    {
                        key:
                            round(
                                value,
                                4
                            )
                        for key, value
                        in values.items()
                    },

                "imageId":
                    item.get(
                        "imageId"
                    ),

                "imageOrder":
                    item.get(
                        "imageOrder"
                    ),

                "checkpointIndex":
                    item.get(
                        "checkpointIndex"
                    ),

            }


            raw_points.append(
                point
            )


            checkpoint_records.append({

                "routeId":
                    route_id,

                "station":
                    mrt_name,

                "homeName":
                    home_name,

                "distance_m":
                    point[
                        "distance_m"
                    ],

                "imageId":
                    point[
                        "imageId"
                    ],

                "imageOrder":
                    point[
                        "imageOrder"
                    ],

                "checkpointIndex":
                    point[
                        "checkpointIndex"
                    ],

                "values":
                    point[
                        "values"
                    ],

            })


        raw_points.sort(
            key=lambda point:
                point[
                    "distance_m"
                ]
        )


        # ----------------------------------------------------
        # Basic smoothed profile
        #
        # This is only for V2 compatibility.
        # V3 will rebuild the final 15 m profiles properly.
        # ----------------------------------------------------

        smoothed_points = []


        if raw_points:

            distances = np.array(
                [
                    point[
                        "distance_m"
                    ]
                    for point
                    in raw_points
                ],
                dtype=float
            )


            feature_keys = [
                "sky",
                "greenery",
                "road",
                "pedestrian",
                "building",
                "vehicle",
            ]


            sigma = 70.0

            step = 15.0


            if route_distance <= 0:

                route_distance = float(
                    distances.max()
                )


            positions = np.arange(
                0.0,
                route_distance + step,
                step
            )


            if (
                len(
                    positions
                )
                and
                positions[-1]
                <
                route_distance
            ):

                positions = np.append(
                    positions,
                    route_distance
                )


            for position in positions:

                delta = (
                    distances
                    -
                    position
                )


                weights = np.exp(
                    -0.5
                    *
                    (
                        delta
                        /
                        sigma
                    )
                    ** 2
                )


                # Don't let observations influence absurdly far
                weights[
                    np.abs(
                        delta
                    )
                    >
                    180
                ] = 0.0


                values = {}


                for feature_key in feature_keys:

                    source_values = np.array(
                        [
                            point[
                                "values"
                            ][
                                feature_key
                            ]
                            for point
                            in raw_points
                        ],
                        dtype=float
                    )


                    valid = (
                        weights
                        >
                        0
                    )


                    if not valid.any():

                        values[
                            feature_key
                        ] = None

                    else:

                        values[
                            feature_key
                        ] = round(

                            float(
                                np.average(
                                    source_values[
                                        valid
                                    ],
                                    weights=
                                        weights[
                                            valid
                                        ]
                                )
                            ),

                            4

                        )


                smoothed_points.append({

                    "distance_m":
                        round(
                            float(
                                position
                            ),
                            2
                        ),

                    "values":
                        values

                })


        context = (

            context_for_station(
                mrt_name
            )

            if (
                mrt_name
                and
                "context_for_station" in globals()
            )

            else
            {}

        )


        som_cell = (
            context.get(
                "somCell"
            )
        )


        mrt_cluster = (
            f"SOM-{som_cell}"
            if som_cell is not None
            else
            "UNKNOWN"
        )


        profile = {

            "routeId":
                route_id,

            "station":
                mrt_name,

            "mrtName":
                mrt_name,

            "homeName":
                home_name,

            "destination":
                home_name,

            "somCell":
                som_cell,

            "mrtCluster":
                mrt_cluster,

            "analysedCheckpointCount":
                len(
                    raw_points
                ),

            "expectedCheckpointCount":
                len(
                    route_images
                ),

            "visualCoverage":
                1.0
                if route_images
                else
                0.0,

            "routeSeries": {

                "distanceOrigin":
                    "MRT",

                "distanceDestination":
                    "HOME",

                "distanceConvention":
                    "MRT_0_TO_HOME",

                "routeDistance_m":
                    round(
                        route_distance,
                        2
                    ),

                "step_m":
                    15.0,

                "smoothingSigma_m":
                    70.0,

                "raw":
                    raw_points,

                "smoothed":
                    smoothed_points,

            },

        }


        route_profiles.append(
            profile
        )


        if mrt_name:

            station_groups[
                mrt_name
            ].append(
                route_id
            )


        mrt_cluster_groups[
            mrt_cluster
        ].append(
            route_id
        )


    # --------------------------------------------------------
    # Station summaries
    # --------------------------------------------------------

    stations = []


    for station, route_ids in sorted(
        station_groups.items()
    ):

        stations.append({

            "station":
                station,

            "routeCount":
                len(
                    route_ids
                ),

            "routeIds":
                route_ids,

        })


    # --------------------------------------------------------
    # MRT cluster summaries
    # --------------------------------------------------------

    mrt_clusters = []


    for cluster_name, route_ids in sorted(
        mrt_cluster_groups.items()
    ):

        mrt_clusters.append({

            "mrtCluster":
                cluster_name,

            "routeCount":
                len(
                    route_ids
                ),

            "routeIds":
                route_ids,

        })


    return {

        "routes":
            route_profiles,

        "stations":
            stations,

        "mrtClusters":
            mrt_clusters,

        "checkpoints":
            checkpoint_records,

    }


# ============================================================
# 2. BALANCED COMPILER CELLS
#
# Pick representative images across routes rather than simply
# taking the first 100.
# ============================================================

def balanced_compiler_cells(
    analysed_images,
    max_cells=100
):

    if not analysed_images:
        return []


    grouped = defaultdict(
        list
    )


    for item in analysed_images:

        route_id = str(
            item.get(
                "routeId"
            )
            or
            "unknown-route"
        )


        grouped[
            route_id
        ].append(
            item
        )


    for route_id in grouped:

        grouped[
            route_id
        ].sort(

            key=lambda item:

                float(
                    item.get(
                        "distanceFromMrt_m",
                        item.get(
                            "distanceAlongRoute_m",
                            0
                        )
                    )
                    or
                    0
                )

        )


    route_ids = sorted(
        grouped.keys()
    )


    selected = []


    # --------------------------------------------------------
    # Round robin across routes
    # --------------------------------------------------------

    offset = 0


    while (
        len(
            selected
        )
        <
        max_cells
    ):

        added = False


        for route_id in route_ids:

            items = grouped[
                route_id
            ]


            if offset < len(
                items
            ):

                selected.append(
                    items[
                        offset
                    ]
                )

                added = True


                if (
                    len(
                        selected
                    )
                    >=
                    max_cells
                ):
                    break


        if not added:
            break


        offset += 1


    cells = []


    for slot, item in enumerate(
        selected
    ):

        cells.append({

            "slot":
                slot,

            "imageId":
                item.get(
                    "imageId"
                ),

            "originalImage":
                item.get(
                    "originalImage"
                ),

            "segmentationImage":
                item.get(
                    "segmentationImage"
                ),

            "station":
                item.get(
                    "mrtName"
                )
                or
                item.get(
                    "station"
                ),

            "mrtName":
                item.get(
                    "mrtName"
                )
                or
                item.get(
                    "station"
                ),

            "homeName":
                item.get(
                    "homeName"
                ),

            "routeId":
                item.get(
                    "routeId"
                ),

            "routeOrder":
                item.get(
                    "routeOrder"
                ),

            "routeKey":
                item.get(
                    "routeKey"
                ),

            "imageOrder":
                item.get(
                    "imageOrder"
                ),

            "imageKey":
                item.get(
                    "imageKey"
                ),

            "crawlKey":
                item.get(
                    "crawlKey"
                ),

            "checkpointIndex":
                item.get(
                    "checkpointIndex"
                ),

            "distanceFromMrt_m":
                item.get(
                    "distanceFromMrt_m"
                ),

            "cluster":
                int(
                    item.get(
                        "visualCluster",
                        item.get(
                            "cluster",
                            0
                        )
                    )
                    or
                    0
                ),

            "visualCluster":
                int(
                    item.get(
                        "visualCluster",
                        item.get(
                            "cluster",
                            0
                        )
                    )
                    or
                    0
                ),

            "somCell":
                item.get(
                    "somCell"
                ),

            "mrtCluster":
                item.get(
                    "mrtCluster"
                ),

            "groupShares":
                item.get(
                    "groupShares"
                ),

            "visualMetrics":
                item.get(
                    "visualMetrics"
                ),

        })


    return cells


print(
    "✓ build_profiles loaded"
)

print(
    "✓ balanced_compiler_cells loaded"
)

✓ build_profiles loaded
✓ balanced_compiler_cells loaded


In [26]:
RESULT_V2 = compile_route_environment_v2(
    cluster_count=10,
    force_crawl=False,
    force_images=False,
    mrt_filter=None
)


ROUTE-ORDERED IMAGE ANALYSIS V2
Distance convention: MRT = 0 m -> HOME
Image target spacing: 60.0 m
Smoothing sigma: 90.0 m
Maximum smoothing gap: 180.0 m

Reloading completed route intelligence...
Route folder: C:\Users\MSI\OneDrive\Desktop\family-toilet-search\data\generated\route_intelligence
latest_batch.json: 192 routes
Individual route records: 192

ROUTES LOADED: 192
• ADMIRALTY__R01 | ADMIRALTY MRT | checkpoints: 79
• ALJUNIED__R01 | ALJUNIED MRT | checkpoints: 3
• ANG_MO_KIO__R01 | ANG MO KIO MRT | checkpoints: 93
• BEAUTY_WORLD__R01 | BEAUTY WORLD MRT | checkpoints: 13
• BEDOK_NORTH__R01 | BEDOK NORTH MRT | checkpoints: 46
• BEDOK_RESERVOIR__R01 | BEDOK RESERVOIR MRT | checkpoints: 22
• BEDOK__R01 | BEDOK MRT | checkpoints: 95
• BISHAN__R01 | BISHAN MRT | checkpoints: 45
• BOON_KENG__R01 | BOON KENG MRT | checkpoints: 84
• BOON_LAY__R01 | BOON LAY MRT | checkpoints: 5
Routes: 192

[1/192]
ADMIRALTY MRT → Residential Cluster 113
Route: residential-0113__admiralty-mrt
Distance

In [29]:
# ============================================================
# ROUTE ENVIRONMENT V3
#
# STANDALONE POST-PROCESSING OF THE COMPLETED V2 DATASET
#
# INPUT:
#   data/generated/image_compiler_route_v2.json
#   data/generated/image_analysis_route_v2.json
#
# OUTPUT:
#   data/generated/image_compiler_route_v3.json
#   data/generated/image_analysis_route_v3.json
#
# DOES NOT:
# - crawl Mapillary
# - download images
# - rerun SegFormer
# - rerun K-means
#
# V3 PURPOSE:
#
# 1. Exact route only
# 2. MRT = 0 m
# 3. HOME = actual journey distance
# 4. Smooth route profile every 15 m
# 5. Individual environmental feature profiles
# 6. Human-readable visual-cluster labels
# ============================================================


from pathlib import Path
from collections import defaultdict, Counter
from datetime import datetime

import json
import math
import numpy as np


# ============================================================
# PROJECT PATH
# ============================================================

cwd = Path.cwd()

PROJECT_ROOT_V3 = (
    cwd.parent
    if cwd.name.lower() == "notebooks"
    else cwd
)


GENERATED_DIR_V3 = (
    PROJECT_ROOT_V3
    / "data"
    / "generated"
)


V2_COMPILER_PATH = (
    GENERATED_DIR_V3
    / "image_compiler_route_v2.json"
)


V2_FULL_PATH = (
    GENERATED_DIR_V3
    / "image_analysis_route_v2.json"
)


V3_COMPILER_PATH = (
    GENERATED_DIR_V3
    / "image_compiler_route_v3.json"
)


V3_FULL_PATH = (
    GENERATED_DIR_V3
    / "image_analysis_route_v3.json"
)


# ============================================================
# CHECK V2
# ============================================================

if not V2_COMPILER_PATH.exists():

    raise RuntimeError(
        "V2 compiler does not exist:\n"
        f"{V2_COMPILER_PATH}"
    )


if not V2_FULL_PATH.exists():

    raise RuntimeError(
        "V2 full analysis does not exist:\n"
        f"{V2_FULL_PATH}"
    )


print()
print(
    "✓ V2 compiler found"
)

print(
    "✓ V2 full analysis found"
)


# ============================================================
# LOAD V2
# ============================================================

with open(
    V2_COMPILER_PATH,
    "r",
    encoding="utf-8"
) as file:

    compiler_v2 = json.load(
        file
    )


with open(
    V2_FULL_PATH,
    "r",
    encoding="utf-8"
) as file:

    full_v2 = json.load(
        file
    )


images_v2 = (
    full_v2.get("images")
    or
    compiler_v2.get("allImages")
    or
    []
)


routes_v2 = (
    compiler_v2.get("routes")
    or
    []
)


clusters_v2 = (
    compiler_v2.get("clusters")
    or
    []
)


print()
print(
    "V2 images:",
    len(images_v2)
)

print(
    "V2 routes:",
    len(routes_v2)
)

print(
    "V2 visual clusters:",
    len(clusters_v2)
)


if not images_v2:

    raise RuntimeError(
        "V2 contains no analysed images."
    )


if not routes_v2:

    raise RuntimeError(
        "V2 contains no route profiles."
    )


# ============================================================
# V3 ROUTE PROFILE SETTINGS
#
# This is the smooth-gradient behaviour we discussed.
# ============================================================

V3_ROUTE_STEP_M = 15.0

V3_SMOOTHING_SIGMA_M = 90.0

V3_MAX_INTERPOLATION_GAP_M = 180.0


print()
print(
    "V3 route interval:",
    V3_ROUTE_STEP_M,
    "m"
)

print(
    "V3 smoothing sigma:",
    V3_SMOOTHING_SIGMA_M,
    "m"
)

print(
    "V3 maximum data gap:",
    V3_MAX_INTERPOLATION_GAP_M,
    "m"
)


# ============================================================
# ENVIRONMENTAL FEATURES
# ============================================================

FEATURES = {

    "greenery": {
        "source":
            "vegetation",

        "label":
            "Greenery",
    },


    "pedestrian": {
        "source":
            "pedestrian_surface",

        "label":
            "Pedestrian surface",
    },


    "road": {
        "source":
            "road",

        "label":
            "Road exposure",
    },


    "vehicle": {
        "source":
            "vehicle",

        "label":
            "Vehicle presence",
    },


    "sky": {
        "source":
            "sky",

        "label":
            "Open-sky exposure",
    },


    "building": {
        "source":
            "building",

        "label":
            "Built frontage",
    },

}


# ============================================================
# HELPERS
# ============================================================

def v3_float(
    value,
    default=0.0
):

    try:

        result = float(
            value
        )

        if math.isfinite(
            result
        ):

            return result

    except Exception:

        pass


    return default


def v3_json_dump(
    path,
    payload
):

    path.parent.mkdir(
        parents=True,
        exist_ok=True
    )


    with open(
        path,
        "w",
        encoding="utf-8"
    ) as file:

        json.dump(
            payload,
            file,
            indent=2,
            ensure_ascii=False
        )


# ============================================================
# DISTANCE FROM MRT
# ============================================================

def image_distance_from_mrt(
    item
):

    value = item.get(
        "distanceFromMrt_m"
    )


    if value is None:

        value = item.get(
            "distanceAlongRoute_m"
        )


    return v3_float(
        value,
        np.nan
    )


# ============================================================
# CONVERT SEGMENTATION SHARE INTO PERCENT
# ============================================================

def segmentation_percent(
    item,
    source_key
):

    shares = (
        item.get(
            "groupShares"
        )
        or
        {}
    )


    value = v3_float(
        shares.get(
            source_key
        ),
        np.nan
    )


    if not math.isfinite(
        value
    ):

        return np.nan


    # Existing semantic shares are expected to be 0–1.
    if value <= 1.5:

        value *= 100.0


    return max(
        0.0,
        min(
            100.0,
            value
        )
    )


# ============================================================
# ROUTE DISTANCE
# ============================================================

def get_route_distance(
    route
):

    series = (
        route.get(
            "routeSeries"
        )
        or
        {}
    )


    candidates = [

        series.get(
            "routeDistance_m"
        ),

        route.get(
            "routeDistance_m"
        ),

        route.get(
            "distance_m"
        ),

        route.get(
            "distanceM"
        ),

    ]


    for candidate in candidates:

        value = v3_float(
            candidate,
            0.0
        )


        if value > 0:

            return value


    return 0.0


# ============================================================
# GROUP ANALYSED IMAGES BY EXACT ROUTE ID
# ============================================================

images_by_route = defaultdict(
    list
)


for item in images_v2:

    route_id = str(
        item.get(
            "routeId"
        )
        or
        ""
    )


    if not route_id:

        continue


    distance = (
        image_distance_from_mrt(
            item
        )
    )


    if not math.isfinite(
        distance
    ):

        continue


    images_by_route[
        route_id
    ].append(
        item
    )


for route_id in images_by_route:

    images_by_route[
        route_id
    ].sort(

        key=
            image_distance_from_mrt

    )


print()
print(
    "Routes containing analysed imagery:",
    len(
        images_by_route
    )
)


# ============================================================
# BUILD RAW ROUTE OBSERVATIONS
# ============================================================

def build_raw_route_points(
    route_images
):

    output = []


    for item in route_images:

        distance = (
            image_distance_from_mrt(
                item
            )
        )


        if not math.isfinite(
            distance
        ):

            continue


        values = {}


        for feature_key, config in (
            FEATURES.items()
        ):

            value = segmentation_percent(
                item,
                config[
                    "source"
                ]
            )


            values[
                feature_key
            ] = (

                round(
                    float(
                        value
                    ),
                    3
                )

                if math.isfinite(
                    value
                )

                else None

            )


        output.append({

            "distance_m":
                round(
                    float(
                        distance
                    ),
                    2
                ),

            "imageId":
                item.get(
                    "imageId"
                ),

            "imageOrder":
                item.get(
                    "imageOrder"
                ),

            "checkpointIndex":
                item.get(
                    "checkpointIndex"
                ),

            "originalImage":
                item.get(
                    "originalImage"
                ),

            "segmentationImage":
                item.get(
                    "segmentationImage"
                ),

            "values":
                values,

        })


    output.sort(
        key=lambda point:
            point[
                "distance_m"
            ]
    )


    return output


# ============================================================
# 15 m GAUSSIAN SMOOTHING
#
# IMPORTANT:
#
# This does not stretch data to another route.
#
# Every route receives its OWN x-axis:
#
# MRT 0 m ---------------------- HOME
#
# Nearby observations have high influence.
# Distant observations have low influence.
#
# If there is no observation within 180 m,
# result remains null rather than being invented.
# ============================================================

def smooth_route_profile(
    raw_points,
    route_distance
):

    if (
        not raw_points
        or
        route_distance <= 0
    ):

        return []


    source_distances = np.asarray(

        [
            point[
                "distance_m"
            ]
            for point
            in raw_points
        ],

        dtype=np.float64

    )


    positions = np.arange(

        0.0,

        route_distance
        +
        V3_ROUTE_STEP_M,

        V3_ROUTE_STEP_M,

        dtype=np.float64

    )


    # Make sure the actual HOME distance is represented.
    if (
        not len(
            positions
        )
        or
        abs(
            positions[-1]
            -
            route_distance
        )
        >
        0.001
    ):

        positions = np.append(
            positions,
            route_distance
        )


    smoothed = []


    for position in positions:

        difference = (

            source_distances
            -
            position

        )


        absolute_difference = np.abs(
            difference
        )


        nearest_observation = float(
            np.min(
                absolute_difference
            )
        )


        values = {}


        # ----------------------------------------------------
        # No evidence nearby:
        #
        # do NOT fake the value.
        # ----------------------------------------------------

        if (
            nearest_observation
            >
            V3_MAX_INTERPOLATION_GAP_M
        ):

            for feature_key in FEATURES:

                values[
                    feature_key
                ] = None


            smoothed.append({

                "distance_m":
                    round(
                        float(
                            position
                        ),
                        2
                    ),

                "values":
                    values,

                "nearestObservation_m":
                    round(
                        nearest_observation,
                        2
                    ),

                "supportCount":
                    0,

                "confidence":
                    0.0,

            })


            continue


        # ----------------------------------------------------
        # Gaussian weight:
        #
        # exp(-0.5 * (distance / sigma)^2)
        # ----------------------------------------------------

        weights = np.exp(

            -0.5
            *
            (
                difference
                /
                V3_SMOOTHING_SIGMA_M
            )
            ** 2

        )


        # Do not use very distant images.
        weights[

            absolute_difference
            >
            V3_MAX_INTERPOLATION_GAP_M

        ] = 0.0


        for feature_key in FEATURES:

            source_values = np.asarray(

                [

                    (
                        np.nan

                        if point[
                            "values"
                        ].get(
                            feature_key
                        )
                        is None

                        else

                        float(
                            point[
                                "values"
                            ][
                                feature_key
                            ]
                        )
                    )

                    for point
                    in raw_points

                ],

                dtype=np.float64

            )


            valid = (

                np.isfinite(
                    source_values
                )

                &

                (
                    weights
                    >
                    0
                )

            )


            if not valid.any():

                values[
                    feature_key
                ] = None

                continue


            weighted_value = np.average(

                source_values[
                    valid
                ],

                weights=
                    weights[
                        valid
                    ]

            )


            values[
                feature_key
            ] = round(
                float(
                    weighted_value
                ),
                3
            )


        support_count = int(

            (
                absolute_difference
                <=
                (
                    V3_SMOOTHING_SIGMA_M
                    *
                    2
                )
            ).sum()

        )


        confidence = min(
            1.0,
            support_count
            /
            3.0
        )


        smoothed.append({

            "distance_m":
                round(
                    float(
                        position
                    ),
                    2
                ),

            "values":
                values,

            "nearestObservation_m":
                round(
                    nearest_observation,
                    2
                ),

            "supportCount":
                support_count,

            "confidence":
                round(
                    confidence,
                    3
                ),

        })


    return smoothed


# ============================================================
# FEATURE SUMMARY FOR ONE ROUTE
# ============================================================

def feature_summaries(
    smoothed
):

    output = {}


    for feature_key, config in (
        FEATURES.items()
    ):

        valid_points = [

            point

            for point
            in smoothed

            if (
                point[
                    "values"
                ].get(
                    feature_key
                )
                is not None
            )

        ]


        if not valid_points:

            output[
                feature_key
            ] = {

                "label":
                    config[
                        "label"
                    ],

                "averagePercent":
                    None,

                "minimumPercent":
                    None,

                "maximumPercent":
                    None,

                "peakDistance_m":
                    None,

                "minimumDistance_m":
                    None,

            }


            continue


        values = np.asarray(

            [
                point[
                    "values"
                ][
                    feature_key
                ]
                for point
                in valid_points
            ],

            dtype=float

        )


        maximum_index = int(
            np.argmax(
                values
            )
        )


        minimum_index = int(
            np.argmin(
                values
            )
        )


        output[
            feature_key
        ] = {

            "label":
                config[
                    "label"
                ],

            "averagePercent":
                round(
                    float(
                        np.mean(
                            values
                        )
                    ),
                    2
                ),

            "minimumPercent":
                round(
                    float(
                        np.min(
                            values
                        )
                    ),
                    2
                ),

            "maximumPercent":
                round(
                    float(
                        np.max(
                            values
                        )
                    ),
                    2
                ),

            "peakDistance_m":
                round(
                    float(
                        valid_points[
                            maximum_index
                        ][
                            "distance_m"
                        ]
                    ),
                    2
                ),

            "minimumDistance_m":
                round(
                    float(
                        valid_points[
                            minimum_index
                        ][
                            "distance_m"
                        ]
                    ),
                    2
                ),

        }


    return output


# ============================================================
# BUILD ALL ROUTE PROFILES
# ============================================================

routes_v3 = []


for route_number, route in enumerate(
    routes_v2,
    start=1
):

    route_id = str(
        route.get(
            "routeId"
        )
        or
        ""
    )


    route_images = (
        images_by_route.get(
            route_id
        )
        or
        []
    )


    route_distance = (
        get_route_distance(
            route
        )
    )


    # --------------------------------------------------------
    # Fallback to furthest route image.
    # --------------------------------------------------------

    if (
        route_distance <= 0
        and
        route_images
    ):

        route_distance = max(

            image_distance_from_mrt(
                item
            )

            for item
            in route_images

        )


    raw = (
        build_raw_route_points(
            route_images
        )
    )


    smoothed = (
        smooth_route_profile(
            raw,
            route_distance
        )
    )


    summaries = (
        feature_summaries(
            smoothed
        )
    )


    route_series_v3 = {

        "distanceOrigin":
            "MRT",

        "distanceDestination":
            "HOME",

        "distanceConvention":
            "MRT_0_TO_HOME",

        "routeDistance_m":
            round(
                route_distance,
                2
            ),

        "step_m":
            V3_ROUTE_STEP_M,

        "smoothingMethod":
            "GAUSSIAN_DISTANCE_WEIGHTED",

        "smoothingSigma_m":
            V3_SMOOTHING_SIGMA_M,

        "maximumInterpolationGap_m":
            V3_MAX_INTERPOLATION_GAP_M,

        "rawObservationCount":
            len(
                raw
            ),

        "raw":
            raw,

        "smoothed":
            smoothed,

        "featureSummaries":
            summaries,

    }


    routes_v3.append({

        **route,

        # Main frontend version.
        "routeSeries":
            route_series_v3,

        # Keep explicit V3 copy too.
        "routeSeriesV3":
            route_series_v3,

        "routeProfileVersion":
            3,

        "routeProfileMode":
            "SELECTED_ROUTE_ONLY",

        "distanceConvention":
            "MRT_0_TO_HOME",

    })


    if (
        route_number <= 10
    ):

        print(

            f"[{route_number}/{len(routes_v2)}] ",

            route_id,

            "| distance:",

            round(
                route_distance,
                1
            ),

            "m",

            "| images:",

            len(
                raw
            ),

            "| profile points:",

            len(
                smoothed
            )

        )


# ============================================================
# HUMAN-READABLE K-MEANS LABELS
#
# We keep the original cluster number.
#
# Labels are descriptive interpretations of centroid
# composition, not newly trained classes.
# ============================================================

CLUSTER_FEATURES = [

    "vegetation",

    "pedestrian_surface",

    "road",

    "vehicle",

    "sky",

    "building",

]


CLUSTER_PRETTY = {

    "vegetation":
        "greenery",

    "pedestrian_surface":
        "pedestrian space",

    "road":
        "road exposure",

    "vehicle":
        "vehicle presence",

    "sky":
        "open sky",

    "building":
        "built frontage",

}


PAIR_LABELS = {

    frozenset(
        [
            "vegetation",
            "pedestrian_surface"
        ]
    ):
        "Green pedestrian corridor",


    frozenset(
        [
            "road",
            "vehicle"
        ]
    ):
        "Traffic-exposed roadway",


    frozenset(
        [
            "building",
            "pedestrian_surface"
        ]
    ):
        "Built pedestrian frontage",


    frozenset(
        [
            "sky",
            "road"
        ]
    ):
        "Open road corridor",


    frozenset(
        [
            "vegetation",
            "building"
        ]
    ):
        "Green built-edge street",


    frozenset(
        [
            "sky",
            "vegetation"
        ]
    ):
        "Open green corridor",


    frozenset(
        [
            "building",
            "road"
        ]
    ):
        "Built road corridor",


    frozenset(
        [
            "road",
            "pedestrian_surface"
        ]
    ):
        "Mixed pedestrian-road edge",


    frozenset(
        [
            "sky",
            "building"
        ]
    ):
        "Open built frontage",


    frozenset(
        [
            "vehicle",
            "building"
        ]
    ):
        "Vehicle-facing frontage",

}


# ============================================================
# GET CLUSTER CENTROID VALUES
# ============================================================

cluster_rows = []


for cluster in clusters_v2:

    mean_shares = (
        cluster.get(
            "meanGroupShares"
        )
        or
        {}
    )


    row = [

        v3_float(
            mean_shares.get(
                feature
            ),
            0.0
        )

        for feature
        in CLUSTER_FEATURES

    ]


    cluster_rows.append(
        row
    )


cluster_matrix = np.asarray(
    cluster_rows,
    dtype=float
)


if len(
    cluster_matrix
):

    cluster_means = (
        np.mean(
            cluster_matrix,
            axis=0
        )
    )


    cluster_stds = (
        np.std(
            cluster_matrix,
            axis=0
        )
    )


    cluster_stds[
        cluster_stds
        <
        1e-9
    ] = 1.0


    cluster_z = (

        cluster_matrix
        -
        cluster_means

    ) / cluster_stds

else:

    cluster_z = np.empty(
        (
            0,
            len(
                CLUSTER_FEATURES
            )
        )
    )


# ============================================================
# SEMANTIC LABEL
# ============================================================

def cluster_semantic_label(
    cluster_index,
    cluster
):

    if (
        cluster_index
        >=
        len(
            cluster_z
        )
    ):

        return {

            "label":
                "Mixed visual environment",

            "description":
                (
                    "This cluster does not have enough "
                    "centroid information for a stronger label."
                ),

        }


    z_values = (
        cluster_z[
            cluster_index
        ]
    )


    order = np.argsort(
        z_values
    )[::-1]


    first_feature = (
        CLUSTER_FEATURES[
            order[0]
        ]
    )


    second_feature = (
        CLUSTER_FEATURES[
            order[1]
        ]
    )


    pair = frozenset(
        [
            first_feature,
            second_feature
        ]
    )


    label = (
        PAIR_LABELS.get(
            pair
        )
        or
        (
            f"{CLUSTER_PRETTY[first_feature].title()} + "
            f"{CLUSTER_PRETTY[second_feature]}"
        )
    )


    mean_shares = (
        cluster.get(
            "meanGroupShares"
        )
        or
        {}
    )


    first_value = (
        v3_float(
            mean_shares.get(
                first_feature
            )
        )
        *
        100
    )


    second_value = (
        v3_float(
            mean_shares.get(
                second_feature
            )
        )
        *
        100
    )


    description = (

        f"Relatively characterised by "
        f"{CLUSTER_PRETTY[first_feature]} "
        f"({first_value:.1f}%) and "
        f"{CLUSTER_PRETTY[second_feature]} "
        f"({second_value:.1f}%)."

    )


    return {

        "label":
            label,

        "description":
            description,

        "primaryFeature":
            first_feature,

        "secondaryFeature":
            second_feature,

    }


# ============================================================
# APPLY LABELS
# ============================================================

clusters_v3 = []


for index, cluster in enumerate(
    clusters_v2
):

    semantics = (
        cluster_semantic_label(
            index,
            cluster
        )
    )


    clusters_v3.append({

        **cluster,

        "semanticLabel":
            semantics[
                "label"
            ],

        "semanticDescription":
            semantics[
                "description"
            ],

        "primaryFeature":
            semantics.get(
                "primaryFeature"
            ),

        "secondaryFeature":
            semantics.get(
                "secondaryFeature"
            ),

    })


cluster_label_lookup = {}


for index, cluster in enumerate(
    clusters_v3
):

    cluster_id = int(

        cluster.get(
            "cluster",
            index
        )

    )


    cluster_label_lookup[
        cluster_id
    ] = cluster.get(
        "semanticLabel"
    )


print()
print(
    "Visual cluster labels:"
)


for cluster in clusters_v3:

    print(

        f"Cluster {cluster.get('cluster')}:",

        cluster.get(
            "semanticLabel"
        )

    )


# ============================================================
# ATTACH LABEL TO EVERY IMAGE
# ============================================================

images_v3 = []


for item in images_v2:

    cluster_id = int(

        item.get(
            "visualCluster",

            item.get(
                "cluster",
                0
            )

        )
        or
        0

    )


    images_v3.append({

        **item,

        "clusterLabel":
            cluster_label_lookup.get(
                cluster_id,
                f"Visual type {cluster_id}"
            ),

    })


# ============================================================
# DOMINANT VISUAL TYPE FOR EACH ROUTE
# ============================================================

images_v3_by_route = defaultdict(
    list
)


for item in images_v3:

    route_id = str(
        item.get(
            "routeId"
        )
        or
        ""
    )


    if route_id:

        images_v3_by_route[
            route_id
        ].append(
            item
        )


final_routes_v3 = []


for route in routes_v3:

    route_id = str(
        route.get(
            "routeId"
        )
        or
        ""
    )


    route_images = (
        images_v3_by_route.get(
            route_id
        )
        or
        []
    )


    counts = Counter(

        int(
            item.get(
                "visualCluster",
                item.get(
                    "cluster",
                    0
                )
            )
            or
            0
        )

        for item
        in route_images

    )


    dominant_cluster = (

        counts.most_common(
            1
        )[0][0]

        if counts

        else None

    )


    dominant_label = (

        cluster_label_lookup.get(
            dominant_cluster
        )

        if dominant_cluster
        is not None

        else None

    )


    final_routes_v3.append({

        **route,

        "dominantVisualCluster":
            dominant_cluster,

        "dominantVisualClusterLabel":
            dominant_label,

    })


# ============================================================
# REBUILD ML DISTRIBUTION WITH LABELS
# ============================================================

ml_v2 = (
    compiler_v2.get(
        "mlEvidence"
    )
    or
    {}
)


cluster_distribution = []


for index, cluster in enumerate(
    clusters_v3
):

    cluster_id = int(
        cluster.get(
            "cluster",
            index
        )
    )


    count = sum(

        1

        for item
        in images_v3

        if int(

            item.get(
                "visualCluster",

                item.get(
                    "cluster",
                    0
                )

            )
            or
            0

        )
        ==
        cluster_id

    )


    cluster_distribution.append({

        "cluster":
            cluster_id,

        "label":
            cluster.get(
                "semanticLabel"
            ),

        "description":
            cluster.get(
                "semanticDescription"
            ),

        "primaryFeature":
            cluster.get(
                "primaryFeature"
            ),

        "secondaryFeature":
            cluster.get(
                "secondaryFeature"
            ),

        "count":
            count,

        "meanGroupShares":
            cluster.get(
                "meanGroupShares",
                {}
            ),

        "meanVisualMetrics":
            cluster.get(
                "meanVisualMetrics",
                {}
            ),

    })


ml_v3 = {

    **ml_v2,

    "imageCount":
        len(
            images_v3
        ),

    "visualClusterCount":
        len(
            clusters_v3
        ),

    "clusterDistribution":
        cluster_distribution,

    "routeProfileMethodology": {

        "profileMode":
            "SELECTED_ROUTE_ONLY",

        "distanceOrigin":
            "MRT",

        "distanceDestination":
            "HOME",

        "distanceConvention":
            "MRT_0_TO_HOME",

        "step_m":
            V3_ROUTE_STEP_M,

        "smoothingMethod":
            "GAUSSIAN_DISTANCE_WEIGHTED",

        "smoothingSigma_m":
            V3_SMOOTHING_SIGMA_M,

        "maximumInterpolationGap_m":
            V3_MAX_INTERPOLATION_GAP_M,

        "featureSource":
            "SEGFORMER_SEMANTIC_PROPORTIONS",

    },

}


# ============================================================
# FRONTEND LIGHTWEIGHT IMAGES
# ============================================================

frontend_images_v3 = []


for item in images_v3:

    frontend_images_v3.append({

        "imageId":
            item.get(
                "imageId"
            ),

        "mrtName":
            item.get(
                "mrtName"
            )
            or
            item.get(
                "station"
            ),

        "station":
            item.get(
                "station"
            )
            or
            item.get(
                "mrtName"
            ),

        "homeName":
            item.get(
                "homeName"
            ),

        "routeId":
            item.get(
                "routeId"
            ),

        "routeOrder":
            item.get(
                "routeOrder"
            ),

        "routeKey":
            item.get(
                "routeKey"
            ),

        "imageOrder":
            item.get(
                "imageOrder"
            ),

        "imageKey":
            item.get(
                "imageKey"
            ),

        "crawlKey":
            item.get(
                "crawlKey"
            ),

        "distanceFromMrt_m":
            item.get(
                "distanceFromMrt_m"
            ),

        "checkpointIndex":
            item.get(
                "checkpointIndex"
            ),

        "originalImage":
            item.get(
                "originalImage"
            ),

        "segmentationImage":
            item.get(
                "segmentationImage"
            ),

        "visualCluster":
            item.get(
                "visualCluster"
            ),

        "cluster":
            item.get(
                "visualCluster",
                item.get(
                    "cluster"
                )
            ),

        "clusterLabel":
            item.get(
                "clusterLabel"
            ),

        "somCell":
            item.get(
                "somCell"
            ),

        "mrtCluster":
            item.get(
                "mrtCluster"
            ),

        "groupShares":
            item.get(
                "groupShares"
            ),

        "visualMetrics":
            item.get(
                "visualMetrics"
            ),

    })


# ============================================================
# BUILD V3 COMPILER
# ============================================================

compiler_v3 = {

    **compiler_v2,

    "version":
        7,

    "dataset":
        "route_v3",

    "createdAt":
        datetime.now()
        .isoformat(
            timespec="seconds"
        ),

    "imageCount":
        len(
            images_v3
        ),

    "clusterCount":
        len(
            clusters_v3
        ),

    "distanceConvention":
        "MRT_0_TO_HOME",

    "routeProfileMode":
        "SELECTED_ROUTE_ONLY",

    "routeProfileStep_m":
        V3_ROUTE_STEP_M,

    "routeSmoothingSigma_m":
        V3_SMOOTHING_SIGMA_M,

    "routeMaximumGap_m":
        V3_MAX_INTERPOLATION_GAP_M,

    "clusters":
        clusters_v3,

    "allImages":
        frontend_images_v3,

    "routes":
        final_routes_v3,

    "mlEvidence":
        ml_v3,

    "fullAnalysisFile":
        "./data/generated/image_analysis_route_v3.json",

}


# ============================================================
# BUILD FULL V3
# ============================================================

full_v3 = {

    **full_v2,

    "version":
        7,

    "dataset":
        "route_v3",

    "createdAt":
        compiler_v3[
            "createdAt"
        ],

    "imageCount":
        len(
            images_v3
        ),

    "clusterCount":
        len(
            clusters_v3
        ),

    "distanceConvention":
        "MRT_0_TO_HOME",

    "routeProfileMode":
        "SELECTED_ROUTE_ONLY",

    "routeProfileStep_m":
        V3_ROUTE_STEP_M,

    "routeSmoothingSigma_m":
        V3_SMOOTHING_SIGMA_M,

    "routeMaximumGap_m":
        V3_MAX_INTERPOLATION_GAP_M,

    "images":
        images_v3,

    "clusters":
        clusters_v3,

    "routes":
        final_routes_v3,

    "mlEvidence":
        ml_v3,

}


# ============================================================
# SAVE
# ============================================================

v3_json_dump(
    V3_COMPILER_PATH,
    compiler_v3
)


v3_json_dump(
    V3_FULL_PATH,
    full_v3
)


# ============================================================
# QUALITY REPORT
# ============================================================

route_image_counts = [

    len(
        images_v3_by_route.get(
            str(
                route.get(
                    "routeId"
                )
                or
                ""
            ),
            []
        )
    )

    for route
    in final_routes_v3

]


routes_with_images = sum(
    1
    for count
    in route_image_counts
    if count > 0
)


routes_with_10_images = sum(
    1
    for count
    in route_image_counts
    if count >= 10
)


profile_point_counts = [

    len(
        (
            route.get(
                "routeSeries"
            )
            or
            {}
        ).get(
            "smoothed",
            []
        )
    )

    for route
    in final_routes_v3

]


# ============================================================
# DONE
# ============================================================

print()
print(
    "================================================"
)

print(
    "ROUTE ENVIRONMENT V3 GENERATED"
)

print(
    "================================================"
)

print(
    "Images:",
    len(
        images_v3
    )
)

print(
    "Routes:",
    len(
        final_routes_v3
    )
)

print(
    "Routes with imagery:",
    routes_with_images
)

print(
    "Routes with >= 10 images:",
    routes_with_10_images
)

print(
    "Visual clusters:",
    len(
        clusters_v3
    )
)

print()
print(
    "Route profile:"
)

print(
    "  MRT = 0 m"
)

print(
    "  HOME = exact route distance"
)

print(
    "  Interval:",
    V3_ROUTE_STEP_M,
    "m"
)

print(
    "  Gaussian sigma:",
    V3_SMOOTHING_SIGMA_M,
    "m"
)

print(
    "  Maximum evidence gap:",
    V3_MAX_INTERPOLATION_GAP_M,
    "m"
)

print()
print(
    "Average V3 profile points / route:",
    round(
        float(
            np.mean(
                profile_point_counts
            )
        ),
        1
    )
)

print()
print(
    "Saved compiler:"
)

print(
    V3_COMPILER_PATH
)

print()
print(
    "Saved full analysis:"
)

print(
    V3_FULL_PATH
)

print(
    "================================================"
)


✓ V2 compiler found
✓ V2 full analysis found

V2 images: 1130
V2 routes: 192
V2 visual clusters: 10

V3 route interval: 15.0 m
V3 smoothing sigma: 90.0 m
V3 maximum data gap: 180.0 m

Routes containing analysed imagery: 170
[1/192]  ADMIRALTY__R01 | distance: 1157.3 m | images: 5 | profile points: 80
[2/192]  ALJUNIED__R01 | distance: 23.2 m | images: 0 | profile points: 0
[3/192]  ANG_MO_KIO__R01 | distance: 1367.6 m | images: 1 | profile points: 94
[4/192]  BEAUTY_WORLD__R01 | distance: 177.4 m | images: 2 | profile points: 14
[5/192]  BEDOK_NORTH__R01 | distance: 662.8 m | images: 4 | profile points: 47
[6/192]  BEDOK_RESERVOIR__R01 | distance: 311.9 m | images: 4 | profile points: 23
[7/192]  BEDOK__R01 | distance: 1396.9 m | images: 15 | profile points: 96
[8/192]  BISHAN__R01 | distance: 651.8 m | images: 0 | profile points: 0
[9/192]  BOON_KENG__R01 | distance: 1238.7 m | images: 6 | profile points: 85
[10/192]  BOON_LAY__R01 | distance: 55.1 m | images: 0 | profile points: 0



In [30]:
from pathlib import Path

generated = PROJECT_ROOT / "data" / "generated"

checks = {
    "V3 compiler":
        generated / "image_compiler_route_v3.json",

    "V3 full":
        generated / "image_analysis_route_v3.json",
}

for name, path in checks.items():
    print(
        "✓" if path.exists() else "✗",
        name,
        "->",
        path
    )

✓ V3 compiler -> C:\Users\MSI\OneDrive\Desktop\family-toilet-search\data\generated\image_compiler_route_v3.json
✓ V3 full -> C:\Users\MSI\OneDrive\Desktop\family-toilet-search\data\generated\image_analysis_route_v3.json


In [31]:
# ============================================================
# ROUTE ENVIRONMENT V4
#
# INDIVIDUAL SEMANTIC DIAGNOSTICS
# + ROUTE-SPECIFIC DESIGN INTERVENTIONS
#
# INPUT:
#   image_compiler_route_v3.json
#   image_analysis_route_v3.json
#
# OUTPUT:
#   image_compiler_route_v4.json
#   image_analysis_route_v4.json
#
# DIAGNOSTIC IMAGES:
#   diagnostic_segmentations_v4/
#
# IMPORTANT:
#
# - NO MAPILLARY RECRAWL
# - NO IMAGE REDOWNLOAD
# - NO K-MEANS RERUN
# - REUSES V3 15 m SMOOTH ROUTE PROFILES
#
# - MRT = 0 m
# - HOME = ACTUAL ROUTE DISTANCE
#
# ============================================================


from pathlib import Path
from collections import defaultdict
from datetime import datetime

import json
import math
import shutil
import numpy as np

from PIL import Image

import torch

from transformers import (
    AutoImageProcessor,
    AutoModelForSemanticSegmentation
)


# ============================================================
# PROJECT PATH
# ============================================================

cwd = Path.cwd()

PROJECT_ROOT_V4 = (
    cwd.parent
    if cwd.name.lower() == "notebooks"
    else cwd
)


GENERATED_DIR_V4 = (
    PROJECT_ROOT_V4
    / "data"
    / "generated"
)


V3_COMPILER_PATH = (
    GENERATED_DIR_V4
    / "image_compiler_route_v3.json"
)


V3_FULL_PATH = (
    GENERATED_DIR_V4
    / "image_analysis_route_v3.json"
)


V4_COMPILER_PATH = (
    GENERATED_DIR_V4
    / "image_compiler_route_v4.json"
)


V4_FULL_PATH = (
    GENERATED_DIR_V4
    / "image_analysis_route_v4.json"
)


V4_DIAGNOSTIC_ROOT = (
    GENERATED_DIR_V4
    / "diagnostic_segmentations_v4"
)


V4_DIAGNOSTIC_ROOT.mkdir(
    parents=True,
    exist_ok=True
)


# ============================================================
# CHECK INPUTS
# ============================================================

if not V3_COMPILER_PATH.exists():

    raise RuntimeError(
        "V3 compiler does not exist:\n"
        f"{V3_COMPILER_PATH}"
    )


if not V3_FULL_PATH.exists():

    raise RuntimeError(
        "V3 full analysis does not exist:\n"
        f"{V3_FULL_PATH}"
    )


print()
print("✓ V3 compiler found")
print("✓ V3 full analysis found")


# ============================================================
# LOAD V3
# ============================================================

with open(
    V3_COMPILER_PATH,
    "r",
    encoding="utf-8"
) as file:

    compiler_v3 = json.load(
        file
    )


with open(
    V3_FULL_PATH,
    "r",
    encoding="utf-8"
) as file:

    full_v3 = json.load(
        file
    )


images_v3 = (
    full_v3.get("images")
    or
    compiler_v3.get("allImages")
    or
    []
)


routes_v3 = (
    compiler_v3.get("routes")
    or
    []
)


clusters_v3 = (
    compiler_v3.get("clusters")
    or
    []
)


print()
print(
    "V3 images:",
    len(images_v3)
)

print(
    "V3 routes:",
    len(routes_v3)
)

print(
    "V3 clusters:",
    len(clusters_v3)
)


if not images_v3:

    raise RuntimeError(
        "V3 contains no analysed images."
    )


# ============================================================
# V4 SETTINGS
# ============================================================

SEGFORMER_MODEL_NAME_V4 = (
    "nvidia/segformer-b2-finetuned-ade-512-512"
)


DEVICE_V4 = (
    "cuda"
    if torch.cuda.is_available()
    else
    "cpu"
)


# If diagnostic image already exists,
# do not regenerate it.
FORCE_V4_DIAGNOSTICS = False


# ============================================================
# DESIGN CONDITIONS
#
# PRIMARY:
# pedestrian space / walking continuity
#
# SUPPORTING:
# greenery / road / vehicles / sky / building
# ============================================================

CONDITIONS_V4 = {

    "pedestrian": {

        "label":
            "Pedestrian space",

        "semanticGroup":
            "pedestrian_surface",

        "problemDirection":
            "low",

        "primary":
            True,

        "description":
            (
                "Visible walking surface available for "
                "pedestrian and stroller movement."
            ),

        "recommendation":
            (
                "Review continuous footpath width, stroller "
                "clearance, obstructions, level changes and "
                "pedestrian-priority space."
            ),

    },


    "greenery": {

        "label":
            "Greenery",

        "semanticGroup":
            "vegetation",

        "problemDirection":
            "low",

        "primary":
            False,

        "description":
            (
                "Visible vegetation and planted landscape "
                "along the walking journey."
            ),

        "recommendation":
            (
                "Investigate street-tree planting, planted "
                "buffers and landscape shade where greenery "
                "is persistently limited."
            ),

    },


    "road": {

        "label":
            "Road exposure",

        "semanticGroup":
            "road",

        "problemDirection":
            "high",

        "primary":
            False,

        "description":
            (
                "Visual proportion occupied by roadway."
            ),

        "recommendation":
            (
                "Review pedestrian separation, traffic buffers, "
                "crossing distance, refuge opportunities and "
                "traffic-calming measures."
            ),

    },


    "vehicle": {

        "label":
            "Vehicle presence",

        "semanticGroup":
            "vehicle",

        "problemDirection":
            "high",

        "primary":
            False,

        "description":
            (
                "Visible vehicle presence along the route."
            ),

        "recommendation":
            (
                "Inspect locations with persistent vehicle "
                "presence for traffic conflict, drop-off activity "
                "and pedestrian protection."
            ),

    },


    "sky": {

        "label":
            "Open-sky exposure",

        "semanticGroup":
            "sky",

        "problemDirection":
            "high",

        "primary":
            False,

        "description":
            (
                "Visible open sky. This is used as a visual "
                "exposure proxy rather than a direct thermal "
                "measurement."
            ),

        "recommendation":
            (
                "Investigate shade trees, covered walkways or "
                "weather-protection structures where visual "
                "exposure remains high."
            ),

    },


    "building": {

        "label":
            "Built frontage",

        "semanticGroup":
            "building",

        "problemDirection":
            "context",

        "primary":
            False,

        "description":
            (
                "Visible building frontage and built enclosure."
            ),

        "recommendation":
            (
                "Use frontage changes to inspect inactive edges, "
                "enclosure, wayfinding and spatial legibility."
            ),

    },

}


# ============================================================
# ADE20K SEMANTIC GROUP KEYWORDS
#
# Derived from the model's id2label map.
# ============================================================

GROUP_KEYWORDS_V4 = {

    "pedestrian_surface": [
        "sidewalk",
        "pavement",
        "path",
        "walkway",
        "floor"
    ],

    "vegetation": [
        "tree",
        "grass",
        "plant",
        "field",
        "flower"
    ],

    "road": [
        "road",
        "street"
    ],

    "vehicle": [
        "car",
        "truck",
        "bus",
        "van",
        "motorbike",
        "motorcycle",
        "bicycle"
    ],

    "sky": [
        "sky"
    ],

    "building": [
        "building",
        "house",
        "skyscraper"
    ],

}


# ============================================================
# BASIC HELPERS
# ============================================================

def v4_float(
    value,
    default=0.0
):

    try:

        result = float(
            value
        )

        if math.isfinite(
            result
        ):

            return result

    except Exception:

        pass


    return default


def v4_slugify(
    value
):

    text = str(
        value
        or
        ""
    ).strip().lower()


    output = []


    previous_dash = False


    for character in text:

        if character.isalnum():

            output.append(
                character
            )

            previous_dash = False

        else:

            if not previous_dash:

                output.append(
                    "-"
                )

                previous_dash = True


    return (
        "".join(
            output
        )
        .strip("-")
        or
        "unknown"
    )


def v4_json_dump(
    path,
    payload
):

    path.parent.mkdir(
        parents=True,
        exist_ok=True
    )


    with open(
        path,
        "w",
        encoding="utf-8"
    ) as file:

        json.dump(
            payload,
            file,
            indent=2,
            ensure_ascii=False
        )


# ============================================================
# BROWSER PATH
# ============================================================

def v4_browser_path(
    path
):

    path = Path(
        path
    ).resolve()


    try:

        relative = path.relative_to(
            PROJECT_ROOT_V4.resolve()
        )


        return (
            "./"
            +
            str(relative)
            .replace(
                "\\",
                "/"
            )
        )


    except Exception:

        return str(
            path
        ).replace(
            "\\",
            "/"
        )


# ============================================================
# RESOLVE STORED BROWSER PATH TO LOCAL FILE
# ============================================================

def v4_local_path(
    browser_value
):

    if not browser_value:
        return None


    text = str(
        browser_value
    ).strip()


    if text.startswith(
        "./"
    ):

        text = text[2:]


    candidate = (
        PROJECT_ROOT_V4
        /
        text
    )


    if candidate.exists():

        return candidate


    raw = Path(
        browser_value
    )


    if raw.exists():

        return raw


    return None


# ============================================================
# LOAD SEGFORMER
# ============================================================

print()
print(
    "Loading SegFormer for V4 diagnostics..."
)

print(
    "Device:",
    DEVICE_V4
)


processor_v4 = (
    AutoImageProcessor
    .from_pretrained(
        SEGFORMER_MODEL_NAME_V4
    )
)


model_v4 = (
    AutoModelForSemanticSegmentation
    .from_pretrained(
        SEGFORMER_MODEL_NAME_V4
    )
    .to(
        DEVICE_V4
    )
)


model_v4.eval()


# ============================================================
# CLASS LABELS
# ============================================================

ID2LABEL_V4 = {

    int(key):
        str(value).lower()

    for key, value
    in model_v4.config.id2label.items()

}


def v4_group_class_ids(
    group_name
):

    keywords = (
        GROUP_KEYWORDS_V4[
            group_name
        ]
    )


    output = []


    for class_id, label in (
        ID2LABEL_V4.items()
    ):

        if any(
            keyword in label
            for keyword
            in keywords
        ):

            output.append(
                class_id
            )


    return output


GROUP_CLASS_IDS_V4 = {

    group_name:
        v4_group_class_ids(
            group_name
        )

    for group_name
    in GROUP_KEYWORDS_V4

}


print()
print(
    "V4 semantic groups:"
)


for group_name, class_ids in (
    GROUP_CLASS_IDS_V4.items()
):

    print(
        " ",
        group_name,
        "→",
        len(
            class_ids
        ),
        "classes"
    )


# ============================================================
# SEGMENT IMAGE INTO CLASS MAP
# ============================================================

def v4_segmentation_map(
    image_path
):

    image = (
        Image
        .open(
            image_path
        )
        .convert(
            "RGB"
        )
    )


    inputs = processor_v4(
        images=image,
        return_tensors="pt"
    )


    inputs = {

        key:
            value.to(
                DEVICE_V4
            )

        for key, value
        in inputs.items()

    }


    with torch.no_grad():

        outputs = model_v4(
            **inputs
        )


    logits = torch.nn.functional.interpolate(

        outputs.logits,

        size=(
            image.height,
            image.width
        ),

        mode="bilinear",

        align_corners=False

    )


    prediction = (
        logits
        .argmax(
            dim=1
        )[0]
        .detach()
        .cpu()
        .numpy()
        .astype(
            np.int16
        )
    )


    return (
        image,
        prediction
    )


# ============================================================
# MAKE INDIVIDUAL DIAGNOSTIC
#
# Target condition = original colour
# Everything else = muted grayscale
# ============================================================

def v4_make_diagnostic(
    original_image,
    prediction,
    class_ids,
    output_path
):

    output_path.parent.mkdir(
        parents=True,
        exist_ok=True
    )


    array = np.asarray(
        original_image
    ).astype(
        np.float32
    )


    mask = np.isin(
        prediction,
        np.asarray(
            class_ids
        )
    )


    # --------------------------------------------------------
    # Contextual grayscale background
    # --------------------------------------------------------

    grey = np.mean(
        array,
        axis=2,
        keepdims=True
    )


    muted = np.repeat(
        grey,
        3,
        axis=2
    )


    muted *= 0.28


    result = muted


    # --------------------------------------------------------
    # Target pixels retain original colour
    # --------------------------------------------------------

    result[
        mask
    ] = np.clip(
        array[
            mask
        ]
        *
        1.15,
        0,
        255
    )


    result = (
        result
        .clip(
            0,
            255
        )
        .astype(
            np.uint8
        )
    )


    Image.fromarray(
        result
    ).save(
        output_path,
        quality=90
    )


# ============================================================
# BUILD DIAGNOSTIC IMAGES
# ============================================================

images_v4 = []


print()
print(
    "Generating V4 individual segmentations..."
)

print(
    "Images:",
    len(
        images_v3
    )
)


for index, item in enumerate(
    images_v3,
    start=1
):

    original_path = (
        v4_local_path(
            item.get(
                "originalImage"
            )
        )
    )


    if original_path is None:

        print(
            f"[{index}/{len(images_v3)}] "
            "missing original image"
        )


        images_v4.append({
            **item,

            "diagnosticSegmentations":
                {}
        })


        continue


    mrt_name = (
        item.get(
            "mrtName"
        )
        or
        item.get(
            "station"
        )
        or
        "Unknown MRT"
    )


    home_name = (
        item.get(
            "homeName"
        )
        or
        "Residential destination"
    )


    route_id = str(
        item.get(
            "routeId"
        )
        or
        "unknown-route"
    )


    route_order = int(
        item.get(
            "routeOrder"
        )
        or
        0
    )


    image_order = int(
        item.get(
            "imageOrder"
        )
        or
        index
    )


    mrt_key = v4_slugify(
        mrt_name
    )


    route_key = (

        item.get(
            "routeKey"
        )

        or

        (
            f"r{route_order:03d}"
            f"__{v4_slugify(home_name)}"
        )

    )


    image_folder = (

        V4_DIAGNOSTIC_ROOT
        /
        mrt_key
        /
        route_key
        /
        f"img{image_order:04d}"

    )


    image_folder.mkdir(
        parents=True,
        exist_ok=True
    )


    diagnostic_paths = {}


    # --------------------------------------------------------
    # Keep reference to existing overall segmentation
    # --------------------------------------------------------

    existing_overall = (
        v4_local_path(
            item.get(
                "segmentationImage"
            )
        )
    )


    if existing_overall:

        diagnostic_paths[
            "overall"
        ] = v4_browser_path(
            existing_overall
        )


    # --------------------------------------------------------
    # Check whether all diagnostics already exist
    # --------------------------------------------------------

    expected_paths = {

        condition_key:
            (
                image_folder
                /
                f"{condition_key}.jpg"
            )

        for condition_key
        in CONDITIONS_V4

    }


    all_cached = all(

        path.exists()

        for path
        in expected_paths.values()

    )


    print(

        f"[{index}/{len(images_v3)}] "

        f"{mrt_name} | "

        f"R{route_order:03d} | "

        f"IMG{image_order:04d}"

    )


    if (
        all_cached
        and
        not FORCE_V4_DIAGNOSTICS
    ):

        for condition_key, path in (
            expected_paths.items()
        ):

            diagnostic_paths[
                condition_key
            ] = v4_browser_path(
                path
            )


        images_v4.append({

            **item,

            "diagnosticSegmentations":
                diagnostic_paths,

        })


        continue


    try:

        original_image, prediction = (
            v4_segmentation_map(
                original_path
            )
        )


        for condition_key, condition in (
            CONDITIONS_V4.items()
        ):

            semantic_group = (
                condition[
                    "semanticGroup"
                ]
            )


            class_ids = (
                GROUP_CLASS_IDS_V4[
                    semantic_group
                ]
            )


            output_path = (
                expected_paths[
                    condition_key
                ]
            )


            v4_make_diagnostic(

                original_image,

                prediction,

                class_ids,

                output_path

            )


            diagnostic_paths[
                condition_key
            ] = v4_browser_path(
                output_path
            )


        images_v4.append({

            **item,

            "diagnosticSegmentations":
                diagnostic_paths,

        })


    except Exception as error:

        print(
            "  FAILED:",
            error
        )


        images_v4.append({

            **item,

            "diagnosticSegmentations":
                diagnostic_paths,

        })


    if (
        DEVICE_V4
        ==
        "cuda"
    ):

        torch.cuda.empty_cache()


# ============================================================
# GROUP V4 IMAGES BY ROUTE
# ============================================================

images_by_route_v4 = defaultdict(
    list
)


for item in images_v4:

    route_id = str(
        item.get(
            "routeId"
        )
        or
        ""
    )


    if route_id:

        images_by_route_v4[
            route_id
        ].append(
            item
        )


for route_id in images_by_route_v4:

    images_by_route_v4[
        route_id
    ].sort(

        key=lambda item:
            v4_float(
                item.get(
                    "distanceFromMrt_m"
                ),
                0.0
            )

    )


# ============================================================
# CONTIGUOUS ROUTE SEGMENTS
#
# Convert 15 m V3 profile values into meaningful
# intervention zones.
# ============================================================

def v4_contiguous_segments(
    points,
    predicate,
    step_m=15.0
):

    groups = []

    current = []


    for point in points:

        value = (
            point.get(
                "value"
            )
        )


        if (
            value is not None
            and
            predicate(
                value
            )
        ):

            current.append(
                point
            )

        else:

            if current:

                groups.append(
                    current
                )

                current = []


    if current:

        groups.append(
            current
        )


    output = []


    for group in groups:

        if not group:
            continue


        start = float(
            group[0][
                "distance_m"
            ]
        )


        end = float(
            group[-1][
                "distance_m"
            ]
        )


        length = (
            end
            -
            start
            +
            step_m
        )


        # ----------------------------------------------------
        # Ignore isolated tiny fluctuations.
        # Require approximately 3 consecutive 15 m positions.
        # ----------------------------------------------------

        if length < 45:

            continue


        values = [

            float(
                point[
                    "value"
                ]
            )

            for point
            in group

        ]


        output.append({

            "start_m":
                round(
                    start,
                    1
                ),

            "end_m":
                round(
                    end,
                    1
                ),

            "length_m":
                round(
                    length,
                    1
                ),

            "averagePercent":
                round(
                    float(
                        np.mean(
                            values
                        )
                    ),
                    2
                ),

            "minimumPercent":
                round(
                    float(
                        np.min(
                            values
                        )
                    ),
                    2
                ),

            "maximumPercent":
                round(
                    float(
                        np.max(
                            values
                        )
                    ),
                    2
                ),

        })


    return output


# ============================================================
# CREATE ROUTE-SPECIFIC INTERVENTION ANALYSIS
#
# Uses V3's existing 15 m smooth gradient.
# ============================================================

routes_v4 = []


for route in routes_v3:

    route_id = str(
        route.get(
            "routeId"
        )
        or
        ""
    )


    series = (

        route.get(
            "routeSeriesV3"
        )

        or

        route.get(
            "routeSeries"
        )

        or

        {}

    )


    smoothed = (
        series.get(
            "smoothed"
        )
        or
        []
    )


    feature_analysis = {}


    for condition_key, condition in (
        CONDITIONS_V4.items()
    ):

        points = []


        for profile_point in smoothed:

            value = (
                profile_point
                .get(
                    "values",
                    {}
                )
                .get(
                    condition_key
                )
            )


            points.append({

                "distance_m":
                    v4_float(
                        profile_point.get(
                            "distance_m"
                        ),
                        0.0
                    ),

                "value":
                    (
                        v4_float(
                            value,
                            np.nan
                        )

                        if value
                        is not None

                        else None
                    ),

            })


        valid_values = np.asarray(

            [

                point[
                    "value"
                ]

                for point
                in points

                if (
                    point[
                        "value"
                    ]
                    is not None

                    and

                    math.isfinite(
                        point[
                            "value"
                        ]
                    )
                )

            ],

            dtype=float

        )


        if not len(
            valid_values
        ):

            feature_analysis[
                condition_key
            ] = {

                "label":
                    condition[
                        "label"
                    ],

                "primary":
                    condition[
                        "primary"
                    ],

                "description":
                    condition[
                        "description"
                    ],

                "averagePercent":
                    None,

                "minimumPercent":
                    None,

                "maximumPercent":
                    None,

                "problemSegments":
                    [],

                "recommendation":
                    condition[
                        "recommendation"
                    ],

            }


            continue


        average = float(
            np.mean(
                valid_values
            )
        )


        minimum = float(
            np.min(
                valid_values
            )
        )


        maximum = float(
            np.max(
                valid_values
            )
        )


        # ----------------------------------------------------
        # ROUTE-RELATIVE THRESHOLDS
        #
        # We are not claiming universal accessibility thresholds.
        #
        # Instead:
        #
        # low conditions:
        # bottom 30% of this exact route
        #
        # high conditions:
        # top 30% of this exact route
        # ----------------------------------------------------

        low_threshold = float(
            np.percentile(
                valid_values,
                30
            )
        )


        high_threshold = float(
            np.percentile(
                valid_values,
                70
            )
        )


        direction = (
            condition[
                "problemDirection"
            ]
        )


        if direction == "low":

            threshold_used = (
                low_threshold
            )


            predicate = (

                lambda value,
                threshold=threshold_used:

                    value
                    <=
                    threshold

            )


        elif direction == "high":

            threshold_used = (
                high_threshold
            )


            predicate = (

                lambda value,
                threshold=threshold_used:

                    value
                    >=
                    threshold

            )


        else:

            threshold_used = None


            predicate = (
                lambda value:
                    False
            )


        problem_segments = (
            v4_contiguous_segments(

                points,

                predicate,

                step_m=
                    v4_float(
                        series.get(
                            "step_m"
                        ),
                        15.0
                    )

            )
        )


        feature_analysis[
            condition_key
        ] = {

            "label":
                condition[
                    "label"
                ],

            "primary":
                condition[
                    "primary"
                ],

            "description":
                condition[
                    "description"
                ],

            "averagePercent":
                round(
                    average,
                    2
                ),

            "minimumPercent":
                round(
                    minimum,
                    2
                ),

            "maximumPercent":
                round(
                    maximum,
                    2
                ),

            "relativeThresholdPercent":
                (
                    round(
                        threshold_used,
                        2
                    )

                    if threshold_used
                    is not None

                    else None
                ),

            "problemDirection":
                direction,

            "problemSegments":
                problem_segments,

            "recommendation":
                condition[
                    "recommendation"
                ],

        }


    # ========================================================
    # PRIMARY DESIGN LENS
    #
    # Pedestrian space / walking continuity
    # ========================================================

    pedestrian = (
        feature_analysis.get(
            "pedestrian"
        )
        or
        {}
    )


    primary_problem = {

        "condition":
            "pedestrian",

        "label":
            "Pedestrian space / walking continuity",

        "reason":
            (
                "Pedestrian space is treated as the primary "
                "young-family design lens because it most directly "
                "affects continuous walking and stroller movement "
                "between the MRT and home."
            ),

        "averagePercent":
            pedestrian.get(
                "averagePercent"
            ),

        "problemSegments":
            pedestrian.get(
                "problemSegments",
                []
            ),

        "recommendation":
            pedestrian.get(
                "recommendation"
            ),

    }


    # ========================================================
    # MOST PROMINENT SUPPORTING CONDITION
    #
    # This does NOT replace pedestrian as the primary lens.
    #
    # It tells the frontend which secondary environmental
    # issue occupies the greatest problem-zone distance.
    # ========================================================

    supporting_scores = []


    for condition_key in [
        "greenery",
        "road",
        "vehicle",
        "sky"
    ]:

        analysis = (
            feature_analysis.get(
                condition_key
            )
            or
            {}
        )


        total_problem_length = sum(

            v4_float(
                segment.get(
                    "length_m"
                ),
                0.0
            )

            for segment
            in analysis.get(
                "problemSegments",
                []
            )

        )


        supporting_scores.append(
            (
                total_problem_length,
                condition_key
            )
        )


    supporting_scores.sort(
        reverse=True
    )


    if supporting_scores:

        _, most_prominent_key = (
            supporting_scores[0]
        )


        most_prominent = {

            "condition":
                most_prominent_key,

            **(
                feature_analysis.get(
                    most_prominent_key
                )
                or
                {}
            )

        }


    else:

        most_prominent = None


    routes_v4.append({

        **route,

        "featureAnalysis":
            feature_analysis,

        "primaryProblem":
            primary_problem,

        "mostProminentSupportingCondition":
            most_prominent,

        "diagnosticImageCount":
            len(
                images_by_route_v4.get(
                    route_id,
                    []
                )
            ),

    })


# ============================================================
# FRONTEND LIGHTWEIGHT IMAGE RECORDS
# ============================================================

frontend_images_v4 = []


for item in images_v4:

    frontend_images_v4.append({

        "imageId":
            item.get(
                "imageId"
            ),

        "mrtName":
            item.get(
                "mrtName"
            )
            or
            item.get(
                "station"
            ),

        "station":
            item.get(
                "station"
            )
            or
            item.get(
                "mrtName"
            ),

        "homeName":
            item.get(
                "homeName"
            ),

        "routeId":
            item.get(
                "routeId"
            ),

        "routeOrder":
            item.get(
                "routeOrder"
            ),

        "routeKey":
            item.get(
                "routeKey"
            ),

        "imageOrder":
            item.get(
                "imageOrder"
            ),

        "distanceFromMrt_m":
            item.get(
                "distanceFromMrt_m"
            ),

        "checkpointIndex":
            item.get(
                "checkpointIndex"
            ),

        "originalImage":
            item.get(
                "originalImage"
            ),

        "segmentationImage":
            item.get(
                "segmentationImage"
            ),

        "diagnosticSegmentations":
            item.get(
                "diagnosticSegmentations",
                {}
            ),

        "visualCluster":
            item.get(
                "visualCluster"
            ),

        "cluster":
            item.get(
                "visualCluster",
                item.get(
                    "cluster"
                )
            ),

        "clusterLabel":
            item.get(
                "clusterLabel"
            ),

        "somCell":
            item.get(
                "somCell"
            ),

        "groupShares":
            item.get(
                "groupShares"
            ),

        "visualMetrics":
            item.get(
                "visualMetrics"
            ),

    })


# ============================================================
# V4 COMPILER
# ============================================================

compiler_v4 = {

    **compiler_v3,

    "version":
        8,

    "dataset":
        "route_v4",

    "createdAt":
        datetime.now()
        .isoformat(
            timespec="seconds"
        ),

    "imageCount":
        len(
            images_v4
        ),

    "distanceConvention":
        "MRT_0_TO_HOME",

    "routeProfileMode":
        "SELECTED_ROUTE_ONLY",

    "routeProfileStep_m":
        15,

    "primaryDesignLens":
        "pedestrian",

    "primaryDesignLensLabel":
        "Pedestrian space / walking continuity",

    "diagnosticConditions":
        CONDITIONS_V4,

    "allImages":
        frontend_images_v4,

    "routes":
        routes_v4,

    "fullAnalysisFile":
        "./data/generated/image_analysis_route_v4.json",

}


# ============================================================
# FULL V4
# ============================================================

full_v4 = {

    **full_v3,

    "version":
        8,

    "dataset":
        "route_v4",

    "createdAt":
        compiler_v4[
            "createdAt"
        ],

    "imageCount":
        len(
            images_v4
        ),

    "distanceConvention":
        "MRT_0_TO_HOME",

    "routeProfileMode":
        "SELECTED_ROUTE_ONLY",

    "primaryDesignLens":
        "pedestrian",

    "diagnosticConditions":
        CONDITIONS_V4,

    "images":
        images_v4,

    "routes":
        routes_v4,

}


# ============================================================
# SAVE
# ============================================================

v4_json_dump(
    V4_COMPILER_PATH,
    compiler_v4
)


v4_json_dump(
    V4_FULL_PATH,
    full_v4
)


# ============================================================
# REPORT
# ============================================================

diagnostic_counts = defaultdict(
    int
)


for item in images_v4:

    diagnostics = (
        item.get(
            "diagnosticSegmentations"
        )
        or
        {}
    )


    for key in diagnostics:

        diagnostic_counts[
            key
        ] += 1


routes_with_primary_segments = sum(

    1

    for route
    in routes_v4

    if (
        route.get(
            "primaryProblem",
            {}
        )
        .get(
            "problemSegments"
        )
    )

)


print()
print(
    "================================================"
)

print(
    "ROUTE ENVIRONMENT V4 COMPLETE"
)

print(
    "================================================"
)

print(
    "Images:",
    len(
        images_v4
    )
)

print(
    "Routes:",
    len(
        routes_v4
    )
)

print()
print(
    "Primary design lens:"
)

print(
    "  Pedestrian space / walking continuity"
)

print()
print(
    "Routes with identified pedestrian weak zones:",
    routes_with_primary_segments
)

print()
print(
    "Diagnostic segmentation counts:"
)


for key in [
    "overall",
    "pedestrian",
    "greenery",
    "road",
    "vehicle",
    "sky",
    "building"
]:

    print(
        " ",
        key,
        ":",
        diagnostic_counts.get(
            key,
            0
        )
    )


print()
print(
    "Saved compiler:"
)

print(
    V4_COMPILER_PATH
)

print()
print(
    "Saved full analysis:"
)

print(
    V4_FULL_PATH
)

print()
print(
    "Diagnostic folder:"
)

print(
    V4_DIAGNOSTIC_ROOT
)

print(
    "================================================"
)


✓ V3 compiler found
✓ V3 full analysis found

V3 images: 1130
V3 routes: 192
V3 clusters: 10

Loading SegFormer for V4 diagnostics...
Device: cpu


Loading weights: 100%|██████████| 380/380 [00:00<00:00, 12131.77it/s]



V4 semantic groups:
  pedestrian_surface → 3 classes
  vegetation → 6 classes
  road → 2 classes
  vehicle → 5 classes
  sky → 2 classes
  building → 3 classes

Generating V4 individual segmentations...
Images: 1130
[1/1130] ADMIRALTY MRT | R001 | IMG0001
[2/1130] ADMIRALTY MRT | R001 | IMG0002
[3/1130] ADMIRALTY MRT | R001 | IMG0003
[4/1130] ADMIRALTY MRT | R001 | IMG0005
[5/1130] ADMIRALTY MRT | R001 | IMG0006
[6/1130] ADMIRALTY MRT | R001 | IMG0007
[7/1130] ADMIRALTY MRT | R001 | IMG0008
[8/1130] ADMIRALTY MRT | R001 | IMG0010
[9/1130] ADMIRALTY MRT | R001 | IMG0011
[10/1130] ADMIRALTY MRT | R001 | IMG0012
[11/1130] ADMIRALTY MRT | R001 | IMG0013
[12/1130] ADMIRALTY MRT | R001 | IMG0014
[13/1130] ADMIRALTY MRT | R001 | IMG0015
[14/1130] ADMIRALTY MRT | R001 | IMG0016
[15/1130] ADMIRALTY MRT | R001 | IMG0017
[16/1130] ADMIRALTY MRT | R001 | IMG0018
[17/1130] ADMIRALTY MRT | R001 | IMG0019
[18/1130] ADMIRALTY MRT | R002 | IMG0001
[19/1130] ADMIRALTY MRT | R002 | IMG0002
[20/1130] ADM